# 0️⃣ InteractivePipelineLoadFromPickle (Independent Load-only Visualization Notebook) - Imports

In [1]:
from pyphoplacecellanalysis.Analysis.Decoder.context_dependent import GenericDecoderDictDecodedEpochsDictResult
%config IPCompleter.use_jedi = False
# %xmode Verbose
# %xmode context
%pdb off
%load_ext autoreload
%autoreload 3

# ==================================================================================================================================================================================================================================================================================== #
# PyQtInspect:                                                                                                                                                                                                                                                                         #
# ==================================================================================================================================================================================================================================================================================== #
# 1. Launch `pqi-server` in a new terminal BEFORE running this notebook cell. Be sure to click 'Serve' button in the GUI that appears so this notebook can connect.

# # IMPORTANT: Call settrace BEFORE importing PyQt5
# import PyQtInspect.pqi as pqi

# # Connect to the server (default: localhost:19394)
# # Make sure the server is already running!
# pqi.settrace(
#     host='127.0.0.1',
#     port=19394,  # Default port, or use the port shown in the server GUI
#     qt_support='pyqt5',  # or 'auto' for auto-detection
#     patch_multiprocessing=False
# )

# # # !pip install viztracer
# %load_ext viztracer
# from viztracer import VizTracer

# %load_ext memory_profiler

import sys
from pathlib import Path

# required to enable non-blocking interaction:
%gui qt5

import importlib
from copy import deepcopy
from numba import jit
import numpy as np
import pandas as pd
pd.options.mode.chained_assignment = None  # default='warn'
# pd.options.mode.dtype_backend = 'pyarrow' # use new pyarrow backend instead of numpy
from attrs import define, field, fields, Factory, make_class
import tables as tb
from datetime import datetime, timedelta

# Pho's Formatting Preferences
import builtins

import IPython
from IPython.core.formatters import PlainTextFormatter
from IPython import get_ipython
from pyphocorehelpers.gui.Jupyter.AsyncExecutionHelper import run_async

from pyphocorehelpers.preferences_helpers import set_pho_preferences, set_pho_preferences_concise, set_pho_preferences_verbose
set_pho_preferences_concise()
# Jupyter-lab enable printing for any line on its own (instead of just the last one in the cell)
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

# BEGIN PPRINT CUSTOMIZATION ___________________________________________________________________________________________ #

## IPython pprint
from pyphocorehelpers.pprint import wide_pprint, wide_pprint_ipython, wide_pprint_jupyter, MAX_LINE_LENGTH
# Override default pprint
builtins.pprint = wide_pprint

ip = get_ipython()

from pyphocorehelpers.ipython_helpers import CustomFormatterMagics

# Register the magic
ip.register_magics(CustomFormatterMagics) # %%ndarray_preview height=500, width=None, include_plaintext_repr=False, include_shape=False, horizontal_layout=True


text_formatter: PlainTextFormatter = ip.display_formatter.formatters['text/plain']
text_formatter.max_width = MAX_LINE_LENGTH
text_formatter.for_type(object, wide_pprint_jupyter)


from pyphocorehelpers.pho_jupyter_preview_widget.ipython_helpers import PreviewWidgetMagics

ip.register_magics(PreviewWidgetMagics)


# END PPRINT CUSTOMIZATION ___________________________________________________________________________________________ #

from pyphocorehelpers.print_helpers import get_now_time_str, get_now_day_str
from pyphocorehelpers.indexing_helpers import get_dict_subset

## Pho's Custom Libraries:
from pyphocorehelpers.Filesystem.path_helpers import find_first_extant_path, file_uri_from_path
from pyphocorehelpers.Filesystem.open_in_system_file_manager import reveal_in_system_file_manager
import pyphocorehelpers.programming_helpers as programming_helpers
from pyphocorehelpers.print_helpers import render_scrollable_colored_table_from_dataframe, render_scrollable_colored_table

# NeuroPy (Diba Lab Python Repo) Loading
# from neuropy import core
from typing import Dict, List, Tuple, Optional, Callable, Union, Any
from typing_extensions import TypeAlias

import nptyping as ND
from nptyping import NDArray

from neuropy.utils.indexing_helpers import PandasHelpers, NumpyHelpers
from neuropy.utils.indexing_helpers import flatten
from neuropy.analyses.placefields import PlacefieldComputationParameters
from neuropy.core.epoch import NamedTimerange, Epoch, EpochsAccessor, ensure_dataframe, ensure_Epoch
from neuropy.core.ratemap import Ratemap
from neuropy.core.session.Formats.BaseDataSessionFormats import DataSessionFormatRegistryHolder, DataSessionFormatBaseRegisteredClass
# from neuropy.core.session.Formats.Specific.KDibaOldDataSessionFormat import KDibaOldDataSessionFormatRegisteredClass
# from neuropy.core.session.Formats.Specific.BapunDataSessionFormat import BapunDataSessionFormatRegisteredClass
# from neuropy.core.session.Formats.Specific.RachelDataSessionFormat import RachelDataSessionFormat
from neuropy.core.session.Formats.Specific.BapunDataSessionFormat import BapunDataSessionFormatRegisteredClass

from neuropy.utils.matplotlib_helpers import matplotlib_file_only, matplotlib_configuration, matplotlib_configuration_update
from neuropy.core.neuron_identities import NeuronIdentityTable, neuronTypesList, neuronTypesEnum
from neuropy.utils.mixins.AttrsClassHelpers import AttrsBasedClassHelperMixin, serialized_field, serialized_attribute_field, non_serialized_field, custom_define
from neuropy.utils.mixins.HDF5_representable import HDF_DeserializationMixin, post_deserialize, HDF_SerializationMixin, HDFMixin, HDF_Converter

## For computation parameters:
from neuropy.analyses.placefields import PlacefieldComputationParameters
from neuropy.utils.dynamic_container import DynamicContainer
from neuropy.utils.result_context import IdentifyingContext
from neuropy.core.session.Formats.BaseDataSessionFormats import find_local_session_paths
from neuropy.core.user_annotations import UserAnnotationsManager

from pyphocorehelpers.print_helpers import print_object_memory_usage, print_dataframe_memory_usage, print_value_overview_only, DocumentationFilePrinter, print_keys_if_possible, generate_html_string, document_active_variables
from pyphocorehelpers.programming_helpers import metadata_attributes
from pyphocorehelpers.function_helpers import function_attributes
## Pho Programming Helpers:
from pyphocorehelpers.print_helpers import DocumentationFilePrinter, TypePrintMode, print_keys_if_possible, debug_dump_object_member_shapes, print_value_overview_only, document_active_variables
from pyphocorehelpers.programming_helpers import IPythonHelpers, PythonDictionaryDefinitionFormat, MemoryManagement, inspect_callable_arguments, get_arguments_as_optional_dict, GeneratedClassDefinitionType, CodeConversion
from pyphocorehelpers.gui.Qt.TopLevelWindowHelper import TopLevelWindowHelper, print_widget_hierarchy
from pyphocorehelpers.indexing_helpers import reorder_columns, reorder_columns_relative, dict_to_full_array
from pyphocorehelpers.DataStructure.RenderPlots.MatplotLibRenderPlots import MatplotlibRenderPlots

doc_output_parent_folder: Path = Path('../EXTERNAL/DEVELOPER_NOTES/DataStructureDocumentation').resolve() # ../.
print(f"doc_output_parent_folder: {doc_output_parent_folder}")
assert doc_output_parent_folder.exists()

from pyphocorehelpers.notebook_helpers import NotebookCellExecutionLogger, NotebookProcessor

_notebook_path:Path = Path(IPythonHelpers.try_find_notebook_filepath(IPython.extract_module_locals())).resolve() # Finds the path of THIS notebook
_notebook_execution_logger: NotebookCellExecutionLogger = NotebookCellExecutionLogger(notebook_path=_notebook_path, enable_logging_to_file=False) # Builds a logger that records info about this notebook
_notebook_processor: NotebookProcessor = NotebookProcessor(path=_notebook_path)

# pyPhoPlaceCellAnalysis:
from pyphoplacecellanalysis.General.Pipeline.NeuropyPipeline import NeuropyPipeline # get_neuron_identities
from pyphoplacecellanalysis.General.Mixins.ExportHelpers import export_pyqtgraph_plot
from pyphoplacecellanalysis.General.Batch.NonInteractiveProcessing import batch_load_session, batch_extended_computations, batch_evaluate_required_computations
from pyphoplacecellanalysis.General.Pipeline.NeuropyPipeline import PipelineSavingScheme # used in perform_pipeline_save
from pyphoplacecellanalysis.GUI.IPyWidgets.pipeline_ipywidgets import PipelineJupyterHelpers, CustomProcessingPhases
from pyphocorehelpers.assertion_helpers import Assert
import pyphoplacecellanalysis.General.type_aliases as types # import neuropy.utils.type_aliases as types


import pyphoplacecellanalysis.External.pyqtgraph as pg

from pyphocorehelpers.exception_helpers import ExceptionPrintingContext, CapturedException
from pyphoplacecellanalysis.General.Batch.NonInteractiveProcessing import batch_perform_all_plots
from pyphoplacecellanalysis.General.Pipeline.Stages.ComputationFunctions.MultiContextComputationFunctions.LongShortTrackComputations import JonathanFiringRateAnalysisResult
from pyphoplacecellanalysis.General.Mixins.CrossComputationComparisonHelpers import _find_any_context_neurons
from pyphoplacecellanalysis.General.Batch.runBatch import BatchSessionCompletionHandler # for `post_compute_validate(...)`
from pyphoplacecellanalysis.Analysis.Decoder.reconstruction import BasePositionDecoder
from pyphoplacecellanalysis.SpecificResults.AcrossSessionResults import AcrossSessionsResults
from pyphoplacecellanalysis.General.Mixins.CrossComputationComparisonHelpers import SplitPartitionMembership
from pyphoplacecellanalysis.General.Pipeline.Stages.ComputationFunctions.MultiContextComputationFunctions.DirectionalPlacefieldGlobalComputationFunctions import DirectionalPlacefieldGlobalComputationFunctions, DirectionalLapsResult, TrackTemplates, DecoderDecodedEpochsResult
from pyphoplacecellanalysis.General.Pipeline.Stages.ComputationFunctions.MultiContextComputationFunctions.DirectionalPlacefieldGlobalComputationFunctions import TrackTemplates
from pyphoplacecellanalysis.General.Pipeline.Stages.ComputationFunctions.ComputationFunctionRegistryHolder import ComputationFunctionRegistryHolder, computation_precidence_specifying_function, global_function
from pyphoplacecellanalysis.General.Pipeline.Stages.ComputationFunctions.MultiContextComputationFunctions.SequenceBasedComputations import WCorrShuffle, SequenceBasedComputationsContainer
from neuropy.utils.mixins.binning_helpers import transition_matrix
from pyphoplacecellanalysis.Analysis.Decoder.transition_matrix import TransitionMatrixComputations
from pyphoplacecellanalysis.General.Pipeline.Stages.ComputationFunctions.MultiContextComputationFunctions.DirectionalPlacefieldGlobalComputationFunctions import TrackTemplates, get_proper_global_spikes_df
from pyphocorehelpers.Filesystem.path_helpers import set_posix_windows
from pyphoplacecellanalysis.Analysis.Decoder.reconstruction import BasePositionDecoder, DecodedFilterEpochsResult, SingleEpochDecodedResult
from neuropy.core.session.Formats.BaseDataSessionFormats import HardcodedProcessingParameters

from pyphocorehelpers.assertion_helpers import Assert

# Plotting
# import pylustrator # customization of figures
import matplotlib
import matplotlib as mpl
import matplotlib.pyplot as plt
_bak_rcParams = mpl.rcParams.copy()

matplotlib.use('Qt5Agg')
# %matplotlib inline
# %matplotlib auto

# _restore_previous_matplotlib_settings_callback = matplotlib_configuration_update(is_interactive=True, backend='Qt5Agg')
_restore_previous_matplotlib_settings_callback = matplotlib_configuration_update(is_interactive=True, backend='Qt5Agg')

import seaborn as sns

# import pylustrator # call `pylustrator.start()` before creating your first figure in code.
from pyphoplacecellanalysis.Pho2D.matplotlib.visualize_heatmap import visualize_heatmap, visualize_heatmap_pyqtgraph # used in `plot_kourosh_activity_style_figure`
from pyphoplacecellanalysis.General.Pipeline.Stages.DisplayFunctions.SpikeRasters import plot_multiple_raster_plot, plot_raster_plot
from pyphoplacecellanalysis.General.Mixins.DataSeriesColorHelpers import UnitColoringMode, DataSeriesColorHelpers
from pyphoplacecellanalysis.General.Pipeline.Stages.DisplayFunctions.SpikeRasters import _build_default_tick, build_scatter_plot_kwargs
from pyphoplacecellanalysis.GUI.PyQtPlot.Widgets.Mixins.Render2DScrollWindowPlot import Render2DScrollWindowPlotMixin, ScatterItemData
from pyphoplacecellanalysis.General.Pipeline.Stages.ComputationFunctions.SpikeAnalysis import SpikeRateTrends
from pyphoplacecellanalysis.General.Mixins.SpikesRenderingBaseMixin import SpikeEmphasisState
from pyphoplacecellanalysis.General.Model.SpecificComputationParameterTypes import ComputationKWargParameters
# from pyphoplacecellanalysis.SpecificResults.fourthYearPresentation import *

# Jupyter Widget Interactive
import ipywidgets as widgets
from IPython.display import display, HTML
from pyphocorehelpers.Filesystem.open_in_system_file_manager import reveal_in_system_file_manager
from pyphoplacecellanalysis.GUI.IPyWidgets.pipeline_ipywidgets import interactive_pipeline_widget, interactive_pipeline_files
from pyphocorehelpers.gui.Jupyter.simple_widgets import fullwidth_path_widget, render_colors

from datetime import datetime, date, timedelta
from pyphocorehelpers.print_helpers import get_now_day_str, get_now_rounded_time_str

from neuropy.core.session.Formats.BaseDataSessionFormats import HardcodedProcessingParameters

known_data_session_type_properties_dict = DataSessionFormatRegistryHolder.get_registry_known_data_session_type_dict()
active_data_session_types_registered_classes_dict = DataSessionFormatRegistryHolder.get_registry_data_session_type_class_name_dict()

DAY_DATE_STR: str = date.today().strftime("%Y-%m-%d")
DAY_DATE_TO_USE = f'{DAY_DATE_STR}' # used for filenames throught the notebook
print(f'DAY_DATE_STR: {DAY_DATE_STR}, DAY_DATE_TO_USE: {DAY_DATE_TO_USE}')

NOW_DATETIME: str = get_now_rounded_time_str()
NOW_DATETIME_TO_USE = f'{NOW_DATETIME}' # used for filenames throught the notebook
print(f'NOW_DATETIME: {NOW_DATETIME}, NOW_DATETIME_TO_USE: {NOW_DATETIME_TO_USE}')

def get_global_variable(var_name):
    """ used by `PipelineJupyterHelpers._build_pipeline_custom_processing_mode_selector_widget(...)` to update the notebook's variables """
    return globals()[var_name]
    
def update_global_variable(var_name, value):
    """ used by `PipelineJupyterHelpers._build_pipeline_custom_processing_mode_selector_widget(...)` to update the notebook's variables """
    globals()[var_name] = value

from pyphocorehelpers.gui.Jupyter.simple_widgets import build_global_data_root_parent_path_selection_widget
all_paths = [Path(r'/home/halechr/FastData'), Path('/Volumes/SwapSSD/Data'), Path('/Users/pho/data'), Path(r'/media/halechr/MAX/Data'), Path(r'H:\Data'), Path(r'W:\Data'), Path(r'/home/halechr/cloud/turbo/Data'), Path(r'/Volumes/MoverNew/data'), Path(r'/home/halechr/turbo/Data'), Path(r'/Users/pho/cloud/turbo/Data')] # Path('/Volumes/FedoraSSD/FastData'), 
global_data_root_parent_path = None
def on_user_update_path_selection(new_path: Path):
    global global_data_root_parent_path
    new_global_data_root_parent_path = new_path.resolve()
    global_data_root_parent_path = new_global_data_root_parent_path
    print(f'global_data_root_parent_path changed to {global_data_root_parent_path}')
    assert global_data_root_parent_path.exists(), f"global_data_root_parent_path: {global_data_root_parent_path} does not exist! Is the right computer's config commented out above?"
            
global_data_root_parent_path_widget = build_global_data_root_parent_path_selection_widget(all_paths, on_user_update_path_selection)
global_data_root_parent_path_widget

H:\TEMP\Spike3DEnv_ExploreUpgrade\Spike3DWorkEnv\NeuroPy\neuropy\utils\mixins\time_slicing.py:405: UserWarning: registration of accessor <class 'neuropy.utils.mixins.time_slicing.TimePointEventAccessor'> under name 'time_point_event' for type <class 'pandas.core.frame.DataFrame'> is overriding a preexisting attribute with the same name.
  class TimePointEventAccessor(TimeColumnAliasesProtocol, TimeSlicableObjectProtocol, DataframeMetadataProtocol):


Automatic pdb calling has been turned OFF
doc_output_parent_folder: H:\TEMP\Spike3DEnv_ExploreUpgrade\Spike3DWorkEnv\Spike3D\EXTERNAL\DEVELOPER_NOTES\DataStructureDocumentation
[{'cell_type': 'markdown', 'id': '5a45f3f6', 'metadata': {'tags': ['all']}, 'source': '# 0️⃣ InteractivePipelineLoadFromPickle (Independent Load-only Visualization Notebook) - Imports'}, {'cell_type': 'code', 'execution_count': None, 'id': '62f3f47a', 'metadata': {'tags': ['run-group-0']}, 'outputs': [], 'source': 'from pyphoplacecellanalysis.Analysis.Decoder.context_dependent import GenericDecoderDictDecodedEpochsDictResult\n%config IPCompleter.use_jedi = False\n# %xmode Verbose\n# %xmode context\n%pdb off\n%load_ext autoreload\n%autoreload 3\n\n# ============================================================================================================================================================================================================================================================================

h:\TEMP\Spike3DEnv_ExploreUpgrade\Spike3DWorkEnv\Spike3D\.venv\lib\site-packages\hdf5storage\utilities.py:44: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import parse_version


field.name: "merged_directional_placefields", variable_name: "merged_directional_placefields"
field.name: "rank_order_shuffle_analysis", variable_name: "rank_order_shuffle_analysis"
field.name: "directional_decoders_decode_continuous", variable_name: "directional_decoders_decode_continuous"
field.name: "directional_decoders_evaluate_epochs", variable_name: "directional_decoders_evaluate_epochs"
field.name: "directional_decoders_epoch_heuristic_scoring", variable_name: "directional_decoders_epoch_heuristic_scoring"
field.name: "directional_train_test_split", variable_name: "directional_train_test_split"
field.name: "long_short_decoding_analyses", variable_name: "long_short_decoding_analyses"
field.name: "long_short_rate_remapping", variable_name: "long_short_rate_remapping"
field.name: "long_short_inst_spike_rate_groups", variable_name: "long_short_inst_spike_rate_groups"
field.name: "wcorr_shuffle_analysis", variable_name: "wcorr_shuffle_analysis"
field.name: "non_pbe_epochs_results", 

ToggleButtons(description='Data Root:', layout=Layout(width='auto'), options=(WindowsPath('H:/Data'), WindowsPath('W:/Data')), style=ToggleButtonsStyle(button_width='max-content'), tooltip='global_data_root_parent_path', value=WindowsPath('H:/Data'))

# 0️⃣ Load Pipeline

In [2]:
# ==================================================================================================================== #
# Load Data                                                                                                            #
# ==================================================================================================================== #

# ==================================================================================================================== #
# BAPUN data format                                                                                                    #
# ==================================================================================================================== #
active_data_mode_name = 'bapun'
local_session_root_parent_context = IdentifyingContext(format_name=active_data_mode_name) # , animal_name='', configuration_name='one', session_name=a_sess.session_name
local_session_root_parent_path = global_data_root_parent_path.joinpath('Bapun')

# [*] - indicates bad or session with a problem
# 0, 1, 2, 3, 4, 5, 6, 7, [8], [9], 10, 11, [12], 13, 14, [15], [16], 17, 
curr_context = IdentifyingContext(format_name='bapun',animal='RatU', session_name='Day5OpenfieldSD') # NEW 2025-12-15 -- working but the epochs are a little weird, I had to manually rename them and they still don't seem right

# Create a dictionary with the parameters to override
override_parameters = {
    'preprocessing.laps.use_direction_dependent_laps': False,
    # 'preprocessing.laps.use_direction_dependent_laps': False
}

local_session_parent_path: Path = local_session_root_parent_path.joinpath(curr_context.animal) # 'gor01', 'one' - probably not needed anymore
basedir: Path = local_session_parent_path.joinpath(curr_context.session_name) #.resolve()

# basedir: Path = Path('/media/halechr/MAX/Data/Rachel/cho/cho_241117_2_merged') # DO NOT `.resolve()``
# basedir: Path = Path(r'H:\Data\Bapun\RatS\Day5TwoNovel')
print(f'basedir: {str(basedir)}')
Assert.path_exists(basedir)

epoch_name_includelist = None
active_computation_functions_name_includelist = ['pf_computation', 'pfdt_computation', 'position_decoding']

# Read if possible:
saving_mode = PipelineSavingScheme.SKIP_SAVING
force_reload = False

# Force write:
# saving_mode = PipelineSavingScheme.TEMP_THEN_OVERWRITE
# # saving_mode = PipelineSavingScheme.OVERWRITE_IN_PLACE
# force_reload = True

selector, on_value_change = PipelineJupyterHelpers._build_pipeline_custom_processing_mode_selector_widget(update_global_variable_fn=update_global_variable, debug_print=False, enable_full_view=True)
# selector.value = 'clean_run'
selector.value = 'continued_run'
# selector.value = 'final_run'
on_value_change(dict(new=selector.value)) ## do update manually so the workspace variables reflect the set values
## TODO: if loading is not possible, we need to change the `saving_mode` so that the new results are properly saved.
print(f"saving_mode: {saving_mode}, force_reload: {force_reload}")

basedir: H:\Data\Bapun\RatU\Day5OpenfieldSD


saving_mode: PipelineSavingScheme.SKIP_SAVING, force_reload: False


# Resume

In [3]:
from pyphoplacecellanalysis.GUI.IPyWidgets.pipeline_ipywidgets import PipelineJupyterHelpers, CustomProcessingPhases, PipelinePickleFileSelectorWidget

# ## INPUTS: basedir
# active_session_pickle_file_widget = PipelinePickleFileSelectorWidget(directory=basedir)

extended_computations_include_includelist_phase_dict: Dict[str, CustomProcessingPhases] = CustomProcessingPhases.get_extended_computations_include_includelist_phase_dict()

current_phase: CustomProcessingPhases = CustomProcessingPhases[selector.value]  # Assuming selector.value is an instance of CustomProcessingPhases
extended_computations_include_includelist: List[str] = [key for key, value in extended_computations_include_includelist_phase_dict.items() if value <= current_phase]
display(extended_computations_include_includelist)
force_recompute_override_computations_includelist = None
# force_recompute_override_computations_includelist = ['split_to_directional_laps', 'merged_directional_placefields', 'rank_order_shuffle_analysis', 'directional_decoders_decode_continuous'] # 

# ## INPUTS: basedir
active_session_pickle_file_widget = PipelinePickleFileSelectorWidget(directory=basedir, on_update_global_variable_callback=update_global_variable, on_get_global_variable_callback=get_global_variable)

_subfn_load, _subfn_save, _subfn_compute, _subfn_compute_new = active_session_pickle_file_widget._build_load_save_callbacks(global_data_root_parent_path=global_data_root_parent_path, active_data_mode_name=active_data_mode_name, basedir=basedir, saving_mode=saving_mode, force_reload=force_reload,
                                                             extended_computations_include_includelist=extended_computations_include_includelist, force_recompute_override_computations_includelist=force_recompute_override_computations_includelist)


# Display the widget
display(active_session_pickle_file_widget.servable())
# active_session_pickle_file_widget.local_file_browser_widget.servable()
# active_session_pickle_file_widget.global_file_browser_widget.servable()
# display(active_session_pickle_file_widget.local_file_browser_widget.servable())
# display(active_session_pickle_file_widget.global_file_browser_widget.servable())

# OUTPUTS: active_session_pickle_file_widget, widget.active_local_pkl, widget.active_global_pkl

if selector.value == 'clean_run':
    ## handle a clean run specially, this will create the pkls and not load them
    print(f'clean run!')
    default_selected_local_file_name: str = 'loadedSessPickle.pkl'
    default_selected_global_file_name: str = 'global_computation_results.pkl'
    # active_session_pickle_file_widget.is_compute_button_disabled = False # enable the compute button always during a clean run
    # active_session_pickle_file_widget.is_load_button_disabled = True
    
    new_default_local_pkl_file: Path = active_session_pickle_file_widget.directory.joinpath(default_selected_local_file_name).resolve()
    print(f'new_default_local_pkl_file: {new_default_local_pkl_file}')

    active_session_pickle_file_widget.selected_local_pkl_files = [new_default_local_pkl_file]
    active_session_pickle_file_widget.selected_global_pkl_files = []
    active_session_pickle_file_widget._update_load_save_button_disabled_state()
    print(f'active_session_pickle_file_widget.is_load_button_disabled: {active_session_pickle_file_widget.is_load_button_disabled}')
    print(f'active_session_pickle_file_widget.is_compute_button_disabled: {active_session_pickle_file_widget.is_compute_button_disabled}')
    print(f'active_local_pkl: "{active_session_pickle_file_widget.active_local_pkl}"')
    print(f'active_global_pkl: "{active_session_pickle_file_widget.active_global_pkl}"')
    active_session_pickle_file_widget.load_button.disabled = False
    active_session_pickle_file_widget.compute_button.disabled = False
else:
    # not `clean_run` mode, continuing processing which might include loading from pickles
    ## try selecting the first
    did_find_valid_selection: bool = active_session_pickle_file_widget.try_select_first_valid_files()

    ## Set default local comp pkl:
    default_selected_local_file_name: str = 'loadedSessPickle.pkl'
    if not active_session_pickle_file_widget.is_local_file_names_list_empty:
        default_local_section_indicies = [active_session_pickle_file_widget.local_file_browser_widget._data['File Name'].tolist().index(default_selected_local_file_name)]
        active_session_pickle_file_widget.local_file_browser_widget.selection = default_local_section_indicies

    ## Set default global computation pkl:
    default_selected_global_file_name: str = 'global_computation_results.pkl'
    if not active_session_pickle_file_widget.is_global_file_names_list_empty:
        default_global_section_indicies = [active_session_pickle_file_widget.global_file_browser_widget._data['File Name'].tolist().index(default_selected_global_file_name)]
        active_session_pickle_file_widget.global_file_browser_widget.selection = default_global_section_indicies



['lap_direction_determination',
 'pf_computation',
 'pfdt_computation',
 'position_decoding',
 'firing_rate_trends',
 'extended_stats',
 'long_short_decoding_analyses',
 'jonathan_firing_rate_analysis',
 'long_short_fr_indicies_analyses',
 'long_short_post_decoding',
 'long_short_inst_spike_rate_groups',
 'long_short_endcap_analysis',
 'split_to_directional_laps',
 'merged_directional_placefields',
 'directional_decoders_decode_continuous',
 'directional_decoders_evaluate_epochs',
 'directional_decoders_epoch_heuristic_scoring',
 'non_PBE_epochs_results',
 'generalized_specific_epochs_decoding']

Column
    [0] Tabulator(disabled=True, height=400, page_size=10, pagination='local', show_index=False, sorters=[{'field': 'Modification D...], value=              ...)
    [1] Tabulator(disabled=True, height=400, page_size=10, pagination='local', show_index=False, sorters=[{'field': 'Modification D...], value=              ...)
    [2] Column(margin=(10, 0))
        [0] HTML(str)
        [1] Row
            [0] IntInput(end=100, name='Min FR (Hz)', start=0, value=2, width=150)
            [1] TextInput(name='QClu Values', placeholder='Enter as list: [1, ..., value='[1, 2, 4, 6, 7, 8, 9]')
    [3] Row
        [0] Button(button_type='success', name='Save')
        [1] Button(button_type='primary', disabled=True, name='Load')
        [2] Button(button_type='warning', name='Compute')
        [3] Button(button_type='primary', name='Compute New')

In [4]:
did_find_valid_selection: bool = active_session_pickle_file_widget.try_select_first_valid_files()
did_find_valid_selection


True

In [ ]:
# if did_find_valid_selection:
#     _subfn_load()
    
did_find_valid_selection = True

In [5]:
if did_find_valid_selection:
    curr_active_pipeline, custom_suffix, proposed_load_pkl_path = active_session_pickle_file_widget.on_load_local(global_data_root_parent_path=global_data_root_parent_path, active_data_mode_name=active_data_mode_name, basedir=basedir, saving_mode=saving_mode, force_reload=force_reload)
    print(f'on_load_local(...) complete. workspace variables updated: curr_active_pipeline, custom_suffix, proposed_load_pkl_path')
    

custom_suffix: ""
Computing loaded session pickle file results : "H:/Data/Bapun/RatU/Day5OpenfieldSD/loadedSessPickle.pkl"... 	done.
	done.
	done.
	done.
	done.
	done.
	done.
	done.
	done.
	done.
	done.
	done.
	done.
	done.
	done.
	done.
build_logger(full_logger_string="2026-06-22_14-06-58.Apogee.bapun.RatU.Day5OpenfieldSD", file_logging_dir: None):
done.
Loading pickled pipeline success: H:\Data\Bapun\RatU\Day5OpenfieldSD\loadedSessPickle.pkl.


	 time variable changed from 't_rel_seconds' to 't_seconds'.


	 time variable changed!
properties already present in pickled version. No need to save.
pipeline load success!
using provided computation_functions_name_includelist: ['lap_direction_determination', 'pf_computation', 'firing_rate_trends', 'position_decoding']
not using direction-dependent laps.


saving_mode.shouldSave == False, so not saving at the end of batch_load_session
WARN: `_update_pipeline_missing_preprocessing_parameters(...): non-KDIBA format curr_active_pipeline.active_sess_config.format_name "bapun" is not currently fully implemented/checked for all parameters. Filtered sessions might ahve wrong params.
	trying to process for maze_GLOBAL..
	trying to process for roam..
	trying to process for sprinkle..
were pipeline preprocessing parameters missing and updated?: False
Pipeline loaded from custom pickle!!
# ==================================================================================================================== #
        # on_load_local -- COMPLETE -- 
        # ==================================================================================================================== #
on_load_local(...) complete. workspace variables updated: curr_active_pipeline, custom_suffix, proposed_load_pkl_path


In [ ]:
if did_find_valid_selection:
    try:
        skip_global_load = False
        curr_active_pipeline = active_session_pickle_file_widget.on_load_global(curr_active_pipeline=curr_active_pipeline, basedir=basedir, extended_computations_include_includelist=extended_computations_include_includelist, force_recompute_override_computations_includelist=force_recompute_override_computations_includelist,
                                    skip_global_load=skip_global_load, force_reload=False, override_global_computation_results_pickle_path=active_session_pickle_file_widget.active_global_pkl)
        # Update the global variable after loading global
        print(f'on_load_global(...) complete. workspace variables updated: curr_active_pipeline, custom_suffix, proposed_load_pkl_path')
    except Exception as e:
        print(f'encountered exception loading global e: {e}.')
        pass
        # raise e



## From `test_non_interactive_crash.py`

In [ ]:
### Bapun Open-Field Experiment (2022-08-09 Analysis)
from neuropy.core.session.SessionSelectionAndFiltering import build_custom_epochs_filters # used particularly to build Bapun-style filters

active_data_mode_name = 'bapun'
print(f'active_data_session_types_registered_classes_dict: {active_data_session_types_registered_classes_dict}')
active_data_mode_registered_class = active_data_session_types_registered_classes_dict[active_data_mode_name]
active_data_mode_type_properties = known_data_session_type_properties_dict[active_data_mode_name]

# basedir = Path('/media/halechr/MAX/Data/Rachel/Cho_241117_Session2').resolve()
## INPUTS: basedir 
override_parameters = {'rank_order_shuffle_analysis.minimum_inclusion_fr_Hz': 1.0}
force_reload = force_reload #True
print(f'force_reload: {force_reload}')
curr_active_pipeline = NeuropyPipeline.try_init_from_saved_pickle_or_reload_if_needed(active_data_mode_name, active_data_mode_type_properties, override_basepath=Path(basedir), force_reload=force_reload) # , override_parameters_flat_keypaths_dict=override_parameters

# _test_session = RachelDataSessionFormat.build_session(Path(r'R:\data\Rachel\merged_M1_20211123_raw_phy'))
# _test_session, loaded_file_record_list = RachelDataSessionFormat.load_session(_test_session)
# _test_session

## ~20m

##### Old not needed anymore manual comps

In [ ]:
curr_epoch_names: List[str] = curr_active_pipeline.sess.epochs.to_dataframe()['label'].to_list()
print(f'curr_epoch_names: {curr_epoch_names}')

In [ ]:
from neuropy.core.session.SessionSelectionAndFiltering import build_custom_epochs_filters

# epoch_name_includelist = ['pre', 'maze1', 'post1', 'maze2', 'post2']
# epoch_name_includelist = ['pre', 'roam', 'sprinkle', 'post']
# epoch_name_includelist = ['roam', 'sprinkle']

# active_session_filter_configurations = build_custom_epochs_filters(curr_active_pipeline.sess, epoch_name_includelist=['pre', 'maze1', 'post1', 'maze2', 'post2']) ## ALL possible epochs

# active_session_filter_configurations = build_custom_epochs_filters(curr_active_pipeline.sess, epoch_name_includelist=['maze1', 'maze2', 'maze_GLOBAL']) ## ALL possible epochs
active_session_filter_configurations = build_custom_epochs_filters(curr_active_pipeline.sess, epoch_name_includelist=['maze1', 'maze2', 'maze_GLOBAL']) ## ALL possible epochs

# active_session_filter_configurations = active_data_mode_registered_class.build_default_filter_functions(sess=curr_active_pipeline.sess)
# active_session_filter_configurations = build_custom_epochs_filters(curr_active_pipeline.sess, epoch_name_includelist=['pre', 'roam', 'maze', 'sprinkle', 'post']) ## ALL possible epochs
# active_session_filter_configurations = build_custom_epochs_filters(curr_active_pipeline.sess, epoch_name_includelist=['pre', 'roam', 'sprinkle', 'post']) ## ALL possible epochs

# active_session_filter_configurations = active_data_mode_registered_class.build_default_filter_functions(sess=curr_active_pipeline.sess, epoch_name_includelist=epoch_name_includelist) # build_filters_pyramidal_epochs(sess=curr_kdiba_pipeline.sess)
# active_session_filter_configurations = build_custom_epochs_filters(curr_active_pipeline.sess, epoch_name_includelist=['maze','sprinkle'])
# active_session_filter_configurations = build_custom_epochs_filters(curr_active_pipeline.sess, epoch_name_includelist=['maze', 'sprinkle'])
# active_session_filter_configurations = build_custom_epochs_filters(curr_active_pipeline.sess, epoch_name_includelist=['roam', 'sprinkle']) # , 'maze'

# active_session_filter_configurations = active_data_mode_registered_class.build_filters_pyramidal_epochs(curr_active_pipeline.sess, epoch_name_includelist=['maze','sprinkle'])
# active_session_filter_configurations


In [ ]:
curr_active_pipeline.filter_sessions(active_session_filter_configurations)


In [ ]:
active_session_computation_configs = active_data_mode_registered_class.build_active_computation_configs(sess=curr_active_pipeline.sess, time_bin_size=0.5)
active_session_computation_configs

In [ ]:
active_session_computation_configs[0].pf_params.computation_epochs

In [ ]:
# grid_bin_bounds=(((-83.33747881216672, 110.15967332926644), (-94.89955475226206, 97.07387994733473)))


bapun_open_field_grid_bin_bounds = (((-120.0, 120.0), (-120.0, 120.0)))
curr_active_pipeline.get_all_parameters()
# curr_active_pipeline.update_parameters(grid_bin_bounds = (((-120.0, 120.0), (-120.0, 120.0))))
curr_active_pipeline.sess.config.grid_bin_bounds = (((-120.0, 120.0), (-120.0, 120.0)))


# override_parameters_flat_keypaths_dict = {'grid_bin_bounds': (((-120.0, 120.0), (-120.0, 120.0))), # 'rank_order_shuffle_analysis.minimum_inclusion_fr_Hz': minimum_inclusion_fr_Hz,
# 										#   'sess.config.preprocessing_parameters.laps.use_direction_dependent_laps': False, # lap_estimation_parameters
#                                         }

# curr_active_pipeline.update_parameters(override_parameters_flat_keypaths_dict=override_parameters_flat_keypaths_dict) # should already be updated, but try it again anyway.


In [ ]:
minimum_inclusion_fr_Hz = 1.0
override_parameters_flat_keypaths_dict = {
                            # 'grid_bin_bounds': (((-120.0, 120.0), (-120.0, 120.0))), # 
							'grid_bin': (4, 4), 
                            'rank_order_shuffle_analysis.minimum_inclusion_fr_Hz': minimum_inclusion_fr_Hz,
                        #   'sess.config.preprocessing_parameters.laps.use_direction_dependent_laps': False, # lap_estimation_parameters
                        }

curr_active_pipeline.update_parameters(override_parameters_flat_keypaths_dict=override_parameters_flat_keypaths_dict) # should already be updated, but try it again anyway.

In [ ]:
from neuropy.core.epoch import Epoch, EpochsAccessor, ensure_dataframe, ensure_Epoch


# ==================================================================================================================================================================================================================================================================================== #
# Update computation_epochs to be only the maze ones                                                                                                                                                                                                                                   #
# ==================================================================================================================================================================================================================================================================================== #

## activity_only_epochs_df:
epochs_df = ensure_dataframe(deepcopy(curr_active_pipeline.sess.epochs))
# activity_only_epochs_df: pd.DataFrame = epochs_df[epochs_df['label'].isin(['maze1', 'maze2', 'maze_GLOBAL'])]

activity_only_epochs_df: pd.DataFrame = epochs_df[epochs_df['label'].isin(['maze1', 'maze2'])].epochs.get_non_overlapping_df()
activity_only_epochs: Epoch = ensure_Epoch(activity_only_epochs_df, metadata=curr_active_pipeline.sess.epochs.metadata)

## GLobal only ('maze_GLOBAL')
epochs_df = ensure_dataframe(deepcopy(curr_active_pipeline.sess.epochs))
global_activity_only_epochs_df: pd.DataFrame = epochs_df[epochs_df['label'].isin(['maze_GLOBAL'])].epochs.get_non_overlapping_df()
global_activity_only_epoch: Epoch = ensure_Epoch(global_activity_only_epochs_df, metadata=curr_active_pipeline.sess.epochs.metadata)

## OUTPUTS: activity_only_epochs, global_activity_only_epoch

## OUTPUTS: activity_only_epoch


# active_session_computation_configs[0].pf_params.computation_epochs = deepcopy(curr_active_pipeline.filtered_sessions['maze'].epochs)
# active_session_computation_configs[0].pf_params.computation_epochs = deepcopy(curr_active_pipeline.sess.epochs)
# active_session_computation_configs[0].pf_params.computation_epochs = deepcopy(curr_active_pipeline.sess.epochs) ## prev
active_session_computation_configs[0].pf_params.computation_epochs = deepcopy(activity_only_epochs)

global_only_sess_comp_config = deepcopy(active_session_computation_configs[0])
global_only_sess_comp_config.pf_params.computation_epochs = deepcopy(global_activity_only_epoch)
if len(active_session_computation_configs) < 2:
    active_session_computation_configs.append(global_only_sess_comp_config)
else:
    active_session_computation_configs[1] = global_only_sess_comp_config

# active_session_computation_configs[0].pf_params.computation_epochs = deepcopy(bapun_epochs)
# active_session_computation_configs[1].pf_params.computation_epochs = deepcopy(curr_active_pipeline.filtered_sessions['maze'].epochs.to_dataframe())
active_session_computation_configs
# active_session_computation_configs[0].pf_params.computation_epochs

#    start   stop     label  duration
# 0      0   7407       pre      7407
# 1   7423  11483      maze      4060
# 3  10186  11483  sprinkle      1297
# 2  11497  25987      post     14490

# [4 rows x 4 columns]

## UPDATES: active_session_computation_configs


In [ ]:
activity_only_epochs_df: pd.DataFrame = epochs_df[epochs_df['label'].isin(hardcoded_params.non_global_activity_session_names)]
activity_only_epochs_df

activity_only_epochs_df.loc[1, 'stop'] = activity_only_epochs_df.loc[2, 'start'] - 0.001
activity_only_epochs_df.loc[1, 'label'] = 'roam' 
activity_only_epochs_df['duration'] = activity_only_epochs_df['stop'] -  activity_only_epochs_df['start']
activity_only_epochs_df

hardcoded_params.non_global_activity_session_names = ['roam', 'sprinkle']


1   7125.0  11745.0         maze    4620.0
2   9591.0  11745.0     sprinkle    2154.0

In [ ]:
active_session_computation_configs[0].pf_params.linearization_method = "umap"

for an_epoch_name, a_sess in curr_active_pipeline.filtered_sessions.items():
    ## forcibly compute the linearized position so it doesn't fallback to "isomap" method which eats all the memory
    a_pos_df: pd.DataFrame = a_sess.position.compute_linearized_position(method='umap')
    


In [ ]:
# activity_only_epoch_names: List[str] = ['maze1', 'maze2', 'maze_GLOBAL']
# active_computation_functions_name_includelist
activity_only_epoch_names: List[str] = active_session_computation_configs[0].pf_params.computation_epochs.labels.tolist() ## should be same as config
activity_only_epoch_names

# # Create non-overlapping version
# non_overlapping_epochs = ensure_Epoch(active_session_computation_configs[0].pf_params.computation_epochs.epochs.get_non_overlapping_df())
# active_session_computation_configs[0].pf_params.computation_epochs = non_overlapping_epochs


In [ ]:
curr_active_pipeline.computation_results

In [ ]:
active_session_computation_configs = curr_active_pipeline.

In [ ]:
from pyphoplacecellanalysis.General.Pipeline.NeuropyPipeline import NeuropyPipeline
from pyphoplacecellanalysis.General.Batch.NonInteractiveProcessing import batch_extended_computations

curr_active_pipeline.reload_default_computation_functions()
    
active_computation_functions_name_includelist = ['pf_computation',
                                                'pfdt_computation',
                                                'position_decoding',
                                                #  'position_decoding_two_step',
                                                #  'extended_pf_peak_information',
                                                ] # 'ratemap_peaks_prominence2d'


## Loops through all configs
for i, a_config in enumerate(active_session_computation_configs):
    active_epoch_names: List[str] = a_config.pf_params.computation_epochs.labels.tolist() ## should be same as config
    print(f'i: {i}, active_epoch_names: {active_epoch_names}') # (activity_only_epoch_names)

    # curr_active_pipeline.perform_computations(active_session_computation_configs[0], computation_functions_name_excludelist=['_perform_spike_burst_detection_computation', '_perform_velocity_vs_pf_density_computation', '_perform_velocity_vs_pf_simplified_count_density_computation']) # SpikeAnalysisComputations._perform_spike_burst_detection_computation
    # curr_active_pipeline.perform_computations(active_session_computation_configs[0], computation_functions_name_includelist=active_computation_functions_name_includelist, enabled_filter_names=activity_only_epoch_names, overwrite_extant_results=True, fail_on_exception=False, debug_print=True) # SpikeAnalysisComputations._perform_spike_burst_detection_computation
    curr_active_pipeline.perform_computations(a_config, computation_functions_name_includelist=active_computation_functions_name_includelist, enabled_filter_names=active_epoch_names, overwrite_extant_results=False, fail_on_exception=False, debug_print=True) # SpikeAnalysisComputations._perform_spike_burst_detection_computation



In [ ]:
curr_active_pipeline.sess.epochs.to_dataframe()

In [ ]:
curr_active_pipeline.active_completed_computation_result_names

In [ ]:
# curr_active_pipeline.perform_computations(active_session_computation_configs[0], computation_functions_name_includelist=active_computation_functions_name_includelist, enabled_filter_names=['maze1', 'maze2'], overwrite_extant_results=False, fail_on_exception=False, debug_print=True)

In [ ]:
# curr_active_pipeline.computation_results['maze'].accumulated_errors
curr_active_pipeline.clear_all_failed_computations()

In [ ]:
curr_active_pipeline.prepare_for_display(root_output_dir=r'Output', should_smooth_maze=True) # TODO: pass a display config
# curr_active_pipeline.prepare_for_display(root_output_dir=r'W:\Data\Output', should_smooth_maze=True) # TODO: pass a display config

In [ ]:
curr_active_pipeline.pickle_path
curr_active_pipeline.global_computation_results_pickle_path
curr_active_pipeline.get_output_path()

In [ ]:
# _out = curr_active_pipeline.save_pipeline(saving_mode=PipelineSavingScheme.TEMP_THEN_OVERWRITE)
_out = curr_active_pipeline.save_pipeline(saving_mode=PipelineSavingScheme.OVERWRITE_IN_PLACE)
# _out = curr_active_pipeline.save_pipeline(saving_mode=PipelineSavingScheme.TEMP_THEN_OVERWRITE, active_pickle_filename='loadedSessPickle_2025-02-26.pkl')


In [ ]:
_out = curr_active_pipeline.save_global_computation_results()#save_pipeline(saving_mode=PipelineSavingScheme.TEMP_THEN_OVERWRITE, active_pickle_filename='loadedSessPickle_2025-02-27.pkl')

In [ ]:
# include_includelist = ['pre', 'maze1', 'post1', 'maze2', 'post2', 'maze',]
include_includelist = ['roam', 'sprinkle']
# include_includelist = curr_active_pipeline.filtered_session_names
include_includelist

In [ ]:
curr_active_pipeline.filtered_session_names

In [ ]:
include_includelist = curr_active_pipeline.filtered_session_names
print(f'include_includelist: {include_includelist}')

In [ ]:
## Setup Computation Functions to be executed:
# includelist Mode:
computation_functions_name_includelist=['_perform_baseline_placefield_computation', '_perform_time_dependent_placefield_computation', '_perform_extended_statistics_computation',
                                '_perform_position_decoding_computation', 
                                '_perform_firing_rate_trends_computation',
                                '_perform_pf_find_ratemap_peaks_computation',
                                # '_perform_time_dependent_pf_sequential_surprise_computation'
                                '_perform_two_step_position_decoding_computation',
                                # '_perform_recursive_latent_placefield_decoding'
                            ]  # '_perform_pf_find_ratemap_peaks_peak_prominence2d_computation'
computation_functions_name_excludelist=None

batch_extended_computations(curr_active_pipeline, included_computation_filter_names=computation_functions_name_includelist, include_includelist=include_includelist,
                            include_global_functions=True, fail_on_exception=False, progress_print=True, debug_print=False)


In [ ]:
## Firing rate filter seems too high (5.0Hz, maybe should be lower at like 1.0Hz)?
curr_active_pipeline.sess.config



## NEW BATCH COMPUTE ALL

In [ ]:
from neuropy.utils.matplotlib_helpers import interactive_select_grid_bin_bounds_2D

# # Non-blocking (returns handles for further use):
# fig, ax, rect_selector, set_extents, reset_extents = interactive_select_grid_bin_bounds_2D(
#     curr_active_pipeline, epoch_name='roam', should_block_for_input=False
# )

# Blocking (waits for [Enter] keypress, then returns confirmed extents):
grid_bin_bounds = interactive_select_grid_bin_bounds_2D(
    curr_active_pipeline, epoch_name='roam',
    should_block_for_input=True,
    # should_apply_updates_to_pipeline=False,  # set True to write back to all filtered epochs
    should_apply_updates_to_pipeline=True,  # set True to write back to all filtered epochs
    # grid_bin_bounds = ((3.1635999999999997, 143.6185), (0.6301, 92.3699)),
	# grid_bin_bounds = ((4.441910188658099, 140.27433571428577), (1.889081672096026, 29.322947571677616)),
    # grid_bin_bounds = ((0.0, 142.0), (0.0, 30.0)),
	# grid_bin_bounds = ((1.9826839826839802, 142.0, -31.199134199134207, 171.96103896103898)),
	grid_bin_bounds = ((2.0, 142.0, -32.0, 172.0)),
)
print(f'grid_bin_bounds: {grid_bin_bounds}')


# (1.9826839826839802, 142.0, -31.199134199134207, 171.96103896103898)


In [ ]:
## Find absolute raw outputs:
pos_df: pd.DataFrame = curr_active_pipeline.sess.position.to_dataframe()
x_min, x_max = pos_df['x'].min(), pos_df['x'].max()
y_min, y_max = pos_df['y'].min(), pos_df['y'].max()

((x_min, x_max), (y_min, y_max))


In [ ]:
## find the 2 most promendant disjoint peaks of the occupancy (should be on opposite sides of the track)
curr_active_pipeline.computation_results['roam']

In [ ]:
from neuropy.core.session.Formats.BaseDataSessionFormats import HardcodedProcessingParameters
from neuropy.core.session.Formats.BaseDataSessionFormats import DataSessionFormatRegistryHolder, DataSessionFormatBaseRegisteredClass
from neuropy.core.session.Formats.Specific.BapunDataSessionFormat import BapunDataSessionFormatRegisteredClass
from pyphoplacecellanalysis.SpecificResults.PendingNotebookCode import final_process_bapun_all_comps

# try:
# time_bin_size: float = 0.010 # 10ms bins
time_bin_size: float = 0.020 # 20ms bins
curr_active_pipeline = final_process_bapun_all_comps(curr_active_pipeline=curr_active_pipeline, posthoc_save=False, time_bin_size=time_bin_size)
# curr_active_pipeline = final_process_bapun_all_comps(curr_active_pipeline=curr_active_pipeline, posthoc_save=True)
# except Exception as e:
#     print(f'exception: {e}')
#     # raise e
#     pass    

## 9m

In [ ]:
force_recompute_override_computations_includelist = ['split_to_directional_laps', 'merged_directional_placefields', 'directional_decoders_decode_continuous'] 
# curr_active_pipeline.perform_drop_computed_result(computed_data_keys_to_drop=['DirectionalDecodersDecoded'])

curr_active_pipeline.reload_default_computation_functions()
# curr_active_pipeline.perform_specific_computation(active_computation_params={}, compute )
curr_active_pipeline.perform_specific_computation(computation_functions_name_includelist=force_recompute_override_computations_includelist, fail_on_exception=True, debug_print=False)
# curr_active_pipeline.perform_specific_computation(computation_functions_name_includelist=force_recompute_override_computations_includelist, fail_on_exception=False, debug_print=False)



In [ ]:
# desired_time_bin_size = 0.010 # 10ms
desired_time_bin_size = 0.250 # 250ms
curr_active_pipeline.perform_specific_computation(computation_functions_name_includelist=['directional_decoders_decode_continuous'], computation_kwargs_list=[{'time_bin_size': desired_time_bin_size, 'should_disable_cache': True}], enabled_filter_names=None, fail_on_exception=True, debug_print=False)
# curr_active_pipeline.perform_specific_computation(computation_functions_name_includelist=['directional_decoders_decode_continuous'], computation_kwargs_list=[{'time_bin_size': desired_time_bin_size, 'should_disable_cache': False}], enabled_filter_names=None, fail_on_exception=True, debug_print=False)


# 💾 Save Export Pipeline


In [ ]:
curr_active_pipeline.get_complete_session_context()
custom_save_filepaths, custom_save_filenames, custom_suffix = curr_active_pipeline.get_custom_pipeline_filenames_from_parameters()
custom_save_filenames

In [ ]:
custom_save_filenames['pipeline_pkl']
custom_save_filenames['global_computation_pkl']

pickle_path = 'loadedSessPickle_withNormalComputedReplays-qclu_[1, 2, 4, 6, 7, 9]-frateThresh_5.0_2025-01-20.pkl'
global_computation_pkl = 'global_computation_results_withNormalComputedReplays-qclu_[1, 2, 4, 6, 7, 9]-frateThresh_5.0_2025-01-20.pkl'

In [ ]:
## indicate that it was loaded with a custom suffix
curr_active_pipeline.pickle_path ## correct
curr_active_pipeline.global_computation_results_pickle_path ## correct

# curr_active_pipeline.save_pipeline(saving_mode=PipelineSavingScheme.TEMP_THEN_OVERWRITE, override_pickle_path=curr_active_pipeline.pickle_path, active_pickle_filename=curr_active_pipeline.pickle_path.name) #active_pickle_filename=
# curr_active_pipeline.save_global_computation_results(override_global_pickle_path=curr_active_pipeline.global_computation_results_pickle_path)

In [ ]:
## indicate that it was loaded with a custom suffix
curr_active_pipeline.pickle_path ## correct
curr_active_pipeline.global_computation_results_pickle_path ## correct

if curr_active_pipeline.pickle_path is None:
    active_pickle_filename = 'loadedSessPickle.pkl'
else:
    active_pickle_filename = curr_active_pipeline.pickle_path.name
    
print(f'active_pickle_filename: {active_pickle_filename}')
curr_active_pipeline.save_pipeline(saving_mode=PipelineSavingScheme.TEMP_THEN_OVERWRITE, override_pickle_path=curr_active_pipeline.pickle_path, active_pickle_filename=active_pickle_filename) #active_pickle_filename=


In [ ]:
curr_active_pipeline.save_global_computation_results(override_global_pickle_path=curr_active_pipeline.global_computation_results_pickle_path)

### 2024-06-25 - Load from saved custom

In [ ]:
from pyphocorehelpers.Filesystem.path_helpers import set_posix_windows

# Loads custom pipeline pickles that were saved out via `custom_save_filepaths['pipeline_pkl'] = curr_active_pipeline.save_pipeline(saving_mode=PipelineSavingScheme.TEMP_THEN_OVERWRITE, active_pickle_filename=custom_save_filenames['pipeline_pkl'])`

## INPUTS: global_data_root_parent_path, active_data_mode_name, basedir, saving_mode, force_reload, custom_save_filenames
# custom_suffix: str = '_withNewKamranExportedReplays'

# custom_suffix: str = '_withNewComputedReplays'
# custom_suffix: str = '_withNewComputedReplays-qclu_[1, 2]-frateThresh_5.0'

# custom_save_filenames = {
#     'pipeline_pkl':f'loadedSessPickle{custom_suffix}.pkl',
#     'global_computation_pkl':f"global_computation_results{custom_suffix}.pkl",
#     'pipeline_h5':f'pipeline{custom_suffix}.h5',
# }
# print(f'custom_save_filenames: {custom_save_filenames}')
# custom_save_filepaths = {k:v for k, v in custom_save_filenames.items()}

# # ==================================================================================================================== #
# # PIPELINE LOADING                                                                                                     #
# # ==================================================================================================================== #
# # load the custom saved outputs
# active_pickle_filename = custom_save_filenames['pipeline_pkl'] # 'loadedSessPickle_withParameters.pkl'
# print(f'active_pickle_filename: "{active_pickle_filename}"')
# # assert active_pickle_filename.exists()
# active_session_h5_filename = custom_save_filenames['pipeline_h5'] # 'pipeline_withParameters.h5'
# print(f'active_session_h5_filename: "{active_session_h5_filename}"')

# ==================================================================================================================== #
# Load Pipeline                                                                                                        #
# ==================================================================================================================== #
## DO NOT allow recompute if the file doesn't exist!!
# Computing loaded session pickle file results : "W:/Data/KDIBA/gor01/two/2006-6-07_16-40-19/loadedSessPickle_withNewComputedReplays.pkl"... done.
# Failure loading W:\Data\KDIBA\gor01\two\2006-6-07_16-40-19\loadedSessPickle_withNewComputedReplays.pkl.
# proposed_load_pkl_path = basedir.joinpath(active_pickle_filename).resolve()

## INPUTS: widget.active_global_pkl, widget.active_global_pkl

if active_session_pickle_file_widget.active_global_pkl is None:
    skip_global_load: bool = True
    override_global_computation_results_pickle_path = None
    print(f'skip_global_load: {skip_global_load}')
else:
    skip_global_load: bool = False
    override_global_computation_results_pickle_path = active_session_pickle_file_widget.active_global_pkl.resolve()
    Assert.path_exists(override_global_computation_results_pickle_path)
    print(f'override_global_computation_results_pickle_path: "{override_global_computation_results_pickle_path}"')

proposed_load_pkl_path = active_session_pickle_file_widget.active_local_pkl.resolve()
Assert.path_exists(proposed_load_pkl_path)
proposed_load_pkl_path

custom_suffix: str = active_session_pickle_file_widget.try_extract_custom_suffix()
print(f'custom_suffix: "{custom_suffix}"')

## OUTPUTS: custom_suffix, proposed_load_pkl_path, (override_global_computation_results_pickle_path, skip_global_load)

In [ ]:
epoch_name_includelist

In [ ]:

## INPUTS: proposed_load_pkl_path
# assert proposed_load_pkl_path.exists(), f"for a saved custom the file must exist, but proposed_load_pkl_path: '{proposed_load_pkl_path}' does not!"

epoch_name_includelist=None
# active_computation_functions_name_includelist=['lap_direction_determination', 'pf_computation','firing_rate_trends', 'position_decoding']
active_computation_functions_name_includelist=[]

with set_posix_windows():
    curr_active_pipeline: NeuropyPipeline = batch_load_session(global_data_root_parent_path, active_data_mode_name, basedir, epoch_name_includelist=epoch_name_includelist,
                                            computation_functions_name_includelist=active_computation_functions_name_includelist,
                                            saving_mode=saving_mode, force_reload=force_reload,
                                            skip_extended_batch_computations=True, debug_print=False, fail_on_exception=False, active_pickle_filename=proposed_load_pkl_path, 
                                            override_parameters_flat_keypaths_dict=override_parameters) # , active_pickle_filename = 'loadedSessPickle_withParameters.pkl'

## Post Compute Validate 2023-05-16:
was_updated = BatchSessionCompletionHandler.post_compute_validate(curr_active_pipeline) ## TODO: need to potentially re-save if was_updated. This will fail because constained versions not ran yet.
print(f'Pipeline loaded from custom pickle!!')
## OUTPUT: curr_active_pipeline


In [ ]:
curr_active_pipeline.get_session_context()


In [ ]:
active_data_mode_name = 'bapun'

# curr_active_pipeline.get_session_additional_parameters_context()
# curr_active_pipeline.session_data_type

DataSessionFormatRegistryHolder.get_registry_known_data_session_type_dict()[active_data_mode_name]
DataSessionFormatRegistryHolder.get_registry_data_session_type_class_name_dict()[active_data_mode_name]




In [6]:
from neuropy.core.session.Formats.BaseDataSessionFormats import HardcodedProcessingParameters
from neuropy.core.session.Formats.Specific.BapunDataSessionFormat import BapunDataSessionFormatRegisteredClass

hardcoded_params: HardcodedProcessingParameters = BapunDataSessionFormatRegisteredClass._get_session_specific_parameters(session_context=curr_active_pipeline.get_session_context())
hardcoded_params

hardcoded_params.decoder_building_session_names
hardcoded_params.non_global_activity_session_names

HardcodedProcessingParameters(decoder_building_session_names: list,
	global_session_name: str,
	non_global_activity_session_names: list,
	grid_bin_bounds: tuple,
	lap_estimation_parameters: dict,
	linearization_parameters: dict
)

['roam', 'sprinkle', 'maze_GLOBAL']

['roam', 'sprinkle']

In [ ]:

@define(slots=False, eq=False, repr=False)
class position_decoding_Parameters(HDF_SerializationMixin, AttrsBasedClassHelperMixin, BaseGlobalComputationParameters):
    """ Docstring for position_decoding_Parameters. 
    """
    override_decoding_time_bin_size: Optional[float] = serialized_attribute_field(default=None)
    ## PARAMS - these are class properties
    override_decoding_time_bin_size_PARAM = param.Number(default=None, doc='override_decoding_time_bin_size param', label='override_decoding_time_bin_size')
    # HDFMixin Conformances ______________________________________________________________________________________________ #
    def to_hdf(self, file_path, key: str, **kwargs):
        """ Saves the object to key in the hdf5 file specified by file_path"""
        super().to_hdf(file_path, key=key, **kwargs)
        



session_specific_parameters: Dict[IdentifyingContext, HardcodedProcessingParameters] = {

    IdentifyingContext(format_name= 'bapun', animal= 'RatN', session_name= 'Day4OpenField'): HardcodedProcessingParameters(decoder_building_session_names=['roam', 'sprinkle', 'maze_GLOBAL'],
                                                                                                                           ),
                                                                                                                        
    IdentifyingContext(format_name= 'bapun', animal= 'RatN', session_name= 'Day5TwoNovel'): HardcodedProcessingParameters(decoder_building_session_names=['maze1', 'maze2', 'maze_GLOBAL'],
                                                                                                                           ),
                                                                                                                    
}


In [ ]:
from pyphoplacecellanalysis.General.PipelineParameterClassTemplating import GlobalComputationParametersAttrsClassTemplating

registered_merged_computation_function_default_kwargs_dict, code_str, nested_classes_dict, (imports_dict, imports_list, imports_string) = GlobalComputationParametersAttrsClassTemplating.main_generate_params_classes(curr_active_pipeline=curr_active_pipeline)

code_str

## from `batch_load_session`

In [ ]:
## From `pyphoplacecellanalysis.General.Batch.NonInteractiveProcessing.batch_load_session` 2025-02-26 09:09 
kwargs = {}
epoch_name_includelist=None
# active_computation_functions_name_includelist=['lap_direction_determination', 'pf_computation','firing_rate_trends', 'position_decoding']
active_computation_functions_name_includelist=[]
override_parameters_flat_keypaths_dict = override_parameters
active_pickle_filename = proposed_load_pkl_path
active_session_computation_configs = None
fail_on_exception: bool = False

saving_mode = PipelineSavingScheme.init(saving_mode)
epoch_name_includelist = kwargs.get('epoch_name_includelist', ['maze1','maze2','maze'])
# epoch_name_includelist = ['maze', 'sprinkle']
# epoch_name_includelist = ['pre', 'maze', 'sprinkle', 'post']


debug_print = kwargs.get('debug_print', False)
assert 'skip_save' not in kwargs, f"use saving_mode=PipelineSavingScheme.SKIP_SAVING instead"
# skip_save = kwargs.get('skip_save', False)
# active_pickle_filename = kwargs.get('active_pickle_filename', 'loadedSessPickle.pkl')

# active_session_computation_configs = kwargs.get('active_session_computation_configs', None)
# computation_functions_name_includelist = kwargs.get('computation_functions_name_includelist', None)

known_data_session_type_properties_dict = DataSessionFormatRegistryHolder.get_registry_known_data_session_type_dict(override_parameters_flat_keypaths_dict=override_parameters_flat_keypaths_dict)
active_data_session_types_registered_classes_dict = DataSessionFormatRegistryHolder.get_registry_data_session_type_class_name_dict()

active_data_mode_registered_class = active_data_session_types_registered_classes_dict[active_data_mode_name]
active_data_mode_type_properties = known_data_session_type_properties_dict[active_data_mode_name]

## Begin main run of the pipeline (load or execute):
curr_active_pipeline = NeuropyPipeline.try_init_from_saved_pickle_or_reload_if_needed(active_data_mode_name, active_data_mode_type_properties,
    override_basepath=Path(basedir), force_reload=force_reload, active_pickle_filename=active_pickle_filename, skip_save_on_initial_load=True, override_parameters_flat_keypaths_dict=override_parameters_flat_keypaths_dict)

curr_active_pipeline.update_parameters(override_parameters_flat_keypaths_dict=override_parameters_flat_keypaths_dict) # should already be updated, but try it again anyway.

was_loaded_from_file: bool =  curr_active_pipeline.has_associated_pickle # True if pipeline was loaded from an existing file, False if it was created fresh

# Get the previous configs:
# curr_active_pipeline.filtered_sessions
# ['filtered_session_names', 'filtered_contexts', 'filtered_epochs', 'filtered_sessions']
# loaded_session_filter_configurations = {k:v.filter_config['filter_function'] for k,v in curr_active_pipeline.active_configs.items()}
# loaded_pipeline_computation_configs = {k:v.computation_config for k,v in curr_active_pipeline.active_configs.items()}


## Build updated ones from the current configs:
active_session_filter_configurations = active_data_mode_registered_class.build_default_filter_functions(sess=curr_active_pipeline.sess, epoch_name_includelist=epoch_name_includelist) # build_filters_pyramidal_epochs(sess=curr_kdiba_pipeline.sess)
if debug_print:
    print(f'active_session_filter_configurations: {active_session_filter_configurations}')

## Skip the filtering, it used to be performed bere but NOT NOW

## TODO 2023-05-16 - set `curr_active_pipeline.active_configs[a_name].computation_config.pf_params.computation_epochs = curr_laps_obj` equivalent
## TODO 2023-05-16 - determine appropriate binning from `compute_short_long_constrained_decoders` so it's automatically from the long


In [ ]:
curr_active_pipeline.filtered_session_names

In [ ]:
# epoch_name_includelist = kwargs.get('epoch_name_includelist', ['maze1','maze2','maze'])
# epoch_name_includelist = ['roam', 'sprinkle']
epoch_name_includelist = ['pre', 'roam', 'sprinkle', 'post']
active_session_filter_configurations = active_data_mode_registered_class.build_default_filter_functions(sess=curr_active_pipeline.sess, epoch_name_includelist=epoch_name_includelist) # build_filters_pyramidal_epochs(sess=curr_kdiba_pipeline.sess)


In [ ]:
fail_on_exception: bool = False

In [ ]:
try:
    curr_active_pipeline.save_pipeline(saving_mode=saving_mode, active_pickle_filename=active_pickle_filename, override_pickle_path=kwargs.get('override_pickle_path', None))
except Exception as e:
    exception_info = sys.exc_info()
    an_error = CapturedException(e, exception_info, curr_active_pipeline)
    print(f'WARNING: Failed to save pipeline via `curr_active_pipeline.save_pipeline(...)` with error: {an_error}')
    if fail_on_exception:
        raise

In [ ]:

if active_session_computation_configs is None:
    """
    If there are is provided computation config, get the default:
    """
    # ## Compute shared grid_bin_bounds for all epochs from the global positions:
    # global_unfiltered_session = curr_active_pipeline.sess
    # # ((22.736279243974774, 261.696733348342), (49.989466271998936, 151.2870218547401))
    # first_filtered_session = curr_active_pipeline.filtered_sessions[curr_active_pipeline.filtered_session_names[0]]
    # # ((22.736279243974774, 261.696733348342), (125.5644705153173, 151.21507349463707))
    # second_filtered_session = curr_active_pipeline.filtered_sessions[curr_active_pipeline.filtered_session_names[1]]
    # # ((71.67666779621361, 224.37820920766043), (110.51617463644946, 151.2870218547401))

    # grid_bin_bounding_session = first_filtered_session
    # grid_bin_bounds = PlacefieldComputationParameters.compute_grid_bin_bounds(grid_bin_bounding_session.position.x, grid_bin_bounding_session.position.y)

    ## OR use no grid_bin_bounds meaning they will be determined dynamically for each epoch:
    # grid_bin_bounds = None
    # time_bin_size = 0.03333 #1.0/30.0 # decode at 30fps to match the position sampling frequency
    # time_bin_size = 0.1 # 10 fps
    time_bin_size = kwargs.get('time_bin_size', 0.03333) # 0.03333 = 1.0/30.0 # decode at 30fps to match the position sampling frequency
    # time_bin_size = kwargs.get('time_bin_size', 0.1) # 10 fps

    # lap_estimation_parameters = curr_active_pipeline.sess.config.preprocessing_parameters.epoch_estimation_parameters.laps
    # assert lap_estimation_parameters is not None
    active_session_computation_configs: List[DynamicContainer] = active_data_mode_registered_class.build_active_computation_configs(sess=curr_active_pipeline.sess, time_bin_size=time_bin_size, override_parameters_flat_keypaths_dict=override_parameters_flat_keypaths_dict) # , grid_bin_bounds=grid_bin_bounds

else:
    # Use the provided `active_session_computation_configs`:
    assert 'time_bin_size' not in kwargs, f"time_bin_size kwarg provided but will not be used because a custom active_session_computation_configs was provided as well."

active_session_computation_configs


In [ ]:
# computation_functions_name_includelist = []
computation_functions_name_includelist = ['pf_computation','firing_rate_trends', 'position_decoding']

In [ ]:

## Setup Computation Functions to be executed:
if computation_functions_name_includelist is None:
    # includelist Mode:
    computation_functions_name_includelist=['_perform_baseline_placefield_computation', '_perform_time_dependent_placefield_computation', '_perform_extended_statistics_computation',
                                        '_perform_position_decoding_computation', 
                                        '_perform_firing_rate_trends_computation',
                                        '_perform_pf_find_ratemap_peaks_computation',
                                        # '_perform_time_dependent_pf_sequential_surprise_computation'
                                        # '_perform_two_step_position_decoding_computation',
                                        # '_perform_recursive_latent_placefield_decoding'
                                    ]  # '_perform_pf_find_ratemap_peaks_peak_prominence2d_computation'
    computation_functions_name_excludelist=None
else:
    print(f'using provided computation_functions_name_includelist: {computation_functions_name_includelist}')
    computation_functions_name_excludelist=None

## For every computation config we build a fake (duplicate) filter config).
# OVERRIDE WITH TRUE:
# curr_active_pipeline.sess.config.preprocessing_parameters.epoch_estimation_parameters.laps['use_direction_dependent_laps'] = True # override with True
lap_estimation_parameters = curr_active_pipeline.sess.config.preprocessing_parameters.epoch_estimation_parameters.laps
assert lap_estimation_parameters is not None
use_direction_dependent_laps: bool = lap_estimation_parameters.get('use_direction_dependent_laps', False) # whether to split the laps into left and right directions
# use_direction_dependent_laps: bool = lap_estimation_parameters.get('use_direction_dependent_laps', True) # whether to split the laps into left and right directions

if (use_direction_dependent_laps or (len(active_session_computation_configs) > 3)):
    lap_direction_suffix_list = ['_odd', '_even', '_any'] # ['maze1_odd', 'maze1_even', 'maze1_any', 'maze2_odd', 'maze2_even', 'maze2_any', 'maze_odd', 'maze_even', 'maze_any']
    # lap_direction_suffix_list = ['_odd', '_even', ''] # no '_any' prefix, instead reuses the existing names
    # assert len(lap_direction_suffix_list) == len(active_session_computation_configs), f"len(lap_direction_suffix_list): {len(lap_direction_suffix_list)}, len(active_session_computation_configs): {len(active_session_computation_configs)}, "
else:
    print(f'not using direction-dependent laps.')
    lap_direction_suffix_list = ['']

# active_session_computation_configs: this should contain three configs, one for each Epoch    
active_session_computation_configs = [deepcopy(a_config) for a_config in active_session_computation_configs]

#TODO 2024-10-30 13:22: - [ ] This is where we should override the params using `override_parameters_flat_keypaths_dict`
# if override_parameters_flat_keypaths_dict is not None:
# 	for a_config in active_session_computation_configs:
# 		for k, v in override_parameters_flat_keypaths_dict.items():
# 			try:
# 				a_config.set_by_keypath(k, deepcopy(v))
# 			except Exception as e:
# 				# raise e
# 				print(f'cannot set_by_keypath: {k} -- error: {e}. Skipping for now.')

assert len(lap_direction_suffix_list) == len(active_session_computation_configs)
updated_active_session_pseudo_filter_configs = {} # empty list, woot!


for a_computation_suffix_name, a_computation_config in zip(lap_direction_suffix_list, active_session_computation_configs): # these should NOT be the same length: lap_direction_suffix_list: ['_odd', '_even', '_any']
    # We need to filter and then compute with the appropriate config iteratively.
    for a_filter_config_name, a_filter_config_fn in active_session_filter_configurations.items():
        # TODO: Build a context:
        a_combined_name: str = f'{a_filter_config_name}{a_computation_suffix_name}'
        # if a_computation_suffix_name != '':
        updated_active_session_pseudo_filter_configs[a_combined_name] = deepcopy(a_filter_config_fn) # this copy is just so that the values are recomputed with the appropriate config. This is a HACK
    # end for filter_configs

    ## Actually do the filtering now. We have 
    curr_active_pipeline.filter_sessions(updated_active_session_pseudo_filter_configs, changed_filters_ignore_list=['maze1','maze2','maze'], debug_print=False)

    ## TODO 2023-01-15 - perform_computations for all configs!!
    #TODO 2024-10-30 13:22: - [ ] This is where we should override the params
    # if override_parameters_flat_keypaths_dict is not None:
    # 	for k, v in override_parameters_flat_keypaths_dict.items():
    # 		a_filter_config_fn.set_by_keypath(k, deepcopy(v))

    # if override_parameters_flat_keypaths_dict is not None:
    # 	curr_active_pipeline.update_parameters(override_parameters_flat_keypaths_dict=override_parameters_flat_keypaths_dict) 


    #TODO 2023-10-31 14:58: - [ ] This is where the computations are being done multiple times!
    #TODO 2023-11-13 14:23: - [ ] With this approach, we can't actually properly filter the computation_configs for the relevant sessions ahead of time because they are calculated for a single computation config but across all sessions at once.
    curr_active_pipeline.perform_computations(a_computation_config, computation_functions_name_includelist=computation_functions_name_includelist, computation_functions_name_excludelist=computation_functions_name_excludelist, fail_on_exception=fail_on_exception, debug_print=debug_print) #, overwrite_extant_results=False  ], fail_on_exception=True, debug_print=False)

    if override_parameters_flat_keypaths_dict is not None:
        curr_active_pipeline.update_parameters(override_parameters_flat_keypaths_dict=override_parameters_flat_keypaths_dict) 




In [ ]:
skip_extended_batch_computations = False
fail_on_exception = True
if not skip_extended_batch_computations:
    batch_extended_computations(curr_active_pipeline, include_global_functions=False, fail_on_exception=fail_on_exception, progress_print=True, debug_print=False)
# curr_active_pipeline.perform_computations(active_session_computation_configs[0], computation_functions_name_excludelist=['_perform_spike_burst_detection_computation'], debug_print=False, fail_on_exception=False) # includelist: ['_perform_baseline_placefield_computation']


try:
    curr_active_pipeline.prepare_for_display(root_output_dir=global_data_root_parent_path.joinpath('Output'), should_smooth_maze=True) # TODO: pass a display config
except Exception as e:
    exception_info = sys.exc_info()
    an_error = CapturedException(e, exception_info, curr_active_pipeline)
    print(f'WARNING: Failed to do `curr_active_pipeline.prepare_for_display(...)` with error: {an_error}')
    if fail_on_exception:
        raise

try:
    curr_active_pipeline.save_pipeline(saving_mode=saving_mode, active_pickle_filename=active_pickle_filename, override_pickle_path=kwargs.get('override_pickle_path', None))
except Exception as e:
    exception_info = sys.exc_info()
    an_error = CapturedException(e, exception_info, curr_active_pipeline)
    print(f'WARNING: Failed to save pipeline via `curr_active_pipeline.save_pipeline(...)` with error: {an_error}')
    if fail_on_exception:
        raise

if not saving_mode.shouldSave:
    print(f'saving_mode.shouldSave == False, so not saving at the end of batch_load_session')

## Load pickled global computations:
# If previously pickled global results were saved, they will typically no longer be relevent if the pipeline was recomputed. We need a system of invalidating/versioning the global results when the other computations they depend on change.
# Maybe move into `batch_extended_computations(...)` or integrate with that somehow
# curr_active_pipeline.load_pickled_global_computation_results()


In [ ]:
from pyphoplacecellanalysis.General.Batch.NonInteractiveProcessing import batch_evaluate_required_computations

skip_global_load = True

In [ ]:
# ==================================================================================================================== #
# Global computations loading:                                                                                            #
# ==================================================================================================================== #
# Loads saved global computations that were saved out via: `custom_save_filepaths['global_computation_pkl'] = curr_active_pipeline.save_global_computation_results(override_global_pickle_filename=custom_save_filenames['global_computation_pkl'])`
## INPUTS: custom_save_filenames
## INPUTS: curr_active_pipeline, (override_global_computation_results_pickle_path, skip_global_load), extended_computations_include_includelist

if skip_global_load:
    override_global_computation_results_pickle_path = None
    print(f'skipping global load because skip_global_load==True')
else:
    # override_global_computation_results_pickle_path = custom_save_filenames['global_computation_pkl']
    print(f'override_global_computation_results_pickle_path: "{override_global_computation_results_pickle_path}"')

# Pre-load ___________________________________________________________________________________________________________ #
force_recompute_global = force_reload
needs_computation_output_dict, valid_computed_results_output_list, remaining_include_function_names = batch_evaluate_required_computations(curr_active_pipeline, include_includelist=extended_computations_include_includelist, include_global_functions=True, fail_on_exception=False, progress_print=True,
                                                    force_recompute=force_recompute_global, force_recompute_override_computations_includelist=force_recompute_override_computations_includelist, debug_print=False)
print(f'Pre-load global computations: needs_computation_output_dict: {[k for k,v in needs_computation_output_dict.items() if (v is not None)]}')
# valid_computed_results_output_list

# Try Unpickling Global Computations to update pipeline ______________________________________________________________ #
if (not force_reload) and (not skip_global_load): # not just force_reload, needs to recompute whenever the computation fails.
    try:
        # INPUTS: override_global_computation_results_pickle_path
        with set_posix_windows():
            sucessfully_updated_keys, successfully_loaded_keys = curr_active_pipeline.load_pickled_global_computation_results(override_global_computation_results_pickle_path=override_global_computation_results_pickle_path,
                                                                                            allow_overwrite_existing=True, allow_overwrite_existing_allow_keys=extended_computations_include_includelist, ) # is new
            print(f'sucessfully_updated_keys: {sucessfully_updated_keys}\nsuccessfully_loaded_keys: {successfully_loaded_keys}')
            did_any_paths_change: bool = curr_active_pipeline.post_load_fixup_sess_basedirs(updated_session_basepath=deepcopy(basedir)) ## use INPUT: basedir
            
    except FileNotFoundError as e:
        exception_info = sys.exc_info()
        e = CapturedException(e, exception_info)
        print(f'cannot load global results because pickle file does not exist! Maybe it has never been created? {e}')
    except Exception as e:
        exception_info = sys.exc_info()
        e = CapturedException(e, exception_info)
        print(f'Unhandled exception: cannot load global results: {e}')
        raise



In [ ]:
print(f'force_reload: {force_reload}, saving_mode: {saving_mode}')
force_reload
saving_mode

In [ ]:
## INPUTS: curr_active_pipeline.global_computation_results_pickle_path, skip_global_load
## indicate that it was loaded with a custom suffix
curr_active_pipeline.pickle_path ## correct
curr_active_pipeline.global_computation_results_pickle_path ## correct

print(f'override_pickle_path = "{curr_active_pipeline.pickle_path}",\nactive_pickle_filename = "{curr_active_pipeline.pickle_path.name}"')
print(f'override_global_pickle_path = "{curr_active_pipeline.global_computation_results_pickle_path}")')

## OUTPUTS: `curr_active_pipeline`  0️⃣ 0️⃣ 0️⃣ 0️⃣ 0️⃣ 0️⃣ 0️⃣ 0️⃣ 0️⃣ 0️⃣ 0️⃣ 0️⃣ 0️⃣ 0️⃣ 0️⃣ 0️⃣ 0️⃣ 0️⃣ 0️⃣ 0️⃣ 0️⃣0️⃣ RESUME Normal Pipeline Load

## 0️⃣ Shared Post-Pipeline load stuff

In [7]:
# BATCH_DATE_TO_USE: str = f'{DAY_DATE_TO_USE}_GL'
# BATCH_DATE_TO_USE: str = f'{DAY_DATE_TO_USE}_rMBP' # TODO: Change this as needed, templating isn't actually doing anything rn.
BATCH_DATE_TO_USE: str = f'{DAY_DATE_TO_USE}_Apogee'
# BATCH_DATE_TO_USE: str = f'{DAY_DATE_TO_USE}_Lab'
 
try:
    if custom_suffix is not None:
        BATCH_DATE_TO_USE = f'{BATCH_DATE_TO_USE}{custom_suffix}'
        print(f'Adding custom suffix: "{custom_suffix}" - BATCH_DATE_TO_USE: "{BATCH_DATE_TO_USE}"')
except NameError as err:
    custom_suffix = None
    print(f'NO CUSTOM SUFFIX.')

known_collected_output_paths = [Path(v).resolve() for v in ['/nfs/turbo/umms-kdiba/Data/Output/collected_outputs', '/home/halechr/FastData/collected_outputs/',
                                                           '/home/halechr/cloud/turbo/Data/Output/collected_outputs',
                                                           r'C:\Users\pho\repos\Spike3DWorkEnv\Spike3D\output\collected_outputs',
                                                           r"K:\scratch\collected_outputs",
                                                           '/Users/pho/data/collected_outputs',
                                                          'output/gen_scripts/']]
collected_outputs_path = find_first_extant_path(known_collected_output_paths)
assert collected_outputs_path.exists(), f"collected_outputs_path: {collected_outputs_path} does not exist! Is the right computer's config commented out above?"
# fullwidth_path_widget(scripts_output_path, file_name_label='Scripts Output Path:')
print(f'collected_outputs_path: {collected_outputs_path}')
# collected_outputs_path.mkdir(exist_ok=True)
# assert collected_outputs_path.exists()

## Build the output prefix from the session context:
active_context = curr_active_pipeline.get_session_context()
curr_session_name: str = curr_active_pipeline.session_name # '2006-6-08_14-26-15'
CURR_BATCH_OUTPUT_PREFIX: str = f"{BATCH_DATE_TO_USE}-{curr_session_name}"
print(f'CURR_BATCH_OUTPUT_PREFIX: "{CURR_BATCH_OUTPUT_PREFIX}"')

Adding custom suffix: "" - BATCH_DATE_TO_USE: "2026-06-22_Apogee"
collected_outputs_path: K:\scratch\collected_outputs
CURR_BATCH_OUTPUT_PREFIX: "2026-06-22_Apogee-RatU_Day5OpenfieldSD_2021-08-04_08-44-31"


# 0️⃣ Pho Interactive Pipeline Jupyter Widget

In [ ]:
import ipywidgets as widgets
from IPython.display import display
from pyphocorehelpers.Filesystem.open_in_system_file_manager import reveal_in_system_file_manager
from pyphoplacecellanalysis.GUI.IPyWidgets.pipeline_ipywidgets import interactive_pipeline_widget, interactive_pipeline_files

_pipeline_jupyter_widget = interactive_pipeline_widget(curr_active_pipeline=curr_active_pipeline)
# display(_pipeline_jupyter_widget)
_pipeline_jupyter_widget

# / 🛑 End Run Section 🛑
-------

# 🎨 2024-02-06 - Other Plotting

In [ ]:
from pyphoplacecellanalysis.Pho2D.PyQtPlots.TimeSynchronizedPlotters.TimeSynchronizedPlacefieldsPlotter import TimeSynchronizedPlacefieldsPlotter

_restore_previous_matplotlib_settings_callback = matplotlib_configuration_update(is_interactive=True, backend='Qt5Agg')

#  Create a new `SpikeRaster2D` instance using `_display_spike_raster_pyqtplot_2D` and capture its outputs:
curr_active_pipeline.reload_default_display_functions()
curr_active_pipeline.prepare_for_display()

## `LauncherWidget`: GUI

In [ ]:
from pyphoplacecellanalysis.General.Pipeline.Stages.Display import DisplayFunctionItem
from pyphocorehelpers.gui.Qt.tree_helpers import find_tree_item_by_text
from pyphoplacecellanalysis.GUI.Qt.MainApplicationWindows.LauncherWidget.LauncherWidget import LauncherWidget

widget = LauncherWidget()
treeWidget = widget.mainTreeWidget # QTreeWidget
widget.build_for_pipeline(curr_active_pipeline=curr_active_pipeline)
widget.show()

In [ ]:
widget.show()

In [ ]:
session_id_str: str = curr_active_pipeline.get_complete_session_identifier_string()
widget.setWindowTitle(f'Spike3D Launcher: {session_id_str}')
treeWidget.root
# curr_active_pipeline.get_session_additional_parameters_context()
# curr_active_pipeline.get_complete_session_context()

In [ ]:
_out = dict()
_out['_display_grid_bin_bounds_validation'] = curr_active_pipeline.display(display_function='_display_grid_bin_bounds_validation', active_session_configuration_context=None) # _display_grid_bin_bounds_validation


In [ ]:
_out = dict()
_out['_display_3d_interactive_tuning_curves_plotter'] = curr_active_pipeline.display(display_function='_display_3d_interactive_tuning_curves_plotter', active_session_configuration_context=IdentifyingContext(format_name='bapun',animal='RatN',session_name='Day4OpenField',filter_name='roam')) # _display_3d_interactive_tuning_curves_plotter


In [ ]:
_out['_display_3d_interactive_tuning_curves_plotter_sprinkle'] = curr_active_pipeline.display(display_function='_display_3d_interactive_tuning_curves_plotter', active_session_configuration_context=IdentifyingContext(format_name='bapun',animal='RatN',session_name='Day4OpenField',filter_name='sprinkle')) # _display_3d_interactive_tuning_curves_plotter


In [ ]:
display_output = _out['_display_3d_interactive_tuning_curves_plotter']
# a_pf_pyvista_plotter = display_output['ipcDataExplorer']
display_output['plotter']

 

In [ ]:
# Check if auto_update is available and enable it
if hasattr(display_output['plotter'], 'auto_update'):
    display_output['plotter'].auto_update = True

In [ ]:
from pyphoplacecellanalysis.Pho3D.PyVista.peak_prominences import render_all_neuron_peak_prominence_2d_results_on_pyvista_plotter

display_output = {}
active_config_name = 'roam'
print(f'active_config_name: {active_config_name}')
active_peak_prominence_2d_results = curr_active_pipeline.computation_results[active_config_name].computed_data.get('RatemapPeaksAnalysis', {}).get('PeakProminence2D', None)
pActiveTuningCurvesPlotter = None
display_output = display_output | curr_active_pipeline.display('_display_3d_interactive_tuning_curves_plotter', active_config_name, extant_plotter=display_output.get('pActiveTuningCurvesPlotter', None), panel_controls_mode='Qt', should_nan_non_visited_elements=False, zScalingFactor=2000.0) # Works now!




In [ ]:
curr_active_pipeline.reload_default_computation_functions()

In [ ]:
## INPUTS: curr_active_pipeline, active_config_name
active_config_name = 'roam'
active_peak_prominence_2d_results = curr_active_pipeline.computation_results[active_config_name].computed_data.get('RatemapPeaksAnalysis', {}).get('PeakProminence2D', None)
if active_peak_prominence_2d_results is None:
    curr_active_pipeline.perform_specific_computation(computation_functions_name_includelist=['ratemap_peaks_prominence2d'], enabled_filter_names=[active_config_name], fail_on_exception=True, debug_print=True)
    # curr_active_pipeline.perform_specific_computation(computation_functions_name_includelist=['ratemap_peaks_prominence2d'], enabled_filter_names=[short_LR_name, short_RL_name, long_any_name, short_any_name], fail_on_exception=False, debug_print=False) # or at least
    active_peak_prominence_2d_results = curr_active_pipeline.computation_results[active_config_name].computed_data.get('RatemapPeaksAnalysis', {}).get('PeakProminence2D', None)
    assert active_peak_prominence_2d_results is not None, f"bad even after computation"

# active_peak_prominence_2d_results

In [ ]:
active_peak_prominence_2d_results

In [ ]:
from pyphoplacecellanalysis.Pho3D.PyVista.peak_prominences import render_all_neuron_peak_prominence_2d_results_on_pyvista_plotter

ipcDataExplorer = display_output['ipcDataExplorer']
if 'pActiveTuningCurvesPlotter' not in display_output:
    display_output['pActiveTuningCurvesPlotter'] = display_output.pop('plotter') # rename the key from the generic "plotter" to "pActiveSpikesBehaviorPlotter" to avoid collisions with others
pActiveTuningCurvesPlotter = display_output['pActiveTuningCurvesPlotter']
root_dockAreaWindow, placefieldControlsContainerWidget, pf_widgets = display_output['pane'] # for Qt mode

active_peak_prominence_2d_results = curr_active_pipeline.computation_results[active_config_name].computed_data.get('RatemapPeaksAnalysis', {}).get('PeakProminence2D', None)
render_all_neuron_peak_prominence_2d_results_on_pyvista_plotter(ipcDataExplorer, active_peak_prominence_2d_results, debug_print=True, 
    include_contour_bounding_box=True,
    include_text_labels=True,
    )

In [ ]:
# Now you can toggle them
neuron_id = 4
peaks = ipcDataExplorer.plots['tuningCurvePlotActors'][neuron_id].peaks
peaks.contours.SetVisibility(1)  # Show contours
peaks.boxes.SetVisibility(1)
peaks.text.SetVisibility(1)
peaks.peak_points.SetVisibility(1)
ipcDataExplorer.p.render()

In [ ]:
# Access peaks for a specific neuron
neuron_id = 9
peaks = ipcDataExplorer.plots['tuningCurvePlotActors'][neuron_id].peaks

# # Toggle individual categories
# peaks.contours.SetVisibility(1)  # Show contours
# peaks.boxes.SetVisibility(1)     # Hide bounding boxes
# peaks.text.SetVisibility(1)       # Hide text labels
# peaks.peak_points.SetVisibility(1) # Show peak points

# Or toggle all at once
peaks.SetVisibility(1)  # Show everything
# peaks.SetVisibility(0)   # Hide everything

# Update the display
ipcDataExplorer.p.render()

In [ ]:
# type(ipcDataExplorer.plots['tuningCurvePlotActors'])
dict(ipcDataExplorer.plots['tuningCurvePlotActors'])


In [ ]:
for neuron_id, a_neuron_impl in dict(ipcDataExplorer.plots['tuningCurvePlotActors']).items():
    if a_neuron_impl is not None:
        print(neuron_id)
        print(a_neuron_impl)

        peaks = a_neuron_impl.peaks
        # peaks.SetVisibility(0)   # Hide everything
        peaks.contours.SetVisibility(1)  # Show contours
        # peaks.boxes.SetVisibility(1)     # Hide bounding boxes
        # peaks.text.SetVisibility(1)       # Hide text labels
        # peaks.peak_points.SetVisibility(1) # Show peak points


In [ ]:
active_peak_prominence_2d_results.filtered_flat_peaks_df

In [ ]:
active_peak_prominence_2d_results.flat_peaks_df

In [ ]:
from pyphoplacecellanalysis.Pho3D.PyVista.peak_prominences import render_all_neuron_peak_prominence_2d_results_on_pyvista_plotter

display_output = _out['_display_3d_interactive_tuning_curves_plotter']
ipcDataExplorer = display_output['ipcDataExplorer']
if 'pActiveTuningCurvesPlotter' not in display_output:
    display_output['pActiveTuningCurvesPlotter'] = display_output.pop('plotter') # rename the key from the generic "plotter" to "pActiveSpikesBehaviorPlotter" to avoid collisions with others
pActiveTuningCurvesPlotter = display_output['pActiveTuningCurvesPlotter']
root_dockAreaWindow, placefieldControlsContainerWidget, pf_widgets = display_output['pane'] # for Qt mode

# slab_results_dict: Dict[Tuple, SlabResult] = {k:SlabResult(**a_slab_result_dict) for k, a_slab_result_dict in simplified_obj.results.items()}

# active_peak_prominence_2d_results = curr_active_pipeline.computation_results[active_config_name].computed_data.get('RatemapPeaksAnalysis', {}).get('PeakProminence2D', None)

active_peak_prominence_2d_results = simplified_obj # slab_results_dict: Dict[Tuple, SlabResult] = {k:SlabResult(**a_slab_result_dict) for k, a_slab_result_dict in simplified_obj.results.items()}
render_all_neuron_peak_prominence_2d_results_on_pyvista_plotter(ipcDataExplorer, active_peak_prominence_2d_results)

## ✅ 2025-09-19 - Clean programmmatic figure outputs 

In [ ]:
from pyphocorehelpers.plotting.figure_management import PhoActiveFigureManager2D, capture_new_figures_decorator
fig_man = PhoActiveFigureManager2D(name=f'fig_man') # Initialize a new figure manager
from pyphoplacecellanalysis.GUI.PyQtPlot.Widgets.DockAreaWrapper import DockAreaWrapper
from pyphoplacecellanalysis.General.Mixins.ExportHelpers import programmatic_render_to_file, programmatic_display_to_PDF, extract_figures_from_display_function_output

# fig_man.close_all()

# subset_includelist = ['maze1', 'maze2', 'maze_GLOBAL'] # Day5TwoNovel
# subset_includelist = ['roam', 'sprinkle'] # Day4

subset_includelist = hardcoded_params.decoder_building_session_names
print(f'subset_includelist: {subset_includelist}')

In [ ]:
# display_fn_kwargs = dict(subplots=(None, 9))
display_fn_kwargs = dict(subplots=(None, 7))

# _out = dict()
# _out['_display_2d_placefield_result_plot_ratemaps_2D'] = curr_active_pipeline.display(display_function='_display_2d_placefield_result_plot_ratemaps_2D', active_session_configuration_context=IdentifyingContext(format_name='bapun',animal='RatS',session_name='Day5TwoNovel',filter_name='maze1'), **display_fn_kwargs) # _display_2d_placefield_result_plot_ratemaps_2D
# _out['_display_2d_placefield_result_plot_ratemaps_2D'] = curr_active_pipeline.display(display_function='_display_2d_placefield_result_plot_ratemaps_2D', active_session_configuration_context=IdentifyingContext(format_name='bapun',animal='RatS',session_name='Day5TwoNovel',filter_name='maze2'), **display_fn_kwargs) # _display_2d_placefield_result_plot_ratemaps_2D


In [ ]:
_out_list = programmatic_render_to_file(curr_active_pipeline=curr_active_pipeline, curr_display_function_name='_display_2d_placefield_result_plot_ratemaps_2D', subset_includelist=subset_includelist, 
                                        write_vector_format=True, write_png=True, debug_print=True, **display_fn_kwargs)


In [ ]:
_out_list = programmatic_render_to_file(curr_active_pipeline=curr_active_pipeline, curr_display_function_name='_display_2d_placefield_occupancy', subset_includelist=subset_includelist, 
                                        write_vector_format=True, write_png=True, debug_print=True)


## `Spike3DRasterWindowWidget` Cell

In [ ]:
from neuropy.utils.mixins.time_slicing import TimeColumnAliasesProtocol
from neuropy.core.flattened_spiketrains import SpikesAccessor
from pyphoplacecellanalysis.GUI.PyQtPlot.Widgets.SpikeRasterWidgets.Spike2DRaster import Spike2DRaster
from pyphoplacecellanalysis.GUI.PyQtPlot.Widgets.helpers import ScrollableRasterViewOwnerMixin
from pyphoplacecellanalysis.GUI.Qt.SpikeRasterWindows.Spike3DRasterWindowWidget import Spike3DRasterWindowWidget
# from pyphoplacecellanalysis.SpecificResults.PendingNotebookCode import _setup_spike_raster_window_for_debugging
from pyphoplacecellanalysis.GUI.PyQtPlot.Widgets.Mixins.Render2DScrollWindowPlot import ScatterItemData # used in `NewSimpleRaster`

active_spikes_df = deepcopy(curr_active_pipeline.sess.spikes_df)

# INLINEING `build_spikes_data_values_from_df`: ______________________________________________________________________ #
# curr_spike_x, curr_spike_y, curr_spike_pens, all_scatterplot_tooltips_kwargs, all_spots, curr_n = cls.build_spikes_data_values_from_df(spikes_df, config_fragile_linear_neuron_IDX_map, is_spike_included=is_spike_included, should_return_data_tooltips_kwargs=should_return_data_tooltips_kwargs, **kwargs)
# All units at once approach:
active_time_variable_name = active_spikes_df.spikes.time_variable_name
print(f'active_time_variable_name: {active_time_variable_name}')
if active_time_variable_name != 't': 
    active_spikes_df = TimeColumnAliasesProtocol.renaming_synonym_columns_if_needed(active_spikes_df, required_columns_synonym_dict={"t":{active_time_variable_name,'t_rel_seconds', 't_seconds'}})
    active_spikes_df = active_spikes_df.drop(columns=[active_time_variable_name], inplace=False) ## drop the old column    
    active_time_variable_name = 't' ## get the new one
    active_spikes_df.spikes.set_time_variable_name('t')
    # default_datapoint_column_names = [active_spikes_df.spikes.time_variable_name, 'aclu', 'fragile_linear_neuron_IDX']
    # active_datapoint_column_names = default_datapoint_column_names
    active_spikes_df
    
# active_spikes_df.spikes.time_variable_name
# active_spikes_df

# Gets the existing SpikeRasterWindow or creates a new one if one doesn't already exist:
# spike_raster_window, (active_2d_plot, active_3d_plot, main_graphics_layout_widget, main_plot_widget, background_static_scroll_plot_widget) = Spike3DRasterWindowWidget.find_or_create_if_needed(curr_active_pipeline, force_create_new=True, allow_replace_hardcoded_main_plots_with_tracks=True)
# spike_raster_window, (active_2d_plot, active_3d_plot, main_graphics_layout_widget, main_plot_widget, background_static_scroll_plot_widget) = Spike3DRasterWindowWidget.find_or_create_if_needed(curr_active_pipeline, force_create_new=False, allow_replace_hardcoded_main_plots_with_tracks=True)
# spike_raster_window, (active_2d_plot, active_3d_plot, *_all_outputs_dict) = Spike3DRasterWindowWidget.find_or_create_if_needed(curr_active_pipeline, force_create_new=False, allow_replace_hardcoded_main_plots_with_tracks=True)
# spike_raster_window, (active_2d_plot, active_3d_plot, *_all_outputs_dict) = Spike3DRasterWindowWidget.find_or_create_if_needed(curr_active_pipeline, force_create_new=True, allow_replace_hardcoded_main_plots_with_tracks=True, active_session_configuration_context='maze_GLOBAL')
spike_raster_window, (active_2d_plot, active_3d_plot, *_all_outputs_dict) = Spike3DRasterWindowWidget.find_or_create_if_needed(curr_active_pipeline, force_create_new=True, allow_replace_hardcoded_main_plots_with_tracks=True, active_session_configuration_context='maze_GLOBAL')

In [ ]:
pos_df = curr_active_pipeline.filtered_sessions['maze_GLOBAL'].position.df.copy()
pos_df.plot(x='t', y=['x', 'y', 'z'])


In [ ]:
## can you figure this out?
## Why are pf2D missing? They usually aren't, just missing from the fake `maze_GLOBAL_GLOBAL` entry




In [ ]:
from pyphoplacecellanalysis.SpecificResults.PendingNotebookCode import build_proper_epoch_intervals, build_bapun_all_epochs_df

a_rect_item, an_interval_ds = build_proper_epoch_intervals(curr_active_pipeline=curr_active_pipeline, active_2d_plot=active_2d_plot)
curr_paradigm_df: pd.DataFrame = deepcopy(an_interval_ds._df)
curr_paradigm_df

# curr_paradigm_df: pd.DataFrame = deepcopy(an_interval_ds._df).epochs.get_valid_df()
# curr_paradigm_df

In [ ]:
epoch_display_configs = active_2d_plot.extract_interval_display_config_lists()
epoch_display_configs

In [ ]:
from neuropy.core.epoch import Epoch, EpochsAccessor, ensure_dataframe, ensure_Epoch, EpochHelpers

out_col: str = 'overlap_y_offset'
series_vertical_offset_col = 'series_vertical_offset'
series_height_col = 'series_height'
cummulative_height: float = 1.0
intra_series_y_spacing: float = 0.1
curr_paradigm_df = EpochHelpers.assign_overlap_y_offset(df=curr_paradigm_df, start_col='t_start', stop_col='t_end', out_col=out_col)
unique_overlap_y_offsets = np.unique(curr_paradigm_df[out_col])
n_unique: int = len(unique_overlap_y_offsets)
# total_y_padding: float = n_unique

## update output columns
curr_paradigm_df[series_height_col] = (cummulative_height / n_unique) - (intra_series_y_spacing / n_unique)
curr_paradigm_df[series_vertical_offset_col] = (-1.0 * cummulative_height) + (curr_paradigm_df[out_col].astype(float) * curr_paradigm_df[series_height_col]) + (curr_paradigm_df[series_height_col] * (intra_series_y_spacing / n_unique))
curr_paradigm_df

# _BAK_curr_paradigm_df = deepcopy(curr_paradigm_df)


In [ ]:
import shutil
from neuropy.core.epoch import Epoch, ensure_dataframe, ensure_Epoch, EpochsAccessor


def _FIXUP_Bapun_RatU_paradigm_epoch_times(curr_paradigm_df: pd.DataFrame, start_col: str = 'start', stop_col: str = 'stop') -> pd.DataFrame: # 
    """ fixes epoch times given manually observed corrections 
    """

    roam_row_idx: int = np.where(curr_paradigm_df['label'] == 'roam')[0][0]
    sprinkle_row_idx: int = np.where(curr_paradigm_df['label'] == 'sprinkle')[0][0]

    ## 2026-05-06 - Corrected roam/sprinkle start and end times
    corrected_paradigm_epoch_records = [
        ['roam', 8031, 10421],
        ['sprinkle', 10477, 11745],
    ]
    corrected_paradigm_epoch_df: pd.DataFrame = pd.DataFrame.from_records(corrected_paradigm_epoch_records, columns=['label', start_col, stop_col])
    # corrected_paradigm_epoch_df

    curr_paradigm_df.iat[roam_row_idx, 0] = corrected_paradigm_epoch_df[corrected_paradigm_epoch_df['label'] == 'roam'][start_col].iloc[0]
    curr_paradigm_df.iat[roam_row_idx, 1] = corrected_paradigm_epoch_df[corrected_paradigm_epoch_df['label'] == 'roam'][stop_col].iloc[0]

    curr_paradigm_df.iat[sprinkle_row_idx, 0] = corrected_paradigm_epoch_df[corrected_paradigm_epoch_df['label'] == 'sprinkle'][start_col].iloc[0]
    curr_paradigm_df.iat[sprinkle_row_idx, 1] = corrected_paradigm_epoch_df[corrected_paradigm_epoch_df['label'] == 'sprinkle'][stop_col].iloc[0]

    duration_col_name = 't_duration' if 't_duration' in curr_paradigm_df.columns else 'duration'
    curr_paradigm_df[duration_col_name] = curr_paradigm_df[stop_col] - curr_paradigm_df[start_col]

    curr_paradigm_df = curr_paradigm_df.reset_index(drop=True, inplace=False)
    # _time_column_name_synonyms = {"start":{'begin','start','start_t'},
    #     'stop':['end','stop','stop_t'],
    #     "t_duration":['duration'],
    # }
    # curr_paradigm_df = TimeColumnAliasesProtocol.renaming_synonym_columns_if_needed(df=curr_paradigm_df, required_columns_synonym_dict=_time_column_name_synonyms)

    return curr_paradigm_df


## INPUTS: curr_active_pipeline
if not hasattr(curr_active_pipeline.sess, '_BAK_paradigm'):
    curr_active_pipeline.sess._BAK_paradigm = deepcopy(curr_active_pipeline.sess.paradigm) ## make backup of existing epochs/paradigm
    
curr_paradigm_df: pd.DataFrame = ensure_dataframe(curr_active_pipeline.sess.paradigm)
curr_paradigm_df = _FIXUP_Bapun_RatU_paradigm_epoch_times(curr_paradigm_df=curr_paradigm_df)
curr_paradigm_df

did_change: bool = np.all(curr_active_pipeline.sess._BAK_paradigm != curr_paradigm_df)
# did_change: bool = np.all(curr_active_pipeline.sess.paradigm != curr_paradigm_df)

## did_change
print(f'did_change: {did_change}')
if did_change:
    
    def get_resolved_paradigm_path(sess) -> Path:
        return sess.basepath.joinpath(f'{sess.session_name}.paradigm.npy').resolve()
        # for resolved_path, spec in sess.config.resolved_required_filespecs_dict.items():
            # if spec.filename.endswith(".paradigm.npy"):
                # return Path(resolved_path).resolve()
        # raise FileNotFoundError("No .paradigm.npy in resolved_required_filespecs_dict for this format")

    # paradigm_path = getattr(curr_active_pipeline.sess.paradigm, "filename", None)
    fn = getattr(curr_active_pipeline.sess.paradigm, "filename", None)
    paradigm_path = Path(fn).resolve() if fn is not None else get_resolved_paradigm_path(curr_active_pipeline.sess)
    ## just replace path
    print(f'paradigm_path: "{paradigm_path.as_posix()}"')

    backup_path = paradigm_path.with_suffix(paradigm_path.suffix + ".pre_edit.bak")  # e.g. *.npy.pre_edit.bak
    if paradigm_path.is_file() and not backup_path.is_file():
        print(f'creating backup of original paradigm file: "{paradigm_path}" -> "{backup_path}"...')
        shutil.copy2(paradigm_path, backup_path)
        print(f'\tdone.')



    # ==================================================================================================================================================================================================================================================================================== #
    # overwrite the old value:                                                                                                                                                                                                                                                             #
    # ==================================================================================================================================================================================================================================================================================== #
    curr_active_pipeline.sess.paradigm = curr_paradigm_df
    # curr_active_pipeline.sess.epochs = curr_paradigm_df

    ## INPUTS: paradigm_path
    # paradigm_path = Path(r"...") / f"{curr_active_pipeline.sess.session_name}.paradigm.npy"  # or sess.config.resolved path to that file
    modified = Epoch(curr_paradigm_df.copy())  # validates/normalizes via Epoch ctor
    modified.metadata = getattr(curr_active_pipeline.sess.paradigm, "metadata", None)
    modified.filename = paradigm_path.resolve()    
    modified.save(status_print=True)



In [ ]:
np.where(curr_paradigm_df['label'] == 'roam')[0][0]


In [ ]:
# corrected_paradigm_epoch_df['label']
corrected_paradigm_epoch_df[corrected_paradigm_epoch_df['label'] == 'roam']['t_start'].loc[0]



In [ ]:
active_2d_plot.update_epochs_from_configs_widget()
# update_rendered_intervals_visualization_properties

In [ ]:
# Epoch Display Configs Update Dictionary
epoch_display_configs_update_dict = {
    'SessionEpochs': dict(y_location=-1.0, height=0.9, pen_color='#490000', pen_opacity=1.0, brush_color='#f51616', brush_opacity=1.0),
    'custom_paradigm': [
        dict(y_location=0.0, height=1.0, pen_color='#ffffff', pen_opacity=1.0, brush_color='#1f77b4', brush_opacity=1.0),
        dict(y_location=0.0, height=1.0, pen_color='#ffffff', pen_opacity=1.0, brush_color='#d62728', brush_opacity=1.0),
        dict(y_location=0.0, height=1.0, pen_color='#ffffff', pen_opacity=1.0, brush_color='#f7b6d2', brush_opacity=1.0),
        dict(y_location=0.0, height=1.0, pen_color='#ffffff', pen_opacity=1.0, brush_color='#9edae5', brush_opacity=1.0),
    ],
    'Laps': dict(y_location=-2.0, height=0.9, pen_color='#ff0000', pen_opacity=1.0, brush_color='#ff0000', brush_opacity=1.0),
    'PBEs': dict(y_location=-3.0, height=0.9, pen_color='#ffffff', pen_opacity=1.0, brush_color='#808080', brush_opacity=1.0),
}

In [ ]:
# Epoch Display Configs Update Dictionary
epoch_display_configs_update_dict = {
    'epoch_1': dict(y_location=-12.0, height=7.5, pen_color='#00ffff', pen_opacity=0.8, brush_color='#00ffff', brush_opacity=0.5),
    'epoch_2': dict(y_location=-12.0, height=7.5, pen_color='#00ffff', pen_opacity=0.8, brush_color='#00ffff', brush_opacity=0.5),
    'epoch_3': [
        dict(y_location=-12.0, height=7.5, pen_color='#00ffff', pen_opacity=0.8, brush_color='#00ffff', brush_opacity=0.5),
        dict(y_location=-12.0, height=7.5, pen_color='#00ffff', pen_opacity=0.8, brush_color='#00ffff', brush_opacity=0.5),
        dict(y_location=-12.0, height=7.5, pen_color='#00ffff', pen_opacity=0.8, brush_color='#00ffff', brush_opacity=0.5),
        dict(y_location=-12.0, height=7.5, pen_color='#00ffff', pen_opacity=0.8, brush_color='#00ffff', brush_opacity=0.5),
    ],
    'epoch_4': dict(y_location=-12.0, height=7.5, pen_color='#00ffff', pen_opacity=0.8, brush_color='#00ffff', brush_opacity=0.5),
}

In [ ]:
# preview_overview_scatter_plot: pg.ScatterPlotItem  = active_2d_plot.plots.preview_overview_scatter_plot # ScatterPlotItem 
# preview_overview_scatter_plot.setDownsampling(auto=True, method='subsample', dsRate=10)
# main_graphics_layout_widget: pg.GraphicsLayoutWidget = active_2d_plot.ui.main_graphics_layout_widget
wrapper_layout: pg.QtWidgets.QVBoxLayout = active_2d_plot.ui.wrapper_layout
# main_content_splitter = active_2d_plot.ui.main_content_splitter # QSplitter
layout = active_2d_plot.ui.layout
background_static_scroll_window_plot = active_2d_plot.plots.background_static_scroll_window_plot # PlotItem
main_plot_widget = active_2d_plot.plots.main_plot_widget # PlotItem
# active_window_container_layout = active_2d_plot.ui.active_window_container_layout # GraphicsLayout, first item of `main_graphics_layout_widget` -- just the active raster window I think, there is a strange black space above it

In [ ]:
a_menu = active_2d_plot._menuContextAddRenderable
a_menu

In [ ]:
print_keys_if_possible('active_2d_plot', active_2d_plot, max_depth=1)

In [ ]:
active_2d_plot.get_leaf_only_flat_dock_identifiers_list()


In [ ]:

identifer_str: str = 'intervals'
# identifer_str: str = 'new_curves_separate_plot'
# identifer_str: str = 'newDockedWidget'
a_dock, widget = active_2d_plot.find_dock_item_tuple(identifer_str)

In [ ]:
widget

In [ ]:
root_graphics_layout_widget = widget.getRootGraphicsLayoutWidget()
root_graphics_layout_widget
plot_item = widget.getRootPlotItem()
plot_item

In [ ]:
active_parent_menu = widget.menu
active_parent_menu

In [ ]:
active_2d_plot._menuContextAddRenderable

In [ ]:
from PyQt5.QtWidgets import QMenu, QAction
from pyphoplacecellanalysis.GUI.Qt.Menus.LocalMenus_AddRenderable.LocalMenus_AddRenderable import LocalMenus_AddRenderable


# Create your custom menu
custom_menu = QMenu("My Custom Menu")
action1 = QAction("Action 1", custom_menu)
action1.triggered.connect(lambda: print("Action 1 clicked"))
custom_menu.addAction(action1)

action2 = QAction("Action 2", custom_menu)
action2.triggered.connect(lambda: print("Action 2 clicked"))
custom_menu.addAction(action2)

action3 = QAction("Action 3", custom_menu)
action3.triggered.connect(lambda: print("Action 3 clicked"))
custom_menu.addAction(action3)

action4 = QAction("Action 4", custom_menu)
action4.triggered.connect(lambda: print("Action 4 clicked"))
custom_menu.addAction(action4)


# LocalMenus_AddRenderable._helper_append_custom_menu_to_widget_context_menu_universal(parent_widget=new_curves_separate_plot, additional_menu=self._menuContextAddRenderable)
parent_widget = widget # works for MatplotlibTimeSynchronizedWidget 

LocalMenus_AddRenderable._helper_append_custom_menu_to_widget_context_menu_universal(parent_widget=widget, additional_menu=custom_menu, debug_print=True)



In [ ]:

# For a PlotItem
# plot_item = pg.PlotItem()
## INPUTS: plot_item
# Option A: Ensure menu is enabled (if it was disabled)
if hasattr(plot_item, 'vb'):
    plot_item.vb.setMenuEnabled(True)  # This will create the menu if it doesn't exist
    
    # Now check if menu exists
    if plot_item.vb.menu is not None:
        plot_item.vb.menu.addSeparator()
        plot_item.vb.menu.addMenu(custom_menu)
    else:
        print("Warning: Menu is still None after enabling")


In [ ]:

# For a ViewBox directly
# viewbox = pg.ViewBox()
viewbox = plot_item.vb
if hasattr(viewbox, 'menu'):
    viewbox.menu.addSeparator()
    viewbox.menu.addAction(action1)
    

In [ ]:

# For a ViewBox directly
viewbox = pg.ViewBox()
if hasattr(viewbox, 'menu'):
    viewbox.menu.addSeparator()
    viewbox.menu.addAction(action1)

In [ ]:
import PyQtInspect as pyqtinsp

pyqtinsp.pqi.set_widget_highlight(widget=active_2d_plot, highlight=True)


In [ ]:
type(active_2d_plot)

In [ ]:
active_2d_plot.ui


In [ ]:
print_keys_if_possible('active_2d_plot.ui', active_2d_plot.ui, max_depth=2)

In [ ]:
type(spike_raster_window)

In [ ]:
spike_raster_window.ui.wrapper

In [ ]:
active_2d_plot.add_docked_marginal_track(curr_active_pipeline.sess.epochs)
    

In [ ]:
# active_2d_plot.list_all_rendered_intervals()
active_2d_plot.add_laps_intervals(curr_active_pipeline.sess)

active_2d_plot.add_rendered_intervals()



In [ ]:
# curr_active_pipeline.sess.epochs




In [ ]:
from pyphoplacecellanalysis.GUI.PyQtPlot.Widgets.Mixins.RenderTimeEpochs.Specific2DRenderTimeEpochs import SessionEpochs2DRenderTimeEpochs
from neuropy.core.epoch import ensure_dataframe, ensure_Epoch, Epoch, EpochsAccessor
# SessionEpochs2DRenderTimeEpochs.add_render_time_epochs(curr_sess=curr_active_pipeline.sess.epochs, destination_plot=active_2d_plot)

active_ds = ensure_dataframe(curr_active_pipeline.sess.epochs)
active_ds

_out = active_2d_plot.add_rendered_intervals(active_ds, 'SessionEpochs')

# num_epochs = len(curr_active_pipeline.sess.epochs)

# pen_colors = {'pre': pg.mkColor('purple'), 'roam': pg.mkColor('red'), 'sprinkle': pg.mkColor('red'), 'post': pg.mkColor('purple')}
# brush_colors = {'pre': pg.mkColor('purple'), 'roam': pg.mkColor('red'), 'sprinkle': pg.mkColor('red'), 'post': pg.mkColor('purple')}

# pen_color = list(pen_colors.values())
# brush_color = list(brush_colors.values())
# for a_pen_color in pen_color:
#     a_pen_color.setAlphaF(0.8)

# for a_brush_color in brush_color:
#     a_brush_color.setAlphaF(0.5)


# active_df = cls._update_df_visualization_columns(active_df, y_location, height, pen_color, brush_color, **kwargs)


In [ ]:
active_2d_plot.rendered_epochs


In [ ]:
active_ds = ensure_dataframe(curr_active_pipeline.sess.epochs)
global_epoch_only = ensure_Epoch(active_ds[active_ds['label'] == 'maze_GLOBAL'])
global_epoch_only

curr_active_pipeline.sess
sess.position

## 2025-09-19 - Add Session Paradigm Epochs with a different color for each session

In [ ]:
## 2025-09-19 - Add Session Paradigm Epochs with a different color for each session
from pyphoplacecellanalysis.SpecificResults.PendingNotebookCode import build_proper_epoch_intervals, build_bapun_all_epochs_df

a_rect_item, an_interval_ds = build_proper_epoch_intervals(curr_active_pipeline=curr_active_pipeline, active_2d_plot=active_2d_plot)


In [ ]:
an_interval_ds

In [ ]:
a_rect_item

In [ ]:
active_2d_plot: Spike2DRaster = active_2d_plot

In [ ]:
# active_2d_plot.perform_remove_epoch_intervals('SessionEpochs') ## not correct


In [ ]:
active_2d_plot.remove_rendered_intervals('SessionEpochs')


In [ ]:
# curr_x_min, curr_x_max, curr_y_min, curr_y_max = active_2d_plot.get_render_intervals_plot_range()
# (curr_x_min, curr_x_max, curr_y_min, curr_y_max) # (0.14663333333333334, 25986.969433333332, -5.0, 41.0)

In [ ]:
all_series_positioning_dfs, all_series_compressed_positioning_dfs, all_series_compressed_positioning_update_dicts = active_2d_plot.recover_interval_datasources_update_dict_properties()
# all_series_positioning_dfs
all_series_compressed_positioning_dfs # ERROR: series_compressed_positioning_update_dict is None for custom_paradigm. it will not be represented in the output dict.

In [ ]:
out_configs_df = active_2d_plot.extract_interval_display_config_df()
out_configs_df

In [ ]:
from pyphoplacecellanalysis.General.Model.Datasources.IntervalDatasource import IntervalsDatasource
from pyphoplacecellanalysis.PhoPositionalData.plotting.mixins.epochs_plotting_mixins import EpochDisplayConfig
from pyphoplacecellanalysis.GUI.PyQtPlot.Widgets.helpers import RectangleRenderTupleHelpers
from pyphoplacecellanalysis.GUI.PyQtPlot.Widgets.Mixins.RenderTimeEpochs.Render2DEventRectanglesHelper import Render2DEventRectanglesHelper
from pyphocorehelpers.gui.Qt.color_helpers import ColorDataframeColumnHelpers, ColorFormatConverter, QColorColumnsAccessor

custom_paradigm_ds: IntervalsDatasource = active_2d_plot.interval_datasources.custom_paradigm
custom_paradigm_ds


In [ ]:
out_configs_df = active_2d_plot.extract_interval_display_config_df()
out_configs_df
added_col_names_map = out_configs_df.qcolor.convert_QColor_columns_to_hexcolor_columns()
# added_col_names_map
out_configs_df

In [ ]:
out_configs_dict = active_2d_plot.extract_interval_display_config_lists()
out_configs_dict

In [ ]:
custom_paradigm_ds_df = deepcopy(custom_paradigm_ds._df)  # [''
custom_paradigm_ds_df

# ['lap_color', 'lap_accent_color'] ## str (hex-color), str (hex-color)
# ['pen_color', 'brush_color'] ## QColor, QColor
# ['pen', 'brush'] # QPen, QBrush


In [ ]:

custom_paradigm_ds_df.qcolor.find_valid_hex_columns()


In [ ]:

added_col_names_map = custom_paradigm_ds_df.qcolor.convert_QColor_columns_to_hexcolor_columns()
added_col_names_map
# initial_labels: List[str] = deepcopy(list(custom_paradigm_ds_df.columns))

# def is_valid_hex_label(a_label: str) -> bool:
# 	if a_label.startswith('#')
    
# ColorFormatConverter.is_valid_hexstring(

# extant_hex_color_labels = [k for k in initial_labels if k.endswith('_hex')]

# extant_hex_color_labels


In [ ]:
custom_paradigm_ds_df

In [ ]:
custom_paradigm_ds_df['pen'] = 

In [ ]:
series_viz_df = deepcopy(custom_paradigm_ds_df)
series_viz_df['pen_color_hex'] = series_viz_df['pen'].map(lambda x: "#" + ColorDataframeColumnHelpers.QPen_to_dict(x)['color']).str.upper()
series_viz_df['pen_width'] = series_viz_df['pen'].map(lambda x: ColorDataframeColumnHelpers.QPen_to_dict(x)['width'])
series_viz_df['brush_color_hex'] = series_viz_df['brush'].map(lambda x: "#" + ColorDataframeColumnHelpers.QBrush_to_dict(x)['color']).str.upper()
# series_viz_df['brush_color_hex'] = series_viz_df['brush_color_hex'].str.upper()
series_viz_df

In [ ]:
from pyphoplacecellanalysis.GUI.PyQtPlot.Widgets.Mixins.RenderTimeEpochs.Render2DEventRectanglesHelper import Render2DEventRectanglesHelper
from pyphoplacecellanalysis.GUI.Qt.Widgets.EpochRenderConfigWidget.EpochRenderConfigWidget import EpochRenderConfigsListWidget, EpochRenderConfigWidget

an_epochs_display_list_widget: EpochRenderConfigsListWidget = active_2d_plot.ui.get('epochs_render_configs_widget', None)
if an_epochs_display_list_widget is None:
    # create a new one:    
    raise NotImplementedError

update_dict = an_epochs_display_list_widget.config_dicts_from_states()
update_dict


In [ ]:
# brush_colors = [v['brush_color'] for v in update_dict['custom_paradigm']]
# pen_colors = [v['pen_color'] for v in update_dict['custom_paradigm']]

brush_colors_hex = [ColorFormatConverter.qColor_to_hexstring(v['brush_color'], include_alpha=True, use_HexArgb_instead_of_HexRGBA=False) for v in update_dict['custom_paradigm']]
pen_colors_hex = [ColorFormatConverter.qColor_to_hexstring(v['pen_color'], include_alpha=True, use_HexArgb_instead_of_HexRGBA=False) for v in update_dict['custom_paradigm']]



pen_colors_hex
brush_colors_hex

# pen_colors
# brush_colors


In [ ]:
from neuropy.utils.misc import split_list_of_dicts
from pyphoplacecellanalysis.GUI.PyQtPlot.Widgets.GraphicsObjects.IntervalRectsItem import IntervalRectsItem, IntervalRectsItemData
from pyphoplacecellanalysis.GUI.PyQtPlot.Widgets.Mixins.RenderTimeEpochs.Specific2DRenderTimeEpochs import General2DRenderTimeEpochs


def _fixed_for_multi_update_df_visualization_columns(active_df: pd.DataFrame, y_location=None, height=None, pen_color=None, brush_color=None, **kwargs) -> pd.DataFrame:
        """ updates the columns of the provided active_df given the values specified. If values aren't provided, they aren't changed. 
        
        active_df['series_vertical_offset', 'series_height', 'pen', 'brush']
        
        """        
        # Update only the provided columns while leaving the others intact
        if y_location is not None:
            ## y_location:
            if isinstance(y_location, (list, tuple)):
                active_df['series_vertical_offset'] = kwargs.setdefault('series_vertical_offset', [a_y_location for a_y_location in y_location])
            else:
                # Scalar value assignment:
                active_df['series_vertical_offset'] = kwargs.setdefault('series_vertical_offset', y_location)
                
        if height is not None:
            ## series_height:
            if isinstance(height, (list, tuple)):
                active_df['series_height'] = kwargs.setdefault('series_height', [a_height for a_height in height])
            else:
                # Scalar value assignment:
                active_df['series_height'] = kwargs.setdefault('series_height', height)

        if pen_color is not None:
            ## pen_color:
            if isinstance(pen_color, (list, tuple)):
                active_df['pen'] = kwargs.setdefault('pen', [pg.mkPen(a_pen_color) for a_pen_color in pen_color])
            else:
                # Scalar value assignment:
                active_df['pen'] = kwargs.setdefault('pen', pg.mkPen(pen_color)) 
            
        if brush_color is not None:
            ## brush_color:
            if isinstance(brush_color, (list, tuple)):
                active_df['brush'] = kwargs.setdefault('brush', [pg.mkBrush(a_color) for a_color in brush_color])  
            else:
                # Scalar value assignment:
                active_df['brush'] = kwargs.setdefault('brush', pg.mkBrush(brush_color))
        
        return active_df #, kwargs


        

interval_key: str = 'custom_paradigm'
# custom_paradigm_update_dict = 
interval_update_kwargs = update_dict[interval_key]
# Extract visibility settings before updating datasource (handle both single dict and list of dicts)
visibility_settings = None
if isinstance(interval_update_kwargs, (list, tuple)):
    ## list of update dicts - each item can have its own isVisible property
    a_list_interval_update_kwargs = []
    visibility_settings = []
    for a_sub_interval_update_kwargs in interval_update_kwargs:
        if not isinstance(a_sub_interval_update_kwargs, dict):
            a_sub_interval_update_kwargs = a_sub_interval_update_kwargs.to_dict() # deal with EpochDisplayConfig 
        a_list_interval_update_kwargs.append(a_sub_interval_update_kwargs)
        # Extract visibility from each item (can be None if not specified)
        visibility_settings.append(a_sub_interval_update_kwargs.get('isVisible', None))
        # self.interval_datasources[interval_key].update_visualization_properties(lambda active_df, **kwargs: General2DRenderTimeEpochs._update_df_visualization_columns(active_df, **(a_sub_interval_update_kwargs | kwargs))) ## Fully inline
    ## END for a_sub_interval_update_kwargs in interval_update_kwargs...
    
    ## Update with list
    # a_list_interval_update_kwargs = [a_sub_interval_update_kwargs for a_sub_interval_update_kwargs in interval_update_kwargs]
    print(f'a_sub_interval_update_kwargs: {a_sub_interval_update_kwargs}')
    print(f'a_list_interval_update_kwargs: {a_list_interval_update_kwargs}')
    # active_2d_plot.interval_datasources[interval_key].update_visualization_properties(lambda active_df, **kwargs: General2DRenderTimeEpochs._update_df_visualization_columns(active_df, **(a_sub_interval_update_kwargs | kwargs))) ## Fully inline
    active_2d_plot.interval_datasources[interval_key].update_visualization_properties(lambda active_df, **kwargs: General2DRenderTimeEpochs._update_df_visualization_columns(active_df, **(split_list_of_dicts(a_list_interval_update_kwargs) | kwargs))) ##  Fixed for multiple lists
    ## #TODO 2026-02-02 13:13: - [ ] `visibility_settings` is never used.

else:
    ## single update item dict
    if not isinstance(interval_update_kwargs, dict):
        interval_update_kwargs = interval_update_kwargs.to_dict() # deal with EpochDisplayConfig 
    visibility_settings = interval_update_kwargs.get('isVisible', None)
    print(f'interval_update_kwargs: {interval_update_kwargs}')
    active_2d_plot.interval_datasources[interval_key].update_visualization_properties(lambda active_df, **kwargs: General2DRenderTimeEpochs._update_df_visualization_columns(active_df, **(interval_update_kwargs | kwargs))) ## Fully inline


In [ ]:
visibility_settings

In [ ]:
a_list_interval_update_kwargs ## convert from a list of dict to a dict-of-lists
split_list_of_dicts(a_list_interval_update_kwargs)

In [ ]:
interval_update_kwargs

In [ ]:

# Apply visibility setting to rendered items if provided
# For list configs: only apply if all items have the same visibility (or all None)
# For single configs: apply directly
if visibility_settings is not None and interval_key in active_2d_plot.rendered_epochs:
    if isinstance(visibility_settings, list):
        # List case: check if all non-None values are the same
        non_none_visibilities = [v for v in visibility_settings if v is not None]
        if len(non_none_visibilities) > 0:
            # If all non-None values are the same, apply that visibility
            if len(set(non_none_visibilities)) == 1:
                is_visible = non_none_visibilities[0]
                container = active_2d_plot.rendered_epochs[interval_key]
                for a_plot, rect_item in container.items():
                    if not isinstance(a_plot, str) and isinstance(rect_item, IntervalRectsItem):
                        rect_item.setVisible(is_visible)
            # If they differ, we can't set per-rectangle visibility, so skip
            # (IntervalRectsItem is a single graphics item that renders all rectangles)
    else:
        # Single config case: apply directly
        container = active_2d_plot.rendered_epochs[interval_key]
        for a_plot, rect_item in container.items():
            if not isinstance(a_plot, str) and isinstance(rect_item, IntervalRectsItem):
                rect_item.setVisible(visibility_settings)

In [ ]:

# [RectangleRenderTupleHelpers.QColor_to_simple_columns_dict(v)['hexColor'] for v in brush_colors]
# [RectangleRenderTupleHelpers.QColor_to_simple_columns_dict(v)['alpha'] for v in brush_colors]

[ColorFormatConverter.qColor_to_hexstring(v, include_alpha=True) for v in brush_colors]





In [ ]:
# Determine interval_keys that are missing from update_dict but exist in self.interval_datasources
removed_interval_keys = [k for k in active_2d_plot.rendered_epoch_series_names if k not in update_dict] # need to use this and not `self.interval_datasources.keys()` directly because it has non-attribute members like 'name'


In [ ]:
series_viz_df['pen_color'] == custom_paradigm_ds_df['pen_color']

In [ ]:
a_serializable_df = custom_paradigm_ds.get_serialized_data(drop_duplicates=False)
a_serializable_df


In [ ]:
all_series_positioning_dfs, all_series_compressed_positioning_dfs, all_series_compressed_positioning_update_dicts = active_2d_plot.recover_interval_datasources_update_dict_properties()
# all_series_positioning_dfs
all_series_compressed_positioning_dfs # ERROR: series_compressed_positioning_update_dict is None for custom_paradigm. it will not be represented in the output dict.

In [ ]:
out_configs_df = active_2d_plot.extract_interval_display_config_df()
out_configs_df

In [ ]:
bottom_y_min, top_y_max = active_2d_plot.get_interval_y_extrema_locations()
curr_x_min, curr_x_max, curr_y_min, curr_y_max = active_2d_plot.get_render_intervals_plot_range()
new_y_min = min(curr_y_min, bottom_y_min)
new_y_max = max(curr_y_max, top_y_max)
for a_plot in active_2d_plot.interval_rendering_plots:
    a_plot.setYRange(new_y_min, new_y_max, padding=0)



In [ ]:
# active_2d_plot.get_render_intervals_plot_range(debug_print=True)
curr_x_min, curr_x_max, curr_y_min, curr_y_max = active_2d_plot.get_render_intervals_plot_range(debug_print=True)

# active_2d_plot.update_rendered_interval_heights(41.0)

In [ ]:
for a_plot_item in active_2d_plot.interval_rendering_plots:
    # a_plot_item: pg.PlotItem = active_2d_plot.interval_rendering_plots[0]
    a_plot_item.setYRange(curr_y_min, curr_y_max)
    # a_plot_item.update()

In [ ]:
# active_2d_plot.build_epoch_intervals_visual_configs_widget()
spike_raster_window.build_epoch_intervals_visual_configs_widget()


In [ ]:
active_2d_plot.build_or_update_epoch_render_configs_widget()

In [ ]:
# Create a label formatting function that accesses the dataframe
def create_label_format_fn(datasource):
    """Creates a closure that captures the datasource's label column"""
    label_column = datasource.df['label'].tolist()  # or whatever column has the names
    
    def _format_label_for_rect_data(rect_index: int, rect_data_tuple: Tuple) -> str:
        """Returns the label text for this interval. Captures: label_column
        """
        start_t, series_vertical_offset, duration_t, series_height, pen, brush = rect_data_tuple
        end_t = start_t + duration_t
        item_label: str = f"{datasource.custom_datasource_name}[{rect_index}]"
        if rect_index < len(label_column):
            item_label = str(label_column[rect_index])

        label_text = f"{item_label}\nStart: {start_t:.3f}\nEnd: {end_t:.3f}\nDuration: {duration_t:.3f}" # The tooltip is set generically here to 'PBEs', 'Replays' or whatever the dataseries name is
        return label_text        

    return _format_label_for_rect_data


def create_tooltip_format_fn(datasource):
    """Creates a closure that captures the datasource's label column"""
    label_column = datasource.df['label'].tolist()  # or whatever column has the names
    
    def _custom_format_tooltip_for_rect_data(rect_index: int, rect_data_tuple: Tuple) -> str:
        """ Captures: label_column"""
        start_t, series_vertical_offset, duration_t, series_height, pen, brush = rect_data_tuple
        end_t = start_t + duration_t
        item_label: str = f"{datasource.custom_datasource_name}[{rect_index}]"
        if rect_index < len(label_column):
            item_label = str(label_column[rect_index])
            print(f'\tgot specific label: "{item_label}"')
            
        tooltip_text = f"{item_label}\nStart: {start_t:.3f}\nEnd: {end_t:.3f}\nDuration: {duration_t:.3f}" # The tooltip is set generically here to 'PBEs', 'Replays' or whatever the dataseries name is
        return tooltip_text


    return _custom_format_tooltip_for_rect_data



## INPUTS: an_interval_ds, an_interval_rects_item

# Create the label formatter
label_format_fn = create_label_format_fn(an_interval_ds)
tooltip_format_fn = create_tooltip_format_fn(an_interval_ds)

# Update the rendered intervals with labels
# an_interval_rects_item.item_label_format_fn = deepcopy(label_format_fn)
# an_interval_rects_item._current_hovered_item_tooltip_format_fn = deepcopy(tooltip_format_fn)

for plot_name, rect_item_dict in active_2d_plot.get_all_rendered_intervals_dict().items():
    if plot_name == 'custom_paradigm':
        print(f'updating "custom_paradigm" intervals for plot_name: "{plot_name}"...')
        rect_item: IntervalRectsItem = rect_item_dict['RootPlot']
        rect_item.format_item_tooltip_fn = deepcopy(tooltip_format_fn)
        rect_item.item_label_format_fn = deepcopy(label_format_fn)
        # rect_item._current_hovered_item_tooltip_format_fn = deepcopy(tooltip_format_fn)
        # Need to regenerate the labels - this requires recreating the item OR you can manually add labels
        print(f'\tdone.')
        
    else:
        print('WARN: "custom_paradigm" intervals not found!')

In [ ]:
# active_2d_plot.get_all_rendered_intervals_dict()
# active_2d_plot.update() ## crashes the kernel

In [ ]:
# a_final_interval_df
active_2d_plot.build_or_update_epoch_render_configs_widget()

In [ ]:
from pyphoplacecellanalysis.GUI.PyQtPlot.Widgets.GraphicsObjects.IntervalRectsItem import IntervalRectsItem

def _custom_format_tooltip_for_rect_data(rect_index: int, rect_data_tuple: Tuple) -> str:
    start_t, series_vertical_offset, duration_t, series_height, pen, brush = rect_data_tuple
    end_t = start_t + duration_t
    tooltip_text = f"{name}[{rect_index}]\nStart: {start_t:.3f}\nEnd: {end_t:.3f}\nDuration: {duration_t:.3f}" # The tooltip is set generically here to 'PBEs', 'Replays' or whatever the dataseries name is
    return tooltip_text


# active_2d_plot.interval_datasources

_out_rendered_intervals = active_2d_plot.get_all_rendered_intervals_dict()
an_interval_rects_item: IntervalRectsItem = _out_rendered_intervals['custom_paradigm']['RootPlot']
an_interval_rects_item._current_hovered_item_tooltip_format_fn = deepcopy(_custom_format_tooltip_for_rect_data)
an_interval_rects_item

In [ ]:
an_interval_ds.custom_datasource_name

In [ ]:
rect_item

In [ ]:
active_2d_plot.add_rendered_intervals(an_interval_ds, name=f'custom_paradigm', debug_print=False)

In [ ]:
an_interval_rects_item.data

In [ ]:
active_2d_plot.remove_rendered_intervals(

In [ ]:


train_test_split_laps_epochs_formatting_dict = {
    'LapsAll':dict(y_location=-10.0, height=7.5, pen_color=inline_mkColor('white', 0.8), brush_color=inline_mkColor('white', 0.5)),
    'LapsTrain':dict(y_location=-2.0, height=1.5, pen_color=inline_mkColor('purple', 0.8), brush_color=inline_mkColor('purple', 0.5)),
    'LapsTest':dict(y_location=-12.0, height=1.5, pen_color=inline_mkColor('green', 0.8), brush_color=inline_mkColor('green', 0.5)),
}

In [ ]:
curr_active_pipeline.get_output_path()

In [ ]:
# curr_paradigm_df: pd.DataFrame = ensure_dataframe(sess.paradigm)
# curr_paradigm_df: pd.DataFrame = ensure_dataframe(sess.epochs_bak)


curr_paradigm_df: pd.DataFrame = np.load(f"W:/Data/Bapun/RatU/RatUDay5OpenfieldSD/RatU_Day5OpenfieldSD_2021-08-04_08-44-31.paradigm.npy", allow_pickle=True).tolist()['epochs']
curr_paradigm_df['duration'] = curr_paradigm_df['stop'] - curr_paradigm_df['start']
curr_paradigm_df

# curr_paradigm_df

In [ ]:
from pyphoplacecellanalysis.GUI.PyQtPlot.Widgets.GraphicsWidgets.EpochsEditorItem import EpochsEditor # perform_plot_laps_diagnoser
import matplotlib.pyplot as plt
from pyphocorehelpers.gui.Qt.color_helpers import ColormapHelpers, ColorFormatConverter
from neuropy.core.epoch import Epoch, EpochsAccessor, ensure_dataframe, ensure_Epoch, EpochHelpers

def generate_colors(n_epoch):
    cmap = plt.get_cmap('tab20', n_epoch)
    return [plt.matplotlib.colors.rgb2hex(cmap(i)) for i in range(n_epoch)]


sess = curr_active_pipeline.sess # global_session

# pos_df = sess.compute_position_laps() # ensures the laps are computed if they need to be:
position_obj = deepcopy(sess.position)
position_obj.compute_higher_order_derivatives()
pos_df = position_obj.compute_smoothed_position_info(N=20) ## Smooth the velocity curve to apply meaningful logic to it
pos_df = position_obj.to_dataframe()
# Drop rows with missing data in columns: 't', 'velocity_x_smooth' and 2 other columns. This occurs from smoothing
pos_df = pos_df.dropna(subset=['t', 'x_smooth', 'velocity_x_smooth', 'acceleration_x_smooth']).reset_index(drop=True)
# curr_laps_df = sess.laps.to_dataframe()

# curr_paradigm_df = ensure_dataframe(sess.paradigm)

curr_paradigm_df = ensure_dataframe(curr_paradigm_df)
curr_paradigm_df = curr_paradigm_df[np.logical_not(np.isin(curr_paradigm_df['label'], ['maze_GLOBAL', 'maze']))] ## exclude the global epoch
curr_paradigm_df = EpochHelpers.assign_overlap_y_offset(curr_paradigm_df, start_col='start', stop_col='stop', out_col='overlap_y_offset') 
n_epochs: int = len(curr_paradigm_df)
# epoch_color_strs: List[str] = generate_colors(n_epochs)
epoch_color_strs: List[str] = [ColorFormatConverter.qColor_to_hexstring(v, include_alpha=False) for v in ColormapHelpers.mpl_to_pg_colormap(mpl_cmap_name='tab20', resolution=n_epochs).getColors(mode='qcolor')]
curr_paradigm_df['lap_color'] = "#10FF44"
curr_paradigm_df['lap_color'] = epoch_color_strs
curr_paradigm_df['lap_accent_color'] = '#FFFFFF'
curr_paradigm_df
## Create a new window:
custom_epoch_label_kwargs = dict(epoch_label_position=0.05, epoch_label_rotateAxis=(0,0), epoch_label_anchor=(0.0, 1.0))
epochs_editor = EpochsEditor.init_laps_diagnoser(pos_df, curr_paradigm_df, include_velocity=False, include_accel=False, span=(0.05, 0.95), movable=False, **custom_epoch_label_kwargs)

In [ ]:
epochs_editor.plots
# epochs_editor.rebuild_epoch_regions()



In [ ]:
active_2d_plot.plots.main_plot_widget

main_plot_widget = active_2d_plot.plots.main_plot_widget # PlotItem
main_plot_widget.setMinimumHeight(20.0)


In [ ]:
# active_window_container_layout
# main_graphics_layout_widget.ci # GraphicsLayout
main_graphics_layout_widget.ci.childItems()
# main_graphics_layout_widget.setHidden(True) ## hides too much
main_graphics_layout_widget.setHidden(False)

# main_graphics_layout_widget

active_window_container_layout.setBorder(pg.mkPen('yellow', width=4.5))

In [ ]:
# active_window_container_layout.allChildItems()
active_window_container_layout.setPreferredHeight(200.0)
active_window_container_layout.setMaximumHeight(800.0)
active_window_container_layout.setSpacing(0)

In [ ]:
# Set stretch factors to control priority
main_graphics_layout_widget.ci.layout.setRowStretchFactor(0, 400)  # Plot1: lowest priority
main_graphics_layout_widget.ci.layout.setRowStretchFactor(1, 2)  # Plot2: mid priority
main_graphics_layout_widget.ci.layout.setRowStretchFactor(2, 2)  # Plot3: highest priority


In [ ]:
from pyphoplacecellanalysis.GUI.PyQtPlot.Widgets.ParameterTreeWidget import create_parameter_tree_widget
# win, param_tree = create_pipeline_filter_parameter_tree()
win, param_tree = create_parameter_tree_widget(curr_active_pipeline.get_all_parameters())
win.show()

In [ ]:
from pyphoplacecellanalysis.General.Pipeline.Stages.ComputationFunctions.MultiContextComputationFunctions.DirectionalPlacefieldGlobalComputationFunctions import DirectionalDecodersContinuouslyDecodedResult
from pyphoplacecellanalysis.GUI.PyQtPlot.Widgets.SpikeRasterWidgets.Spike2DRaster import SynchronizedPlotMode
from pyphoplacecellanalysis.General.Pipeline.Stages.DisplayFunctions.DecoderPredictionError import plot_1D_most_likely_position_comparsions
from pyphoplacecellanalysis.General.Model.Configs.LongShortDisplayConfig import DecoderIdentityColors
from pyphoplacecellanalysis.SpecificResults.PendingNotebookCode import _perform_plot_multi_decoder_meas_pred_position_track
from pyphoplacecellanalysis.Analysis.Decoder.reconstruction import DecodedFilterEpochsResult

## Build the new dock track:
dock_identifier: str = 'Continuous Decoding Performance'
ts_widget, fig, ax_list, dDisplayItem = active_2d_plot.add_new_matplotlib_render_plot_widget(name=dock_identifier)
## Get the needed data:
directional_decoders_decode_result: DirectionalDecodersContinuouslyDecodedResult = curr_active_pipeline.global_computation_results.computed_data['DirectionalDecodersDecoded']
all_directional_pf1D_Decoder_dict: Dict[str, BasePositionDecoder] = directional_decoders_decode_result.pf1D_Decoder_dict
continuously_decoded_result_cache_dict = directional_decoders_decode_result.continuously_decoded_result_cache_dict
previously_decoded_keys: List[float] = list(continuously_decoded_result_cache_dict.keys()) # [0.03333]
print(F'previously_decoded time_bin_sizes: {previously_decoded_keys}')

time_bin_size: float = directional_decoders_decode_result.most_recent_decoding_time_bin_size
print(f'time_bin_size: {time_bin_size}')
continuously_decoded_dict: Dict[str, DecodedFilterEpochsResult] = directional_decoders_decode_result.most_recent_continuously_decoded_dict
all_directional_continuously_decoded_dict: Dict[types.DecoderName, DecodedFilterEpochsResult] = {k:v for k, v in (continuously_decoded_dict or {}).items() if k in TrackTemplates.get_decoder_names()} ## what is plotted in the `f'{a_decoder_name}_ContinuousDecode'` rows by `AddNewDirectionalDecodedEpochs_MatplotlibPlotCommand`
## OUT: all_directional_continuously_decoded_dict
## Draw the position meas/decoded on the plot widget
## INPUT: fig, ax_list, all_directional_continuously_decoded_dict, track_templates


In [ ]:
directional_decoders_decode_result

In [ ]:

_out_artists =  _perform_plot_multi_decoder_meas_pred_position_track(curr_active_pipeline, fig, ax_list, desired_time_bin_size=0.058, enable_flat_line_drawing=True)

## sync up the widgets
active_2d_plot.sync_matplotlib_render_plot_widget(dock_identifier, sync_mode=SynchronizedPlotMode.TO_WINDOW)

In [ ]:
## split into two dfs for each decoder -- the supported and the unsupported
partition

PandasHelpers.safe_pandas_get_group

In [ ]:
pos_df.dropna(axis='index', subset=['lap', 'truth_decoder_name'], inplace=False)

In [ ]:
laps_df: pd.DataFrame = global_laps_obj.to_dataframe()

In [ ]:
from neuropy.core.epoch import EpochHelpers

## INPUTS: global_laps
_out_split_pseudo2D_posteriors_dict = {}
_out_split_pseudo2D_out_dict = {}
pre_filtered_col_names = ['pre_filtered_most_likely_position_indicies', 'pre_filtered_most_likely_position'] # 'pre_filtered_time_bin_containers', 'pre_filtered_p_x_given_n', 
post_filtered_col_names = [a_col_name.removeprefix('pre_filtered_') for a_col_name in pre_filtered_col_names] # ['time_bin_containers', 'most_likely_position_indicies', 'most_likely_position']
print(post_filtered_col_names)
for a_time_bin_size, pseudo2D_decoder_continuously_decoded_result in continuously_decoded_pseudo2D_decoder_dict.items():
    print(f'a_time_bin_size: {a_time_bin_size}')
    _out_split_pseudo2D_out_dict[a_time_bin_size] = {'pre_filtered_p_x_given_n': None, 'pre_filtered_time_bin_containers': None, 'pre_filtered_most_likely_position_indicies': None, 'pre_filtered_most_likely_position': None, 
                                                     'is_timebin_included': None, 'p_x_given_n': None} # , 'time_window_centers': None
    # pseudo2D_decoder_continuously_decoded_result: DecodedFilterEpochsResult = continuously_decoded_dict.get('pseudo2D', None)
    assert len(pseudo2D_decoder_continuously_decoded_result.p_x_given_n_list) == 1
    p_x_given_n = pseudo2D_decoder_continuously_decoded_result.p_x_given_n_list[0]
    # p_x_given_n = pseudo2D_decoder_continuously_decoded_result.p_x_given_n_list[0]['p_x_given_n']
    time_bin_containers = pseudo2D_decoder_continuously_decoded_result.time_bin_containers[0]
    # time_window_centers = time_bin_containers.centers
    _out_split_pseudo2D_out_dict[a_time_bin_size]['pre_filtered_most_likely_position_indicies'] = deepcopy(pseudo2D_decoder_continuously_decoded_result.most_likely_position_indicies_list[0])
    _out_split_pseudo2D_out_dict[a_time_bin_size]['pre_filtered_most_likely_position'] = deepcopy(pseudo2D_decoder_continuously_decoded_result.most_likely_positions_list[0])
    ## INPUTS: time_bin_containers, global_laps
    left_edges = deepcopy(time_bin_containers.left_edges)
    right_edges = deepcopy(time_bin_containers.right_edges)
    continuous_time_binned_computation_epochs_df: pd.DataFrame = pd.DataFrame({'start': left_edges, 'stop': right_edges, 'label': np.arange(len(left_edges))})
    is_timebin_included: NDArray = EpochHelpers.find_epochs_overlapping_other_epochs(epochs_df=continuous_time_binned_computation_epochs_df, epochs_df_required_to_overlap=deepcopy(global_laps))
    _out_split_pseudo2D_out_dict[a_time_bin_size]['pre_filtered_p_x_given_n'] = p_x_given_n
    _out_split_pseudo2D_out_dict[a_time_bin_size]['pre_filtered_time_bin_containers'] = time_bin_containers
    _out_split_pseudo2D_out_dict[a_time_bin_size]['is_timebin_included'] = is_timebin_included
    # continuous_time_binned_computation_epochs_df['is_in_laps'] = is_timebin_included
    ## filter by whether it's included or not:
    p_x_given_n = p_x_given_n[:, :, is_timebin_included]
    # time_window_centers = 
    _out_split_pseudo2D_out_dict[a_time_bin_size]['p_x_given_n'] = p_x_given_n
    # _out_split_pseudo2D_out_dict[a_time_bin_size]['time_window_centers'] = time_window_centers[is_timebin_included]
    # p_x_given_n.shape # (62, 4, 209389)

    ## Split across the 2nd axis to make 1D posteriors that can be displayed in separate dock rows:
    assert p_x_given_n.shape[1] == 4, f"expected the 4 pseudo-y bins for the decoder in p_x_given_n.shape[1]. but found p_x_given_n.shape: {p_x_given_n.shape}"
    # split_pseudo2D_posteriors_dict = {k:np.squeeze(p_x_given_n[:, i, :]) for i, k in enumerate(('long_LR', 'long_RL', 'short_LR', 'short_RL'))}
    _out_split_pseudo2D_posteriors_dict[a_time_bin_size] = deepcopy(p_x_given_n)
    
    # for a_col_name in pre_filtered_col_names:
    #     filtered_col_name = a_col_name.removeprefix('pre_filtered_')
    #     print(f'a_col_name: {a_col_name}, filtered_col_name: {filtered_col_name}, shape: {np.shape(_out_split_pseudo2D_out_dict[a_time_bin_size][a_col_name])}')
    #     _out_split_pseudo2D_out_dict[a_time_bin_size][filtered_col_name] = _out_split_pseudo2D_out_dict[a_time_bin_size][a_col_name][is_timebin_included, :]
        
    _out_split_pseudo2D_out_dict[a_time_bin_size]['most_likely_position_indicies'] = _out_split_pseudo2D_out_dict[a_time_bin_size]['pre_filtered_most_likely_position_indicies'][:, is_timebin_included]
    _out_split_pseudo2D_out_dict[a_time_bin_size]['most_likely_position'] = _out_split_pseudo2D_out_dict[a_time_bin_size]['pre_filtered_most_likely_position'][is_timebin_included, :]
    

p_x_given_n.shape # (n_position_bins, n_decoding_models, n_time_bins) - (57, 4, 29951)

## OUTPUTS: _out_split_pseudo2D_posteriors_dict, _out_split_pseudo2D_out_dict

In [ ]:
from pyphoplacecellanalysis.General.Pipeline.Stages.DisplayFunctions.DecoderPredictionError import plot_most_likely_position_comparsions

# fig, axs = plot_most_likely_position_comparsions(pho_custom_decoder, axs=ax, sess.position.to_dataframe())
fig, axs = plot_most_likely_position_comparsions(computation_result.computed_data['pf2D_Decoder'], computation_result.sess.position.to_dataframe(), **overriding_dict_with(lhs_dict={'show_posterior':True, 'show_one_step_most_likely_positions_plots':True}, **kwargs))


In [ ]:
hardcoded_params.non_global_activity_session_names

# 🖼️⚓❎ Time Synchronized Plotting with position

In [ ]:
from pyphoplacecellanalysis.Pho2D.PyQtPlots.TimeSynchronizedPlotters.Mixins.AnimalTrajectoryPlottingMixin import AnimalTrajectoryPlottingMixin
from pyphoplacecellanalysis.Pho2D.PyQtPlots.TimeSynchronizedPlotters.TimeSynchronizedPositionDecoderPlotter import TimeSynchronizedPositionDecoderPlotter
from pyphoplacecellanalysis.GUI.PyQtPlot.Widgets.ContainerBased.PhoContainerTool import GenericPyQtGraphContainer
from pyphoplacecellanalysis.SpecificResults.PendingNotebookCode import build_combined_time_synchronized_Bapun_decoders_window
from pyphoplacecellanalysis.GUI.PyQtPlot.Widgets.SpikeRasterWidgets.Spike2DRaster import Spike2DRaster, SynchronizedPlotMode
from pyphoplacecellanalysis.GUI.PyQtPlot.Widgets.DockAreaWrapper import PhoDockAreaContainingWindow
from pyphoplacecellanalysis.GUI.PyQtPlot.DockingWidgets.SpecificDockWidgetManipulatingMixin import SpecificDockWidgetManipulatingMixin
from pyphoplacecellanalysis.General.Pipeline.Stages.ComputationFunctions.MultiContextComputationFunctions.DirectionalPlacefieldGlobalComputationFunctions import DirectionalDecodersContinuouslyDecodedResult, decoding_continuous_cache_key
from pyphoplacecellanalysis.SpecificResults.PendingNotebookCode import build_contextual_pf2D_decoder, decode_using_contextual_pf2D_decoder


hardcoded_params: HardcodedProcessingParameters = BapunDataSessionFormatRegisteredClass._get_session_specific_parameters(session_context=curr_active_pipeline.get_session_context())
# hardcoded_params.decoder_building_session_names
# hardcoded_params.non_global_activity_session_names


# pg.setConfigOptions(useOpenGL=True)  # do this BEFORE creating plots/widgets
# force_recompute: bool = False
force_recompute: bool = True

## Uses the `global_computation_results.computed_data['DirectionalDecodersDecoded']`
# directional_decoders_decode_result: DirectionalDecodersContinuouslyDecodedResult = curr_active_pipeline.global_computation_results.computed_data['DirectionalDecodersDecoded']
directional_decoders_decode_result: DirectionalDecodersContinuouslyDecodedResult = curr_active_pipeline.global_computation_results.computed_data.get('DirectionalDecodersDecoded', None)


if (directional_decoders_decode_result is None) or force_recompute:
    if force_recompute:
        print(f'force_recompute is True, so `directional_decoders_decode_result` will be recomputed...')
    else:
        print(f'directional_decoders_decode_result is missing, recomputing....')

    # active_laps_decoding_time_bin_size = 0.75
    # active_laps_decoding_time_bin_size = 0.025 # 25ms
    active_laps_decoding_time_bin_size = 0.250 # 250ms
    # active_laps_decoding_time_bin_size = 0.250 # 250ms
    active_laps_decoding_slideby = None  # None => non-overlapping; e.g. 0.05 with W=0.25 for sliding
    epochs_to_create_global_from_names = ['roam', 'sprinkle']
    
    ## Build the merged decoder `contextual_pf2D`
    contextual_pf2D_dict, contextual_pf2D, contextual_pf2D_Decoder = build_contextual_pf2D_decoder(curr_active_pipeline, epochs_to_create_global_from_names = epochs_to_create_global_from_names)
    ## Use `contextual_pf2D` to decode specific epochs:
    all_context_filter_epochs_decoder_result, global_only_epoch = decode_using_contextual_pf2D_decoder(curr_active_pipeline, contextual_pf2D_Decoder=contextual_pf2D_Decoder,
                                                                                                        active_laps_decoding_time_bin_size=active_laps_decoding_time_bin_size, slideby=active_laps_decoding_slideby, epochs_to_merge_as_global_epoch_names=epochs_to_create_global_from_names)

    ## Build global result object

    global_spikes_df: pd.DataFrame = deepcopy(curr_active_pipeline.sess.spikes_df)
    directional_decoders_decode_result: DirectionalDecodersContinuouslyDecodedResult = DirectionalDecodersContinuouslyDecodedResult(pf1D_Decoder_dict=contextual_pf2D_dict, pseudo2D_decoder=contextual_pf2D_Decoder, spikes_df=global_spikes_df,
                                                                                                                                     continuously_decoded_result_cache_dict={decoding_continuous_cache_key(active_laps_decoding_time_bin_size, active_laps_decoding_slideby):{'pseudo2D': all_context_filter_epochs_decoder_result}})
    curr_active_pipeline.global_computation_results.computed_data['DirectionalDecodersDecoded'] = directional_decoders_decode_result
    print(f'\tdone recomputing.')


In [ ]:
# _out_container: GenericPyQtGraphContainer = build_combined_time_synchronized_Bapun_decoders_window(curr_active_pipeline, included_filter_names=hardcoded_params.non_global_activity_session_names, fixed_window_duration = 3.0)
_out_container_new: GenericPyQtGraphContainer = build_combined_time_synchronized_Bapun_decoders_window(curr_active_pipeline, included_filter_names=hardcoded_params.non_global_activity_session_names, fixed_window_duration = 1.0,
    directional_decoders_decode_result=directional_decoders_decode_result,
    controlling_widget=active_2d_plot, create_new_controlling_widget=False,
)

active_2d_plot: Spike2DRaster = _out_container_new.ui.controlling_widget
sync_plotters: Dict[str, TimeSynchronizedPositionDecoderPlotter] = _out_container_new.ui.sync_plotters
win: PhoDockAreaContainingWindow = _out_container_new.ui.root_dockAreaWindow
# a_sync_plotter: TimeSynchronizedPositionDecoderPlotter = sync_plotters['roam']
# a_sync_plotter.curr_position

# ## Disable debug print to speed up animation
# for a_plotter_name, a_plotter in sync_plotters.items():
#     a_plotter.params.debug_print = False


## INPUTS: _out_container, active_2d_plot, _out_container, sync_plotters, 


In [ ]:
np.diff(a_plotter.time_window_centers)


In [ ]:
for an_epoch_name, a_plotter in sync_plotters.items():
    # display(a_plotter.params.debug_print)
    # a_plotter.params.debug_print = True
    # display(a_plotter.params.debug_print)
    # a_plotter
    a_plotter.ui.root_plot.setTitle(f'PositionDecoder -  t = {a_plotter.last_window_time}')    


# a_plotter.ui.root_plot.setTitle(f'PositionDecoder -  t = {a_plotter.last_window_time}')

# a_plotter.params
    

In [ ]:
epochs_df: pd.DataFrame = curr_active_pipeline.sess.epochs.to_dataframe()
curr_epoch_info = epochs_df[epochs_df['label'] == 'roam'].iloc[0]
curr_epoch_info['start']
curr_epoch_info['stop']

# curr_active_pipeline.sess.active_

In [ ]:
## Output videos:
## INPUTS: sync_plotters
export_video_paths = {}

export_video_parent_folder = curr_active_pipeline.get_output_path().joinpath('videos').resolve()
export_video_parent_folder.mkdir(exist_ok=True)

an_epoch_name: str = 'roam'
an_export_video_path = export_video_parent_folder.joinpath(f'2026-05-12_decoder_{an_epoch_name}.avi')
epochs_df: pd.DataFrame = curr_active_pipeline.sess.epochs.to_dataframe()
curr_epoch_info = epochs_df[epochs_df['label'] == 'roam'].iloc[0]
start_t: float = curr_epoch_info['start']
end_t: float = curr_epoch_info['stop']

print(f'exporting to "{an_export_video_path}" for start_t: {start_t}, end_t: {end_t}...')
a_plotter = sync_plotters[an_epoch_name]
video_path = a_plotter.export_video(an_export_video_path, start_t=start_t, end_t=end_t, fps=30.0, debug_print=False)

print(f'\texport to video_path: "{video_path.resolve().as_posix()}" complete.')
export_video_paths[an_epoch_name] = video_path


In [ ]:
_out_pbe_tracks, _out_pbe_overview_tracks = _out_container_new.add_pbes_full_result_marginals(pbes_full_result=pbes_full_result)

# grouped_dock_items_dict = _out_container_new.build_overview_and_windowed_dockgroups() ## Not quite ready

## Reorder the dock items

In [ ]:
list(active_2d_plot.dock_manager_widget.dynamic_display_dict.keys())
# original_identifier_order = ['interval_overview', 'intervals', 'rasters[raster_overview]', 'rasters[raster_window]', 'global context', 'global context (overview)', 'pbe[0.06]', 'pbe[0.06] (Overview)']
desired_identifier_order = ['interval_overview', 'rasters[raster_overview]', 'global context (overview)', 'pbe[0.06] (Overview)', 'rasters[raster_window]', 'intervals', 'global context', 'pbe[0.06]'] ## #TODO 2025-09-21 15:41: - [ ] Enforce this order programmatically?!


In [ ]:
active_2d_plot.dock_manager_widget

In [ ]:
## INPUTS: active_2d_plot
grouped_dock_items_dict = active_2d_plot.ui.dynamic_docked_widget_container.get_dockGroup_dock_dict()
nested_dock_items = {}
nested_dynamic_docked_widget_container_widgets = {}
for dock_group_name, flat_group_dockitems_list in grouped_dock_items_dict.items():
    dDisplayItem, nested_dynamic_docked_widget_container = active_2d_plot.ui.dynamic_docked_widget_container.build_wrapping_nested_dock_area(flat_group_dockitems_list, dock_group_name=dock_group_name)
    nested_dock_items[dock_group_name] = dDisplayItem
    nested_dynamic_docked_widget_container_widgets[dock_group_name] = nested_dynamic_docked_widget_container

## OUTPUTS: nested_dock_items, nested_dynamic_docked_widget_container_widgets

In [ ]:
grouped_dock_items_dict = active_2d_plot.ui.dynamic_docked_widget_container.get_dockGroup_dock_dict()
grouped_dock_items_dict

In [ ]:



grouped_dock_items_dict = build_overview_and_windowed_dockgroups(active_2d_plot)
grouped_dock_items_dict


In [ ]:
_out = active_2d_plot.dock_manager_widget.layout_dockGroups()


### 📈🔃❎ Exporting as video

In [ ]:
from PyQt5 import QtWidgets, QtGui, QtCore
import pyphoplacecellanalysis.External.pyqtgraph as pg
from pyqtgraph.exporters import ImageExporter
from PIL import Image
from pyphoplacecellanalysis.GUI.PyQtPlot.Widgets.GraphicsWidgets.CustomGraphicsLayoutWidget import CustomGraphicsLayoutWidget

## "playback" refers to output video:
desired_playback_duration: float = 8 * 60.0 # 8m

# "session" refers to the actual recording session:
desired_session_time_range_duration: float = (16.0 * 60.0) # 1m
# session_start_t: float = 23170.37047485091 #20740.63578640229 # 11631.186907154472 # 11451.186907154472 # 11391.186907154472 # 7665.232126354053 # active_2d_plot.total_data_start_time + 4.0 * 60.0 # 4 minutes into start of recording ## Day 5
# session_start_t: float = 11023.018433333335 # 10843.018433333334 # active_2d_plot.total_data_start_time + 4.0 * 60.0 # 4 minutes into start of recording
# session_start_t: float = 23170.37047485091 #20740.63578640229 # 11631.186907154472 # 11451.186907154472 # 11391.186907154472 # active_2d_plot.total_data_start_time + 4.0 * 60.0 # 4 minutes into start of recording ## day 4

session_start_t: float = active_2d_plot.total_data_start_time + 20.0 * 60.0 # 4 minutes into start of recording ## day 4

desired_session_time_range: Tuple[float, float] = (session_start_t, (session_start_t + desired_session_time_range_duration))

playback_speed_factor: float = (desired_playback_duration / desired_session_time_range_duration)

print(f'playback_speed_factor: {playback_speed_factor}')
time_window_duration: float = active_2d_plot.active_window_duration
print(f'time_window_duration: {time_window_duration}')

## INPUTS: _out_container, active_2d_plot, _out_container, sync_plotters, 
desired_framerate: float = 2.0
desired_frame_duration_sec: float = 1.0/desired_framerate
print(f'desired_frame_duration_sec: {desired_frame_duration_sec}')

# ## All Frames from entire recording (too long)
# total_duration: float = active_2d_plot.total_data_duration
# desired_num_total_frames: int = int(np.ceil((total_duration * desired_framerate)))
# frame_start_indicies = np.linspace(active_2d_plot.total_data_start_time,  active_2d_plot.total_data_end_time, num=desired_num_total_frames)

## Plot only for the range of interest:
desired_num_total_frames: int = int(np.ceil((desired_session_time_range_duration * desired_framerate)))
frame_start_indicies = np.linspace(desired_session_time_range[0], desired_session_time_range[1], num=desired_num_total_frames)
frame_end_indices = frame_start_indicies + desired_frame_duration_sec

print(f'desired_num_total_frames: {desired_num_total_frames}')


In [ ]:
# ## Disable debug print to speed up animation
for a_plotter_name, a_plotter in sync_plotters.items():
    a_plotter.params.debug_print = False
    a_plotter.enable_debug_print = False
    

In [ ]:
active_2d_plot.active_window_start_time

In [ ]:
# next_end_timestamp = next_start_timestamp + self.animation_active_time_window.window_duration

def _frame_update(frame_start_t, frame_end_t):
    active_2d_plot.update_scroll_window_region(frame_start_t, frame_end_t, block_signals=True)
    active_2d_plot.window_scrolled.emit(frame_start_t, frame_end_t)
    QtWidgets.QApplication.processEvents()
    win.repaint()


_frame_update(desired_session_time_range[0], (desired_session_time_range[0]+time_window_duration))


timestep_delta_sec: float = 0.5 # half second step


In [ ]:
## Step one frame:
frame_start_t: float = active_2d_plot.active_window_start_time + timestep_delta_sec ## current time plus delta
_frame_update(frame_start_t, (frame_start_t + time_window_duration))

In [ ]:
for i, (frame_start_t, frame_end_t) in enumerate(zip(frame_start_indicies, frame_end_indices)):
    print(f'frame[{i}]: ({frame_start_t}, {frame_end_t}):')
    # active_2d_plot.on_window_changed(frame_start_t, frame_end_t)
    # active_2d_plot.update_scroll_window_region(frame_start_t, frame_end_t, block_signals=True)
    # active_2d_plot.window_scrolled.emit(frame_start_t, frame_end_t)
    # pg.SignalProxy(driver.window_scrolled, delay=0.2, rateLimit=60, slot=drivable.on_window_changed_rate_limited)
    # QtWidgets.QApplication.processEvents()
    # win.repaint()
    # _frame_update(frame_start_t, frame_end_t)
    _frame_update(frame_start_t, (frame_start_t + time_window_duration))


### Build Marginals over track context and plot them on the timeline

In [ ]:
from pyphoplacecellanalysis.Analysis.Decoder.reconstruction import DecodedFilterEpochsResult, SingleEpochDecodedResult
from pyphoplacecellanalysis.SpecificResults.PendingNotebookCode import _add_context_marginal_to_timeline, _add_context_decoded_epoch_marginals_to_timeline


In [ ]:

# decoded_epochs_track_name: str = f'{epochs_name}[{decoding_time_bin_size}]'


_out = _add_context_marginal_to_timeline(active_2d_plot, a_filter_epochs_decoded_result=all_context_filter_epochs_decoder_result, name='global context')
_out_pbe_tracks = _add_context_decoded_epoch_marginals_to_timeline(active_2d_plot=active_2d_plot, decoded_epochs_result=pbe_decoder_result, name=f"pbe[{decoding_time_bin_size}]")


In [ ]:
from pyphoplacecellanalysis.GUI.PyQtPlot.Widgets.SpikeRasterWidgets.Spike2DRaster import SynchronizedPlotMode

_out_new = _add_context_marginal_to_timeline(active_2d_plot, a_filter_epochs_decoded_result=all_context_filter_epochs_decoder_result, name='global context (overview)')
_out_new

identifier_name, widget, matplotlib_fig, matplotlib_fig_axes, dock_item = _out_new

## 2025-10-21 - Plot Laps in 3D

In [ ]:
directional_decoders_decode_result.pf1D_Decoder_dict

In [ ]:
# hardcoded_params

## Draw the reward zones on the grid_bin_bounds:
reward_zones = hardcoded_params.lap_estimation_parameters['reward_zones'](curr_active_pipeline.filtered_sessions['roam'])
reward_zones

In [ ]:
hardcoded_params.grid_bin_bounds

In [ ]:
from neuropy.core import Laps
from shapely import box, Polygon
from shapely.geometry import LineString, Point
from neuropy.core.position import PositionAccessor, Position
from shapely.plotting import plot_polygon, patch_from_polygon

from neuropy.core.session.Formats.Specific.BapunDataSessionFormat import plot_shapely_maze ## TODO: move somewhere generically accessible
from neuropy.core.session.Formats.BaseDataSessionFormats import HardcodedProcessingParameters
from neuropy.core.session.Formats.Specific.BapunDataSessionFormat import BapunDataSessionFormatRegisteredClass

hardcoded_params: HardcodedProcessingParameters = BapunDataSessionFormatRegisteredClass._get_session_specific_parameters(session_context=curr_active_pipeline.get_session_context())
active_reward_zones_dict = hardcoded_params.lap_estimation_parameters['reward_zones'](curr_active_pipeline.filtered_sessions['roam'])
_out = plot_shapely_maze(grid_bin_bounds=hardcoded_params.grid_bin_bounds, 
                         reward_zones_dict=active_reward_zones_dict,
                         ax=None)
# _out['maze']


In [ ]:
from pyphoplacecellanalysis.Analysis.Decoder.reconstruction import DecodedFilterEpochsResult
from pyphoplacecellanalysis.General.Pipeline.Stages.ComputationFunctions.MultiContextComputationFunctions.DirectionalPlacefieldGlobalComputationFunctions import DirectionalDecodersContinuouslyDecodedResult, DecodingContinuousCacheKey, normalize_continuous_decoding_cache_lookup_key

from pyphoplacecellanalysis.GUI.PyQtPlot.Widgets.ContainerBased.PhoContainerTool import GenericMatplotlibContainer
from pyphoplacecellanalysis.PhoPositionalData.plotting.mixins.decoder_plotting_mixins import DecodedTrajectoryMatplotlibPlotter
from pyphoplacecellanalysis.PhoPositionalData.plotting.laps import plot_lap_trajectories_2d, plot_lap_trajectories_3d
from pyphoplacecellanalysis.PhoPositionalData.plotting.laps import _plot_helper_add_arrow
from neuropy.utils.matplotlib_helpers import perform_update_title_subtitle

from pyphoplacecellanalysis.GUI.PyQtPlot.Widgets.DockAreaWrapper import DockAreaWrapper
from pyphoplacecellanalysis.GUI.PyQtPlot.Widgets.ContainerBased.PhoContainerTool import GenericPyQtGraphContainer

_restore_previous_matplotlib_settings_callback = matplotlib_configuration_update(is_interactive=True, backend='Qt5Agg')

# pseudo3D_decoder
# INPUTS: _lap_burst_detection_results
max_num_subplots: int = 25

# active_container = _container_container.container
# active_container = _container_container.masked_container


# minimum_inclusion_fr_Hz: float = curr_active_pipeline.global_computation_results.computation_config.rank_order_shuffle_analysis.minimum_inclusion_fr_Hz
# included_qclu_values: List[int] = curr_active_pipeline.global_computation_results.computation_config.rank_order_shuffle_analysis.included_qclu_values

# directional_laps_results: DirectionalLapsResult = curr_active_pipeline.global_computation_results.computed_data['DirectionalLaps']
# track_templates: TrackTemplates = directional_laps_results.get_templates(minimum_inclusion_fr_Hz=minimum_inclusion_fr_Hz, included_qclu_values=included_qclu_values) # non-shared-only -- !! Is minimum_inclusion_fr_Hz=None the issue/difference?
# print(f'minimum_inclusion_fr_Hz: {minimum_inclusion_fr_Hz}')
# print(f'included_qclu_values: {included_qclu_values}')

directional_decoders_decode_result: DirectionalDecodersContinuouslyDecodedResult = curr_active_pipeline.global_computation_results.computed_data['DirectionalDecodersDecoded']
# all_directional_pf1D_Decoder_dict: Dict[str, BasePositionDecoder] = directional_decoders_decode_result.pf1D_Decoder_dict
previously_decoded_keys: List[DecodingContinuousCacheKey] = list(directional_decoders_decode_result.continuously_decoded_result_cache_dict.keys())

_fig_out_dict = {}

# epoch_names = ['sprinkle', 'roam']
epoch_names = ['roam', 'sprinkle'] # 
widget_out_dict = {}
for an_epoch_name in epoch_names:
    # k = 'roam'
    # k = 'sprinkle'
    # k = 'maze_GLOBAL'

    a_sess = curr_active_pipeline.filtered_sessions[an_epoch_name]
    pos_df = a_sess.position.to_dataframe()
    pos_df['speed_xy'] = np.sqrt(np.power(pos_df['velocity_x_smooth'], 2) +  np.power(pos_df['velocity_y_smooth'], 2))

    laps_df: pd.DataFrame = ensure_dataframe(a_sess.laps)
    # for k, a_sess in curr_active_pipeline.filtered_sessions.items():
    #     pos_df = a_sess.position.to_dataframe()
    spikes_df = a_sess.spikes_df.spikes.adding_lap_identity_column(laps_epoch_df=a_sess.laps.to_dataframe(), epoch_id_key_name='lap')
    # spikes_df

    lap_only_pos_df: pd.DataFrame = pos_df.dropna(subset=['lap'], inplace=False)
    lap_only_pos_df['lap'] = lap_only_pos_df['lap'].astype(int)
    lap_pos_df_dict = lap_only_pos_df.pho.partition_df_dict('lap')
    # lap_pos_df_dict
    lap_only_pos_df


    a_decoder = directional_decoders_decode_result.pf1D_Decoder_dict[an_epoch_name] # active_container.pf1D_Decoder_dict[an_epoch_name]
    # a_result2D: DecodedFilterEpochsResult = decoded_local_epochs_result.frame_divided_epochs_results[an_epoch_name]
    a_new_global_decoder2D = directional_decoders_decode_result.pf1D_Decoder_dict[an_epoch_name] # active_container.pf1D_Decoder_dict[an_epoch_name]
    # a_result2D = results2D.a_result2D
    # a_new_global_decoder2D = results2D.a_new_global_decoder2D
    ## INPUTS: directional_laps_results, decoder_ripple_filter_epochs_decoder_result_dict
    xbin = deepcopy(a_new_global_decoder2D.xbin)
    xbin_centers = deepcopy(a_new_global_decoder2D.xbin_centers)
    ybin_centers = deepcopy(a_new_global_decoder2D.ybin_centers)
    ybin = deepcopy(a_new_global_decoder2D.ybin)

    plotter_kwargs = dict(xbin=xbin, xbin_centers=xbin_centers, ybin=ybin, ybin_centers=ybin_centers)
    

    # PLOT _______________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________ #
    # arrow_concentration_kwargs = dict(
    #     arrow_skip = 50, time_cmap='viridis',
    #     mutation_scale_multiplier = 20, mutation_scale_constant = 1,
    # 	arrow_length_multiplier = 0.2, arrow_length_constant = 0.05,
    # 	arrow_lw = 0.5,
    # )

    arrow_concentration_kwargs = dict(
        arrow_skip = 50, time_cmap='viridis',
        mutation_scale_multiplier = 20, mutation_scale_constant = 1,
        arrow_length_multiplier = 0.05, arrow_length_constant = 0.01,
        arrow_lw = 0.5,
    )

    plot_lap_trajectories_2d_kwargs = dict(
        curr_num_subplots=(6*5), active_page_index=0, fixed_columns = 6,
    )

    out3: GenericMatplotlibContainer = plot_lap_trajectories_2d(a_sess, **plot_lap_trajectories_2d_kwargs, use_time_gradient_line=True, arrow_concentration_kwargs=arrow_concentration_kwargs, fig_size_inches=None)
    
    # out3: GenericMatplotlibContainer = plot_lap_trajectories_3d(a_sess, **plot_lap_trajectories_2d_kwargs, 
    #                                                             single_combined_plot = True,
    #                                                             # use_time_gradient_line=True, arrow_concentration_kwargs=arrow_concentration_kwargs,
    #                                                             # fig_size_inches=None,
    #                                                             )
    
    _fig_out_dict[an_epoch_name] = out3
    p3, axs, laps_pages3 = out3.fig, out3.axes, out3.plots_data.laps_pages
    perform_update_title_subtitle(fig=out3.fig, ax=None, title_string=f"{an_epoch_name} - 2d runs", subtitle_string=f"{an_epoch_name}")
    

    widget_out_dict[an_epoch_name] = p3.canvas.parent()
    # _fig_out_dict[k] = plot_lap_trajectories_3d_napari(a_sess, lap_id_dependent_z_offset=4.0)
    # viewer, layer, lap_ids = _fig_out_dict[k]
    # viewer will be shown if show=True

    # p3
    
laps_merged_out: GenericPyQtGraphContainer = DockAreaWrapper.wrap_horizontally_with_dockAreaWindow(title='Laps 2D', **widget_out_dict)
laps_merged_out

In [ ]:
import traja
from traja.contrib import rdp
from traja import TrajaCollection

# trajCol1: TrajaCollection = None

print(np.shape(lap_only_pos_df))

a_lap_only_pos_df: pd.DataFrame = lap_only_pos_df.rename(columns={'t': 'time'}, inplace=False)
mask = rdp(a_lap_only_pos_df[['x', 'y']].to_numpy(), algo="iter", return_mask=True)
mask


In [ ]:
np.sum(mask)

In [ ]:
downsampled_lap_only_pos_df = a_lap_only_pos_df[mask]
downsampled_lap_only_pos_df

In [ ]:

downsampled_lap_only_pos_df = traja.resample_time(a_lap_only_pos_df, step_time='250L') ## 500ms
print(np.shape(downsampled_lap_only_pos_df))


rdp(
# (57470, 29)
# (5264, 28)

In [ ]:
from pyphoplacecellanalysis.SpecificResults.MovementBurstDetection import compute_movement_trajectories_from_bursts
    

burst_detector_kwargs = dict(
            min_burst_duration=1.5,      # Minimum burst duration in seconds
            min_rest_duration=0.1,       # Minimum rest period between bursts
            velocity_smoothing=0.15,     # Smoothing for velocity calculation
            bocd_hazard=120,             # Sensitivity of changepoint detection
            # clustering_method='hdbscan',  # Clustering algorithm
            # clustering_method='dbscan',  # Clustering algorithm
            clustering_method='dip',  # Use DipExt from clustpy
            use_gpu=False,             # Set to True if you have CUDA
)
_lap_burst_detection_results, _out_laps = compute_movement_trajectories_from_bursts(curr_active_pipeline, **burst_detector_kwargs)



## 2026-02-10 - Plot Both Laps and interleaved non-laps

In [ ]:
from pyphoplacecellanalysis.PhoPositionalData.plotting.mixins.decoder_plotting_mixins import DecodedTrajectoryMatplotlibPlotter
from pyphoplacecellanalysis.GUI.PyQtPlot.Widgets.ContainerBased.PhoContainerTool import GenericMatplotlibContainer
from pyphoplacecellanalysis.PhoPositionalData.plotting.laps import plot_lap_trajectories_2d, plot_lap_trajectories_3d, plot_lap_trajectories_3d_napari
from pyphoplacecellanalysis.PhoPositionalData.plotting.laps import _plot_helper_add_arrow

from neuropy.utils.matplotlib_helpers import perform_update_title_subtitle
from pyphoplacecellanalysis.PhoPositionalData.plotting.laps import plot_lap_trajectories_3d_napari

from pyphoplacecellanalysis.GUI.PyQtPlot.Widgets.DockAreaWrapper import DockAreaWrapper
from pyphoplacecellanalysis.GUI.PyQtPlot.Widgets.ContainerBased.PhoContainerTool import GenericPyQtGraphContainer

_restore_previous_matplotlib_settings_callback = matplotlib_configuration_update(is_interactive=True, backend='Qt5Agg')

# INPUTS: _lap_burst_detection_results
max_num_subplots: int = 25

# active_container = _container_container.container
active_container = _container_container.masked_container

_fig_out_dict = {}

epoch_names = ['maze']
widget_out_dict = {}
for an_epoch_name in epoch_names:
    # k = 'roam'
    # k = 'sprinkle'
    # k = 'maze_GLOBAL'

    detector, results, analyzer, summary = _lap_burst_detection_results[an_epoch_name]
    a_laps = results['laps_obj'] # Laps
    segments_epoch_df: pd.DataFrame = results['segments_epoch_df']
    segments_epoch_df = segments_epoch_df.epochs.get_valid_df()

    a_sess = curr_active_pipeline.filtered_sessions[an_epoch_name]

    ## INPUTS: a_laps
    # Extract position data for each burst
    df_clean = results['processed_data']
    curr_position_df = df_clean
    # curr_position_df = self.position.to_dataframe() # get the position dataframe from the session
    curr_laps_df = a_laps.to_dataframe()
    curr_position_df = curr_position_df.position.adding_lap_info(laps_df=curr_laps_df, inplace=False)
    
    curr_position_df = curr_position_df.time_point_event.adding_epochs_identity_column(epochs_df=segments_epoch_df, epoch_id_key_name='segment_id', epoch_label_column_name='label', override_time_variable_name='t',
                                                            no_interval_fill_value=-1, should_replace_existing_column=True, drop_non_epoch_events=False)

    
    # curr_position_df_split = curr_position_df.pho.partition_df_dict(partitionColumn='segment_id')
    segment_ids, curr_position_df_split = curr_position_df.pho.partition_df(partitionColumn='segment_id') # : List[pd.DataFrame]
    segment_ids: NDArray = np.array(segment_ids[:max_num_subplots]).astype(int)
    curr_position_df_split: List[pd.DataFrame] = curr_position_df_split[:max_num_subplots]

    # curr_position_df_split = curr_position_df.pho.partition_df_dict(partitionColumn='segment_id')
    # curr_position_df_split ## there are 1000+ of these
    
    ## OUTPUTS: epoch_ids, curr_position_df_split
    
    # PLOT _______________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________ #
    # arrow_concentration_kwargs = dict(
    #     arrow_skip = 50, time_cmap='viridis',
    #     mutation_scale_multiplier = 20, mutation_scale_constant = 1,
    # 	arrow_length_multiplier = 0.2, arrow_length_constant = 0.05,
    # 	arrow_lw = 0.5,
    # )

    arrow_concentration_kwargs = dict(
        arrow_skip = 50, time_cmap='viridis',
        mutation_scale_multiplier = 20, mutation_scale_constant = 1,
        arrow_length_multiplier = 0.05, arrow_length_constant = 0.01,
        arrow_lw = 0.5,
    )

    plot_lap_trajectories_2d_kwargs = dict(
        # curr_num_subplots=(6*5), 
        active_page_index=0, fixed_columns = 6,
    )

    a_decoder = active_container.pf1D_Decoder_dict[an_epoch_name]
    # a_result2D: DecodedFilterEpochsResult = decoded_local_epochs_result.frame_divided_epochs_results[an_epoch_name]
    a_new_global_decoder2D = active_container.pf1D_Decoder_dict[an_epoch_name]
    ## INPUTS: directional_laps_results, decoder_ripple_filter_epochs_decoder_result_dict
    xbin = deepcopy(a_new_global_decoder2D.xbin)
    xbin_centers = deepcopy(a_new_global_decoder2D.xbin_centers)
    ybin_centers = deepcopy(a_new_global_decoder2D.ybin_centers)
    ybin = deepcopy(a_new_global_decoder2D.ybin)

    plotter_kwargs = dict(xbin=xbin, xbin_centers=xbin_centers, ybin=ybin, ybin_centers=ybin_centers)
    ## 2D:
    # Choose the ripple epochs to plot:\
    a_result: DecodedFilterEpochsResult = None # a_decoded_filter_epochs_decoder_result_dict['long'] # 2D
    num_filter_epochs: int = len(curr_position_df_split) # a_result.num_filter_epochs
    print(f'k: {an_epoch_name} has num_filter_epochs: {num_filter_epochs}, len(segment_ids): {len(segment_ids)}')
    a_decoded_traj_plotter = DecodedTrajectoryMatplotlibPlotter(a_result=a_result, **plotter_kwargs)
    fig, axs, laps_pages = a_decoded_traj_plotter.plot_decoded_trajectories_2d(curr_position_df=curr_position_df, epoch_specific_position_dfs=curr_position_df_split, epoch_ids=segment_ids,
                                                                            curr_num_subplots=num_filter_epochs,
                                                                            **plot_lap_trajectories_2d_kwargs, # active_page_index=0, fixed_columns=10,
                                                                            plot_actual_lap_lines=True, use_theoretical_tracks_instead=False)
    _fig_out_dict[an_epoch_name] = GenericMatplotlibContainer.init_from_matplotlib_objects(name=f'splitTrajectories[{an_epoch_name}]', figures=[fig], axes=axs, plots_data={'laps_pages': laps_pages})

    p3, axs, laps_pages3 = _fig_out_dict[an_epoch_name].fig, _fig_out_dict[an_epoch_name].axes, _fig_out_dict[an_epoch_name].plots_data.laps_pages
    perform_update_title_subtitle(fig=_fig_out_dict[an_epoch_name].fig, ax=None, title_string=f"{an_epoch_name} - 2d runs", subtitle_string=f"{an_epoch_name}")
    

    widget_out_dict[an_epoch_name] = p3.canvas.parent()

## END for an_epoch_name in epoch_names...


    
laps_merged_out: GenericPyQtGraphContainer = DockAreaWrapper.wrap_horizontally_with_dockAreaWindow(title='Trajectory Segments via `OptimizedMovementBurstDetector` 2D', **widget_out_dict)
laps_merged_out

In [ ]:
fig.show()

### seemingly laps-only version

In [ ]:
from pyphoplacecellanalysis.GUI.PyQtPlot.Widgets.ContainerBased.PhoContainerTool import GenericMatplotlibContainer
from pyphoplacecellanalysis.PhoPositionalData.plotting.mixins.decoder_plotting_mixins import DecodedTrajectoryMatplotlibPlotter
from pyphoplacecellanalysis.PhoPositionalData.plotting.laps import plot_lap_trajectories_2d, plot_lap_trajectories_3d, plot_lap_trajectories_3d_napari
from pyphoplacecellanalysis.PhoPositionalData.plotting.laps import _plot_helper_add_arrow

from neuropy.utils.matplotlib_helpers import perform_update_title_subtitle
from pyphoplacecellanalysis.PhoPositionalData.plotting.laps import plot_lap_trajectories_3d_napari

from pyphoplacecellanalysis.GUI.PyQtPlot.Widgets.DockAreaWrapper import DockAreaWrapper
from pyphoplacecellanalysis.GUI.PyQtPlot.Widgets.ContainerBased.PhoContainerTool import GenericPyQtGraphContainer

_restore_previous_matplotlib_settings_callback = matplotlib_configuration_update(is_interactive=True, backend='Qt5Agg')



def _subfn_plot_single_decoder_laps(a_sess, k: str, num_pages: int, **kwargs) -> GenericMatplotlibContainer:
    """ plots the grid of trajectories/laps for a single decoding epoch 
    """
    from pyphoplacecellanalysis.GUI.Qt.Widgets.PaginationCtrl.PaginationControlWidget import PaginationControlWidget

    # def _build_page_controls(self, a_past_future_name: str, num_pages: int):
    #     """Build page navigation controls for a trajectory widget using PaginationControlWidget.
        
    #     Note: The controls widget is created but NOT added to the dock here.
    #     It should be added to a container widget that also contains the plot.
    #     This method just creates and stores the control widget.
    #     """
    #     # Check if controls already exist
    #     if a_past_future_name in self.page_controls and 'widget' in self.page_controls[a_past_future_name]:
    #         # Controls already exist, just update them
    #         self._update_page_controls_visibility(a_past_future_name, num_pages)
    #         return
        
    #     # Create PaginationControlWidget
    #     pagination_widget = PaginationControlWidget(n_pages=num_pages)
        
    #     # Connect signals
    #     pagination_widget.jump_to_page.connect(lambda page_idx: self._on_page_jump(a_past_future_name, page_idx))
    #     pagination_widget.jump_previous_page.connect(lambda: self._on_page_change(a_past_future_name, -1))
    #     pagination_widget.jump_next_page.connect(lambda: self._on_page_change(a_past_future_name, 1))
        
    #     # Store references
    #     if a_past_future_name not in self.page_controls:
    #         self.page_controls[a_past_future_name] = {}
    #     self.page_controls[a_past_future_name]['widget'] = pagination_widget
        
    #     # Set initial page index if needed
    #     initial_page_idx = self.trajectory_active_page_idx.get(a_past_future_name, 0)
    #     if initial_page_idx != 0:
    #         pagination_widget.programmatically_update_page_idx(initial_page_idx, block_signals=True)
        
    #     # Set initial visibility
    #     self._update_page_controls_visibility(a_past_future_name, num_pages)


    # def _update_page_controls_visibility(self, a_past_future_name: str, num_pages: int):
    #     """Update visibility and state of page controls based on number of pages."""
    #     if a_past_future_name not in self.page_controls:
    #         return
        
    #     page_controls = self.page_controls[a_past_future_name]
    #     should_show = num_pages > 1
    #     active_page_idx = self.trajectory_active_page_idx.get(a_past_future_name, 0)
        
    #     if 'widget' in page_controls and page_controls['widget'] is not None:
    #         pagination_widget = page_controls['widget']
    #         pagination_widget.setVisible(should_show)
            
    #         if should_show:
    #             # Update the number of pages
    #             if pagination_widget.state.n_pages != num_pages:
    #                 pagination_widget.state.n_pages = num_pages
    #                 pagination_widget._on_update_pagination()
                
    #             # Update the current page index if it changed externally
    #             if pagination_widget.state.current_page_idx != active_page_idx:
    #                 pagination_widget.programmatically_update_page_idx(active_page_idx, block_signals=True)


    # def _on_page_jump(self, a_past_future_name: str, page_idx: int):
    #     """Handle direct page jump from PaginationControlWidget."""
    #     # Update the page index
    #     self.trajectory_active_page_idx[a_past_future_name] = page_idx
        
    #     # Re-render the widget with the new page
    #     self._refresh_trajectory_widget(a_past_future_name)


    # def _on_page_change(self, a_past_future_name: str, direction: int):
    #     """Handle page navigation button clicks (direction: -1 for prev, 1 for next)."""
    #     epochs_pages = self.trajectory_epochs_pages.get(a_past_future_name, [])
    #     num_pages = len(epochs_pages)
    #     if num_pages == 0:
    #         return
        
    #     current_page = self.trajectory_active_page_idx.get(a_past_future_name, 0)
    #     new_page = current_page + direction
    #     new_page = max(0, min(new_page, num_pages - 1))
        
    #     if new_page != current_page:
    #         self.trajectory_active_page_idx[a_past_future_name] = new_page
            
    #         # Update pagination widget if it exists
    #         if a_past_future_name in self.page_controls and 'widget' in self.page_controls[a_past_future_name]:
    #             pagination_widget = self.page_controls[a_past_future_name]['widget']
    #             pagination_widget.programmatically_update_page_idx(new_page, block_signals=True)
            
    #         # Re-render the widget
    #         self._refresh_trajectory_widget(a_past_future_name)

    # ==================================================================================================================================================================================================================================================================================== #
    # BEGIN FUNCTION BODY                                                                                                                                                                                                                                                                  #
    # ==================================================================================================================================================================================================================================================================================== #
    
    # Create pagination controls BEFORE creating container
    # Always create them (even if hidden initially) to ensure single initialization
    # Use num_pages from current data, or 1 as placeholder if no pages yet
    initial_num_pages = max(1, num_pages) if num_pages > 0 else 1


    arrow_concentration_kwargs = dict(
        arrow_skip = 50, time_cmap='viridis',
        mutation_scale_multiplier = 20, mutation_scale_constant = 1,
        arrow_length_multiplier = 0.05, arrow_length_constant = 0.01,
        arrow_lw = 0.5,
    )

    plot_lap_trajectories_2d_kwargs = dict(
        curr_num_subplots=(6*5), active_page_index=0, fixed_columns = 6,
    )

    out3: GenericMatplotlibContainer = plot_lap_trajectories_2d(a_sess, **plot_lap_trajectories_2d_kwargs, use_time_gradient_line=True, arrow_concentration_kwargs=arrow_concentration_kwargs,
                                                                fig_size_inches=None)
    _fig_out_dict[k] = out3
    p3, axs, laps_pages3 = out3.fig, out3.axes, out3.plots_data.laps_pages
    perform_update_title_subtitle(fig=out3.fig, ax=None, title_string=f"{k} - 2d runs", subtitle_string=f"{k}")
    
    widget = p3.canvas.parent()
    out3.ui.owning_widget = widget

    # _build_page_controls(widget, k, initial_num_pages)
    
    return out3 


In [ ]:
# pseudo3D_decoder

_fig_out_dict = {}

epoch_names = ['sprinkle', 'roam']
widget_out_dict = {}
for an_epoch_name in epoch_names:
    # k = 'roam'
    # k = 'sprinkle'
    # k = 'maze_GLOBAL'

    a_sess = curr_active_pipeline.filtered_sessions[an_epoch_name]
    pos_df = a_sess.position.to_dataframe()
    pos_df['speed_xy'] = np.sqrt(np.power(pos_df['velocity_x_smooth'], 2) +  np.power(pos_df['velocity_y_smooth'], 2))

    laps_df: pd.DataFrame = ensure_dataframe(a_sess.laps)
    # for k, a_sess in curr_active_pipeline.filtered_sessions.items():
    #     pos_df = a_sess.position.to_dataframe()
    spikes_df = a_sess.spikes_df.spikes.adding_lap_identity_column(laps_epoch_df=a_sess.laps.to_dataframe(), epoch_id_key_name='lap')
    # spikes_df

    lap_only_pos_df: pd.DataFrame = pos_df.dropna(subset=['lap'], inplace=False)
    lap_only_pos_df['lap'] = lap_only_pos_df['lap'].astype(int)
    lap_pos_df_dict = lap_only_pos_df.pho.partition_df_dict('lap')
    # lap_pos_df_dict
    lap_only_pos_df
    
    # PLOT _______________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________________ #
    # arrow_concentration_kwargs = dict(
    #     arrow_skip = 50, time_cmap='viridis',
    #     mutation_scale_multiplier = 20, mutation_scale_constant = 1,
    # 	arrow_length_multiplier = 0.2, arrow_length_constant = 0.05,
    # 	arrow_lw = 0.5,
    # )

    # arrow_concentration_kwargs = dict(
    #     arrow_skip = 50, time_cmap='viridis',
    #     mutation_scale_multiplier = 20, mutation_scale_constant = 1,
    #     arrow_length_multiplier = 0.05, arrow_length_constant = 0.01,
    #     arrow_lw = 0.5,
    # )

    # plot_lap_trajectories_2d_kwargs = dict(
    #     curr_num_subplots=(6*5), active_page_index=0, fixed_columns = 6,
    # )

    # out3: GenericMatplotlibContainer = plot_lap_trajectories_2d(a_sess, **plot_lap_trajectories_2d_kwargs, use_time_gradient_line=True, arrow_concentration_kwargs=arrow_concentration_kwargs,
    #                                                             fig_size_inches=None)
    # _fig_out_dict[k] = out3
    # p3, axs, laps_pages3 = out3.fig, out3.axes, out3.plots_data.laps_pages
    # perform_update_title_subtitle(fig=out3.fig, ax=None, title_string=f"{k} - 2d runs", subtitle_string=f"{k}")
    

    # widget_out_dict[k] = p3.canvas.parent()
    # _fig_out_dict[k] = plot_lap_trajectories_3d_napari(a_sess, lap_id_dependent_z_offset=4.0)
    # viewer, layer, lap_ids = _fig_out_dict[k]
    # viewer will be shown if show=True
    
    kwargs = dict()

    num_pages: int = 1
    out3: GenericMatplotlibContainer = _subfn_plot_single_decoder_laps(a_sess=a_sess, k=an_epoch_name, num_pages=num_pages, **kwargs)
    widget_out_dict[an_epoch_name] = out3.ui.owning_widget
    
    # p3
    
laps_merged_out: GenericPyQtGraphContainer = DockAreaWrapper.wrap_horizontally_with_dockAreaWindow(title='Laps 2D', **widget_out_dict)
laps_merged_out

In [ ]:

for i, burst in enumerate(results['bursts']):
    mask = (df_clean['t'] >= burst['start']) & (df_clean['t'] <= burst['end'])
    burst_data = df_clean[mask].copy()
    
    # Add burst ID
    burst_data['burst_id'] = i
    
    # You can now analyze each burst individually
    print(f"Burst {i}: {len(burst_data)} points, "
          f"distance={burst['total_distance']:.2f}")

In [ ]:

# Visualize
fig, axes = analyzer.visualize_results(results, save_path='burst_detection_results.png')

In [ ]:
# Run detection on your data
results = detector.detect_bursts(pos_df)

# Analyze results
analyzer = BurstAnalyzer()
summary = analyzer.summarize_bursts(results)


In [ ]:
summary

In [ ]:

# Visualize
analyzer.visualize_results(results, save_path='burst_detection_results.png')

In [ ]:
# Detect bursts
print("\nDetecting bursts with optimized pipeline...")
results = detector.detect_bursts(pos_df)

# Analyze results
analyzer = BurstAnalyzer()
summary = analyzer.summarize_bursts(results)

print(f"\nDetection Summary:")
print(f"  Total bursts: {summary['total_bursts']}")
if summary['total_bursts'] > 0:
    print(f"  Total burst duration: {summary['total_burst_duration']:.1f}s")
    print(f"  Mean burst duration: {summary['mean_burst_duration']:.2f} ± {summary['std_burst_duration']:.2f}s")
    print(f"  Mean burst speed: {summary['mean_burst_speed']:.3f}")
    print(f"  Total distance during bursts: {summary['total_distance']:.2f}")
    if 'burst_frequency' in summary:
        print(f"  Burst frequency: {summary['burst_frequency']:.3f} Hz")

# Display individual bursts
print(f"\nDetected Bursts:")
for i, burst in enumerate(results['bursts']):
    print(f"  Burst {i+1}: {burst['start']:.1f}-{burst['end']:.1f}s "
            f"(dur: {burst['duration']:.1f}s, speed: {burst['mean_speed']:.3f}, "
            f"dist: {burst.get('total_distance', 0):.2f})")



In [ ]:
# Visualize
print("\nGenerating visualization...")
analyzer.visualize_segmentation(results, save_path='optimized_burst_detection.png')


In [ ]:
# Initialize detector (using ensemble method for best results)
detector = MovementBurstDetector(
    min_burst_duration=0.8,
    min_rest_duration=1.2,
    velocity_smoothing=0.15,
    method='combo'  # Best performing ensemble method
)

# Detect bursts
results = detector.detect_bursts(pos_df)

# Display results
print(f"\nDetected {len(results['bursts'])} movement bursts:")
## OUTPUTS: results


In [ ]:

print("-" * 60)
for i, burst in enumerate(results['bursts']):
    print(f"Burst {i+1}:")
    print(f"  Time: {burst['start']:.1f} - {burst['end']:.1f} s "
            f"(duration: {burst['duration']:.1f} s)")
    print(f"  Mean speed: {burst['mean_speed']:.2f}")
    print(f"  Distance: {burst.get('total_distance', 0):.2f}")
    print(f"  Tortuosity: {burst.get('tortuosity', 0):.2f}")
    print()

# Visualize
fig = visualize_bursts(pos_df, results, save_path='burst_detection.png')

# Evaluate different methods
all_results = evaluate_burst_detection(pos_df)


In [ ]:
lap_dir_2D_dict

In [ ]:
laps_df.pho.partition_df_dict('maze_id')


In [ ]:
# pos_df[['velocity_x_smooth', 'velocity_y_smooth']]

# pos_df['speed_xy'] = np.sqrt(np.power(pos_df['velocity_x_smooth'], 2) +  np.power(pos_df['velocity_y_smooth'], 2))
# pos_df

pos_df.plot(x='t', y='speed_xy')

In [ ]:
a_sess.laps.to_dataframe()


In [ ]:
# Performed 3 aggregations grouped on column: 'lap'
each_lap_agg_stats_df = lap_only_pos_df.groupby(['lap']).agg(t_count=('t', 'count'), speed_min=('speed', 'min'), speed_max=('speed', 'max')).reset_index()
each_lap_agg_stats_df

In [ ]:
# a_sess.compute_laps_position_df()
# a_sess.compute_spikes_PBEs()

# if 'lap' not in a_sess.spikes_df.columns:
    # spikes_df = a_sess.spikes_df.spikes.adding_lap_identity_column(laps_epoch_df=a_sess.laps.to_dataframe(), epoch_id_key_name='lap')
spikes_df = a_sess.spikes_df.spikes.adding_lap_identity_column(laps_epoch_df=a_sess.laps.to_dataframe(), epoch_id_key_name='lap')
spikes_df


In [ ]:
from pyphoplacecellanalysis.GUI.PyVista.InteractivePlotter.Mixins.LapsVisualizationMixin import LapsVisualizationMixin
from pyphoplacecellanalysis.PhoPositionalData.plotting.laps import plot_lap_trajectories_3d

## single_combined_plot == True mode (mode 1.):
plotter, laps_pages = plot_lap_trajectories_3d(a_sess, single_combined_plot=True, color_by_speed=False, lap_id_dependent_z_offset=3.5)
plotter.show()


In [ ]:
## single_combined_plot == True mode (mode 1.):
plotter, laps_pages = plot_lap_trajectories_3d(a_sess, single_combined_plot=False, maximum_fixed_columns=10, color_by_speed=False, lap_id_dependent_z_offset=3.5)
plotter.show()


In [ ]:
from pyvistaqt import BackgroundPlotter, MultiPlotter
# p[0,0]

scenes_output_path = Path('data/3d_scenes').resolve()
assert scenes_output_path.exists()


bg_p: BackgroundPlotter = plotter[0,0]
# bg_p.export_gltf(scenes_output_path.joinpath('2025-10-21_3d_laps.gltf').as_posix())
bg_p.export_obj(scenes_output_path.joinpath('2025-10-21_3d_laps.obj').as_posix())
bg_p.export_vtkjs(scenes_output_path.joinpath('2025-10-21_3d_laps').as_posix())
# bg_p.export_html(scenes_output_path.joinpath('2025-10-21_3d_laps.html'))


In [ ]:
## single_combined_plot == False mode (mode 2.):        
p2, laps_pages2 = plot_lap_trajectories_3d(a_sess, single_combined_plot=False, curr_num_subplots=len(curr_active_pipeline.sess.laps.lap_id), active_page_index=1)
p2.show()

In [ ]:
plt.close('all')

In [ ]:
# out3.plots.artists['line_artists']

a_linear_index: int = 0


line

In [ ]:
curr_active_pipeline.export_pipeline_to_h5(override_path=Path('2026-02-10_full_pipeline.h5'))

In [ ]:
curr_exports_path = curr_active_pipeline.get_output_path().joinpath('EXPORTS').resolve()
curr_exports_path.mkdir(exist_ok=True)

curr_out_csv = curr_exports_path.joinpath('position.csv').resolve()

pos_df: pd.DataFrame = curr_active_pipeline.sess.position.to_dataframe()
pos_df.to_csv(curr_out_csv)
print(f'saved: "{curr_out_csv.as_uri()}"')

In [ ]:
pos_df

In [ ]:
for an_epoch_name, a_sess in curr_active_pipeline.filtered_sessions.items():
    a_sess.position


In [ ]:
num_subplots: int = len(out3.plots.artists['line_artists'])
for a_linear_index in np.arange(num_subplots):
    _out_markers =  out3.plots.artists['line_markers'][a_linear_index]
    line = out3.plots.artists['line_artists'][a_linear_index]    
    _out_markers.set_sizes(np.atleast_1d(np.full_like(_out_markers.get_sizes(), 0.1)))
    
p3.canvas.draw_idle()

In [ ]:
line

In [ ]:
line = out3.plots.artists['line_artists'][0]
# --- Extract x/y data back ---
segments = line.get_segments()  # list of (N, 2) arrays
xdata = np.concatenate([seg[:, 0] for seg in segments])
ydata = np.concatenate([seg[:, 1] for seg in segments])

xdata, ydata


In [ ]:
line.get_color().shape # (342, 4)
# line.get_colors()

In [ ]:
line_markers = {}
line_markers['start'] = _plot_helper_add_arrow(line, position=0, position_mode='index', direction='right', size=20, color='green') # start
line_markers['end'] = _plot_helper_add_arrow(line, position=None, position_mode='index', direction='right', size=20, color='yellow') # middle
# _plot_helper_add_arrow(line[0], position=curr_lap_num_points, position_mode='index', direction='right', size=20, color='red') # end
line_markers

In [ ]:
p3.canvas.draw_idle()

In [ ]:
from pyphocorehelpers.plotting.media_output_helpers import save_array_as_video

video_out_path = save_array_as_video(array=active_relative_entropy_results['snapshot_occupancy_weighted_tuning_maps'], video_filename='output/videos/snapshot_occupancy_weighted_tuning_maps.avi', isColor=False)
print(f'video_out_path: {video_out_path}')
reveal_in_system_file_manager(video_out_path)


In [ ]:
active_2d_plot.active_embedded_track_pyqtgraph_time_sync_widgets


In [ ]:
print_keys_if_possible('out', _out_container_new.plots, max_depth=2)


In [ ]:
_out_container_new.plots.parent_root_widget
print_keys_if_possible('', _out_container_new, max_depth=3)


In [ ]:
for name, plotter in _out_container_new.ui.sync_plotters.items():
    plotter.plots_data
    # export_pyqtgraph_plot(plotter.ui.root_graphics_layout_widget)

    plotter.export_

# ❎ 2026-02-12 - Segementation (Just use existing)

In [ ]:
## Correctly adds the laps to the session based on the 'roam' reward zones at least
# hardcoded_params.lap_estimation_parameters ## has no 'lap_estimation_parameters["reward_zones"]'
reward_zones = hardcoded_params.lap_estimation_parameters['reward_zones']
custom_lap_estimation_fn = hardcoded_params.lap_estimation_parameters['custom_lap_estimation_fn']


## Apply it to all relevant sessions
custom_lap_estimation_fn(curr_active_pipeline.sess) ## just takes the session
custom_lap_estimation_fn(curr_active_pipeline.filtered_sessions['roam']) ## just takes the session
custom_lap_estimation_fn(curr_active_pipeline.filtered_sessions['maze_GLOBAL']) ## just takes the session

In [ ]:
from pyphoplacecellanalysis.SpecificResults.PendingNotebookCode import build_non_kdiba_directional_decoders

epochs_decoding_time_bin_size = 1.0
new_decoder_dict, continuous_specific_decoded_results_dict, (contextual_pf2D_Decoder, contextual_pf2D_dict) = build_non_kdiba_directional_decoders(curr_active_pipeline, epochs_decoding_time_bin_size=epochs_decoding_time_bin_size)


In [ ]:
curr_active_pipeline.filtered_sessions['roam'].laps

In [ ]:
from neuropy.core import Laps
from shapely import box
from shapely.geometry import LineString, Point
from neuropy.core.position import PositionAccessor, Position

xmin: float = -85.75619321393464
xmax: float = 112.57838773103435
ymin: float = -96.44772761274268
ymax: float = 98.6220528078153
grid_bin_bounds = box(xmin, ymin, xmax, ymax)

reward_zones_dict = dict(    ## Define the two reward zones
    zone1 = box(xmin, 0.0, -60.0, 40.0),  # box(minx, miny, maxx, maxy, ccw=True)
    zone2 = box(80.0, 0.0, xmax, 40.0), # box(minx, miny, maxx, maxy, ccw=True)
)

@function_attributes(short_name=None, tags=['laps', 'shapely', 'segmentation', 'trajectories', 'position', 'Day4OpenField'], input_requires=[], output_provides=[], uses=['shapely'], used_by=[], creation_date='2026-02-20 06:56', related_items=[])
def build_Bapun_Day4OpenField_laps_from_reward_zones(pos: Position, bapun_Day4OpenField_reward_zones: Dict):
    """ builds correct laps (transitions between the two reward zones on the open field maze for the 'roam' experiment

    Usage:

        curr_session = curr_active_pipeline.filtered_sessions['roam']
        pos: Position = curr_session.position
        laps_obj, pos = build_Bapun_Day4OpenField_laps_from_reward_zones(pos=pos)
        ## Update the current session
        curr_session.position = pos
        curr_session.laps = laps_obj
        ## get the output dataframe:
        pos_df: pd.DataFrame = pos.to_dataframe()
        pos_df

    """
    from shapely import box
    from shapely.geometry import LineString, Point

    ## Define the two reward zones
    zone1 = bapun_Day4OpenField_reward_zones.get('zone1', box(-np.inf, 0.0, -60.0, 40.0)) # box(minx, miny, maxx, maxy, ccw=True)
    zone2 = bapun_Day4OpenField_reward_zones.get('zone2', box(80.0, 0.0, np.inf, 40.0)) # box(minx, miny, maxx, maxy, ccw=True)

    pos_df: pd.DataFrame = pos.to_dataframe()

    points = pos_df.apply(lambda row: Point(row['x'], row['y']), axis=1)
    pos_df['zone_id'] = -1 ## initialize column
    is_zone1 = [p.within(zone1) for p in points]
    is_zone2 = [p.within(zone2) for p in points]

    pos_df.loc[is_zone1, 'zone_id'] = 1
    pos_df.loc[is_zone2, 'zone_id'] = 2

    # changes = pos_df['zone_id'].diff()
    pos_df['zone_id_prev_next'] = list(
        zip(
            pos_df['zone_id'].shift(1),
            pos_df['zone_id'].shift(-1)
        )
    )
    # np.unique(pos_df['zone_id_prev_next']) # [(nan, -1.0), (-1.0, -1.0), (-1.0, 1.0), (-1.0, 2.0), (-1.0, nan), (-1.0, -1.0), (-1.0, 1.0), (-1.0, 2.0), (1.0, -1.0), (1.0, 1.0), (2.0, -1.0), (2.0, 2.0)]
    pos._df['zone_id'] = pos_df['zone_id']
    # pos._data['zone_id_prev_next'] = curr_position_df['zone_id_prev_next'] ## this one we don't need to add, it's just for building laps/transitions
    
    new_lap_epochs_df = []
    last_zone1_exit = None
    last_zone2_exit = None
    last_successful_zone_id = None

    lap_dir_to_lap_dir_integer_mapping = {'L': 0.0, 'R': 1.0}
    for a_row in pos_df.itertuples():
        if a_row.zone_id_prev_next == (-1.0, 1.0):
            ## zone_1_enter
            if (last_successful_zone_id is not None) and (last_successful_zone_id != 1.0) and (last_zone2_exit is not None): ## last condition assumes only 2 zones
                ## this ends a successful leftward lap
                new_lap_epochs_df.append({'lap_dir': lap_dir_to_lap_dir_integer_mapping['L'], 'start': last_zone2_exit, 'stop': a_row.t})
            last_successful_zone_id = 1.0
        elif a_row.zone_id_prev_next == (1.0, -1.0):
            ## zone_1_exit
            last_successful_zone_id = 1.0
            last_zone1_exit = a_row.t
        elif a_row.zone_id_prev_next == (-1.0, 2.0):
            ## zone_2_enter
            if (last_successful_zone_id is not None) and (last_successful_zone_id != 2.0) and (last_zone1_exit is not None): ## last condition assumes only 2 zones
                ## this ends a successful rightward lap
                new_lap_epochs_df.append({'lap_dir': lap_dir_to_lap_dir_integer_mapping['R'], 'start': last_zone1_exit, 'stop': a_row.t})
            last_successful_zone_id = 2.0
        elif a_row.zone_id_prev_next == (2.0, -1.0):
            ## zone_2_exit
            last_successful_zone_id = 2.0
            last_zone2_exit = a_row.t
        else:
            ## catches all self-transitions
            pass

    ## Build the dataframe:
    new_lap_epochs_df = pd.DataFrame.from_records(new_lap_epochs_df)
    new_lap_epochs_df['duration'] = new_lap_epochs_df['stop'] - new_lap_epochs_df['start']
    new_lap_epochs_df['label'] = new_lap_epochs_df.index.astype(int)
    new_lap_epochs_df['lap_id'] = new_lap_epochs_df.index.astype(int)
    # new_lap_epochs_df

    new_laps_obj: Laps = Laps(new_lap_epochs_df)
    new_lap_epochs_df = new_laps_obj.to_dataframe()
    # new_laps_obj

    pos_df = pos_df.position.adding_lap_info(laps_df=new_lap_epochs_df, inplace=False)
    ## OUTPUTS: new_laps_obj, pos_df 
    ## UPDATES: pos_df -- added lap, lap_dir

    # update:
    pos._df['lap'] = pos_df['lap']
    pos._df['lap_dir'] = pos_df['lap_dir']
    
    if 'lap_dir_1D' in pos._df:
        pos._df['lap_dir_1D'] = pos_df['lap_dir']
        
    if 'lap_dir_2D' in pos._df:
        pos._df['lap_dir_2D'] = pos_df['lap_dir']
    
    return new_laps_obj, pos


curr_session = curr_active_pipeline.filtered_sessions['maze']
pos: Position = curr_session.position
laps_obj, pos = build_Bapun_Day4OpenField_laps_from_reward_zones(pos=pos, bapun_Day4OpenField_reward_zones=reward_zones_dict)
## Update the current session
curr_session.position = pos
curr_session.laps = laps_obj
## get the output dataframe:
pos_df: pd.DataFrame = pos.to_dataframe()
pos_df

In [ ]:
laps_obj

In [ ]:
## Update the `pos_df` dataframe on `masked_container.decoding_locality`

# transfer_columns = ['lap', 'lap_dir', 'lap_dir_2D', 'lap_dir_1D', 'zone_id']
# added_columns = ['zone_id']

masked_container.decoding_locality.pos_df.drop(columns=['zone_id'], errors='ignore')
# if 'zone_id' not in masked_container.decoding_locality.pos_df:
# 	masked_container.decoding_locality.pos_df['zone_id'] = -1 ## init to default value

# ## reset all lap values to default only for the roam position bins:
# masked_container.decoding_locality.pos_df.loc[pos_df.index]['zone_id'] = -1
# # masked_container.decoding_locality.pos_df[transfer_columns, pos_df.index] = np.nan
# for a_transfer_column in transfer_columns:
#     masked_container.decoding_locality.pos_df.loc[pos_df.index][a_transfer_column] = np.nan
    
# masked_container.decoding_locality.pos_df.loc[pos_df.index][transfer_columns] = pos_df[transfer_columns]
# masked_container.decoding_locality.pos_df
## This approach works and results in 107 distinct laps:
new_lap_epochs_df: pd.DataFrame = laps_obj.to_dataframe()
masked_container.decoding_locality.pos_df = masked_container.decoding_locality.pos_df.position.adding_lap_info(laps_df=new_lap_epochs_df, inplace=False)
if 'lap_dir_1D' in masked_container.decoding_locality.pos_df:
    masked_container.decoding_locality.pos_df['lap_dir_1D'] = masked_container.decoding_locality.pos_df['lap_dir']
if 'lap_dir_2D' in masked_container.decoding_locality.pos_df:
    masked_container.decoding_locality.pos_df['lap_dir_2D'] = masked_container.decoding_locality.pos_df['lap_dir']
    
masked_container.decoding_locality.pos_df




In [ ]:
curr_active_pipeline.get_all_parameters()
# curr_active_pipeline.sess.config.grid_bin_bounds

In [ ]:
# grid_bin_bounds

# bapun_open_field_grid_bin_bounds = (((-120.0, 120.0), (-120.0, 120.0)))
bapun_open_field_grid_bin_bounds = (((-85.75619321393464, 112.57838773103435), (-96.44772761274268, 98.6220528078153))) #TODO: from (a_decoder.xbin[0], a_decoder.xbin[-1]), (a_decoder.ybin[0], a_decoder.ybin[-1])

curr_active_pipeline.get_all_parameters()
# curr_active_pipeline.update_parameters(grid_bin_bounds = (((-120.0, 120.0), (-120.0, 120.0))))
# curr_active_pipeline.sess.config.grid_bin_bounds = (((-120.0, 120.0), (-120.0, 120.0)))
# curr_active_pipeline.sess.config.grid_bin_bounds = bapun_open_field_grid_bin_bounds


In [ ]:
from neuropy.core import Laps
from shapely import box
from shapely.geometry import LineString, Point
from neuropy.core.position import PositionAccessor, Position
from shapely.plotting import plot_polygon, patch_from_polygon

def _plot_shapely_lap_detect_maze(ax=None):
    xmin: float = -85.75619321393464
    xmax: float = 112.57838773103435
    ymin: float = -96.44772761274268
    ymax: float = 98.6220528078153
    bapun_Day4OpenField_grid_bin_bounds = box(xmin, ymin, xmax, ymax)

    bapun_Day4OpenField_reward_zones = dict(    ## Define the two reward zones
        zone1 = box(xmin, 0.0, -60.0, 40.0),  # box(minx, miny, maxx, maxy, ccw=True)
        zone2 = box(80.0, 0.0, xmax, 40.0), # box(minx, miny, maxx, maxy, ccw=True)
    )

    ## Plot maze sections:
    if ax is None:
        fig, ax = plt.subplots(1, 1)

    _out = {'maze': None, 'reward_zones': {}}
    _out['maze'] = plot_polygon(bapun_Day4OpenField_grid_bin_bounds, ax=ax, color='darkgrey', add_points=False)
    # perform_update_title_subtitle(
    for k, a_zone in bapun_Day4OpenField_reward_zones.items():
        _out['reward_zones'][k] = plot_polygon(a_zone, ax=ax, color='orange', add_points=False)

    return _out


_out = _plot_shapely_lap_detect_maze(ax=None)

# 2026-02-24 - Compare Decoded Occupancy of PBEs between 'roam' and 'sprinkle'

In [ ]:
from pyphoplacecellanalysis.General.Pipeline.Stages.ComputationFunctions.MultiContextComputationFunctions.DirectionalPlacefieldGlobalComputationFunctions import DirectionalDecodersContinuouslyDecodedResult

## Build the new dock track:
## Get the needed data:
directional_decoders_decode_result: DirectionalDecodersContinuouslyDecodedResult = curr_active_pipeline.global_computation_results.computed_data['DirectionalDecodersDecoded']
all_directional_pf1D_Decoder_dict: Dict[str, BasePositionDecoder] = directional_decoders_decode_result.pf1D_Decoder_dict
continuously_decoded_result_cache_dict = directional_decoders_decode_result.continuously_decoded_result_cache_dict
previously_decoded_keys: List[float] = list(continuously_decoded_result_cache_dict.keys()) # [0.03333]
print(F'previously_decoded time_bin_sizes: {previously_decoded_keys}')

time_bin_size: float = directional_decoders_decode_result.most_recent_decoding_time_bin_size
print(f'time_bin_size: {time_bin_size}')
continuously_decoded_dict: Dict[str, DecodedFilterEpochsResult] = directional_decoders_decode_result.most_recent_continuously_decoded_dict
all_directional_continuously_decoded_dict: Dict[types.DecoderName, DecodedFilterEpochsResult] = {k:v for k, v in (continuously_decoded_dict or {}).items() if k in TrackTemplates.get_decoder_names()} ## what is plotted in the `f'{a_decoder_name}_ContinuousDecode'` rows by `AddNewDirectionalDecodedEpochs_MatplotlibPlotCommand`
## OUT: all_directional_continuously_decoded_dict
## Draw the position meas/decoded on the plot widget
## INPUT: fig, ax_list, all_directional_continuously_decoded_dict, track_templates
maze_names = ['roam', 'sprinkle']
# maze_names = ['maze1', 'maze2']
# maze_names = ['maze']


In [ ]:
## Find what computes 'DirectionalDecodersDecoded'

curr_active_pipeline.find_validators_providing_results(probe_provided_result_keys=['DirectionalDecodersDecoded']) # 'directional_decoders_decode_continuous' is the answer

## do compute of ['directional_decoders_decode_continuous']

In [ ]:
curr_active_pipeline.reload_default_computation_functions()
curr_active_pipeline.perform_specific_computation(computation_functions_name_includelist=['directional_decoders_decode_continuous'], computation_kwargs_list=[{'time_bin_size': 0.050,}], enabled_filter_names=['roam', 'sprinkle'], fail_on_exception=True, debug_print=False)

In [ ]:
curr_active_pipeline.filtered_sessions

In [ ]:
from neuropy.core.epoch import ensure_dataframe
from pyphoplacecellanalysis.Analysis.Decoder.reconstruction import BasePositionDecoder, DecodedFilterEpochsResult
from pyphoplacecellanalysis.General.Pipeline.Stages.ComputationFunctions.MultiContextComputationFunctions.DirectionalPlacefieldGlobalComputationFunctions import DirectionalDecodersContinuouslyDecodedResult

directional_decoders_decode_result: DirectionalDecodersContinuouslyDecodedResult = curr_active_pipeline.global_computation_results.computed_data['DirectionalDecodersDecoded']
all_directional_pf1D_Decoder_dict: Dict[str, BasePositionDecoder] = directional_decoders_decode_result.pf1D_Decoder_dict

# time_bins_size: float = 0.025
time_bins_size: float = 0.050
across_all_time_bin_p_x_given_n_dict = {}
pbe_decoded_result_cache_dict = {}

## use keys that exist in all_directional_pf1D_Decoder_dict
decoder_names = maze_names  # e.g. ['maze1', 'maze2'] if those are the decoder keys

for a_decoder_name in decoder_names:
    pbes_df: pd.DataFrame = ensure_dataframe(curr_active_pipeline.filtered_sessions[a_decoder_name].pbe)
    a_decoder: BasePositionDecoder = all_directional_pf1D_Decoder_dict[a_decoder_name]

    decoded_PBEs_result: DecodedFilterEpochsResult = pbe_decoded_result_cache_dict.get(a_decoder_name, None)
    if decoded_PBEs_result is None:
        decoded_PBEs_result = a_decoder.decode_specific_epochs(spikes_df=a_decoder.spikes_df, filter_epochs=pbes_df, decoding_time_bin_size=time_bins_size)
        pbe_decoded_result_cache_dict[a_decoder_name] = decoded_PBEs_result

    n_timebins, flat_time_bin_containers, timebins_p_x_given_n = decoded_PBEs_result.flatten()
    cumm_flattened_p_x_given_n = np.nansum(timebins_p_x_given_n, axis=-1)
    cumm_flattened_p_x_given_n = cumm_flattened_p_x_given_n / float(n_timebins)
    across_all_time_bin_p_x_given_n_dict[a_decoder_name] = cumm_flattened_p_x_given_n

    an_item = pg.image(cumm_flattened_p_x_given_n, title=f"decoded PBE occupancy {a_decoder_name}")
    an_item.show()

across_all_time_bin_p_x_given_n_dict
a_decoder.pf.probability_normalized_occupancy

In [ ]:
a_decoder.pf.occupancy


# ict = {a_decoder_name:np.nansum([np.nansum(v, axis=-1) for v in a_ds.p_x_given_n_list], axis=0) for a_decoder_name, a_ds in decoder_flat_matching_results_list_ds_dict.items()}
# decoded_occupancy_dict


In [ ]:
## DO ONCE:
decoder_cache = {'laps': {}}

In [ ]:
decoder_cache

In [ ]:
from pyphoplacecellanalysis.SpecificResults.PendingNotebookCode import BinnedOccupancyComparisons

occ_comp: BinnedOccupancyComparisons = BinnedOccupancyComparisons()
    
# across_all_time_bin_p_x_given_n_dict, (_subfn_add_single_row, win, cmap, curr_row, curr_session_uid) = occ_comp.plot_decoded_and_measured_occupancies(curr_active_pipeline=curr_active_pipeline, masked_container=masked_container)
# maze_names = hardcoded_params.
maze_names = ['roam', 'sprinkle']
directional_decoders_decode_result = curr_active_pipeline.global_computation_results.computed_data['DirectionalDecodersDecoded']
across_all_time_bin_p_x_given_n_dict, (_subfn_add_single_row, win, cmap, curr_row, curr_session_uid) = occ_comp.plot_decoded_and_measured_occupancies(curr_active_pipeline=curr_active_pipeline, pf1D_Decoder_dict=directional_decoders_decode_result.pf1D_Decoder_dict, epochs_decoded_result_cache_dict={}, decoder_names=maze_names)



In [ ]:
win.size() # PyQt5.QtCore.QSize(926, 916)


In [ ]:
export_fig_path = curr_active_pipeline.get_output_path().joinpath(f"{curr_session_uid.replace('|', '-')}_binned_occupancy_comparisons.svg") # 'H:/Data/Bapun/RatS/Day5TwoNovel/output/bapun-RatS-Day5TwoNovel_binned_occupancy_comparisons.svg'
export_fig_path = occ_comp.export_figure(win=win, out_path=export_fig_path)
export_fig_path

#  ⚓🟢💯 2026-02-24 - Compare Decoded Occupancy of PBEs between 'roam' and 'sprinkle'

In [ ]:
# # ==================================================================================================================================================================================================================================================================================== #
# # MARK: Compute for PBEs:                                                                                                                                                                                                                                                              #
# # ==================================================================================================================================================================================================================================================================================== #
# time_bins_size: float = 0.025

# across_all_time_bin_p_x_given_n_dict = {}
# for a_decoder_name in ['roam', 'sprinkle']:
#     pbes_df: pd.DataFrame = ensure_dataframe(curr_active_pipeline.filtered_sessions[a_decoder_name].pbe)
#     a_decoder = masked_container.pf1D_Decoder_dict[a_decoder_name]
#     ## decode it
#     # a_decoder = masked_container.decoder
#     decoded_PBEs_result: DecodedFilterEpochsResult = masked_container.epochs_decoded_result_cache_dict[time_bins_size][a_decoder_name]
#     # active_decoded_filter_epochs_result: DecodedFilterEpochsResult = a_decoded_PBEs_result

#     ## Mask by valid number of spikes:
#     # a_masked_decoded_PBEs_result, mask_index_tuple = a_decoded_PBEs_result.mask_computed_DecodedFilterEpochsResult_by_required_spike_counts_per_time_bin(spikes_df=a_decoder.spikes_df, masked_bin_fill_mode=masked_bin_fill_mode)
#     # (is_time_bin_active_list, inactive_mask_list, all_time_bin_indicies_list, last_valid_indices_list) = mask_index_tuple
#     ## Outputs: a_masked_decoded_PBEs_result
#     n_timebins, flat_time_bin_containers, timebins_p_x_given_n = decoded_PBEs_result.flatten()

#     cumm_flattened_p_x_given_n = np.nansum(timebins_p_x_given_n, axis=-1)
#     np.shape(cumm_flattened_p_x_given_n) ## 41, 62
#     cumm_flattened_p_x_given_n = cumm_flattened_p_x_given_n / float(n_timebins) ## divided by the total number of flattened timebins between all the epochs
#     across_all_time_bin_p_x_given_n_dict[a_decoder_name] = cumm_flattened_p_x_given_n
#     # np.nansum(decoded_PBEs_result.p_x_given_n_list, axis=-1)
#     an_item = pg.image(cumm_flattened_p_x_given_n, title=f"decoded PBE occupancy {a_decoder_name}") # , title=a_decoder_name
#     an_item.show()

# ## OUTPUTS: across_all_time_bin_p_x_given_n_dict
# across_all_time_bin_p_x_given_n_dict ## need to plot these



In [ ]:
## DO ONCE:
decoder_cache = {'laps': {}}

In [ ]:
@function_attributes(short_name=None, tags=['GREAT'], input_requires=[], output_provides=[], uses=[], used_by=[], creation_date='2026-03-03 16:32', related_items=[])
def plot_decoded_and_measured_occupancies(curr_active_pipeline, masked_container, decoder_cache, decoding_slideby=None): 
    """ plots a comparison between decoded occupancy during various periods and observed """

    def _subfn_add_single_row(win, curr_row, cmap, column_data):
        """Add one row of images + colorbars + labels. column_data is list of (image_array, title) per column. Returns next row index (curr_row + 2)."""
        for col_idx, (image_data, title) in enumerate(column_data):
            win.addLabel(text=title, row=curr_row, col=col_idx * 2, colspan=2)

            vb = win.addViewBox(row=(curr_row + 1), col=col_idx*2)
            img_item = pg.ImageItem(image_data, title=title)
            img_item.setLookupTable(cmap.getLookupTable())
            vb.addItem(img_item)
            vb.setAspectLocked(True)
            cbar = pg.ColorBarItem(colorMap=cmap, label=title, values=(np.nanmin(image_data), np.nanmax(image_data)))
            cbar.setImageItem(img_item)
            win.addItem(cbar, row=(curr_row + 1), col=col_idx*2 + 1)
            
        return curr_row + 2


    # ==================================================================================================================================================================================================================================================================================== #
    # BEGIN FUNCTION BODY                                                                                                                                                                                                                                                                  #
    # ==================================================================================================================================================================================================================================================================================== #
    from pyphoplacecellanalysis.General.Pipeline.Stages.ComputationFunctions.MultiContextComputationFunctions.DirectionalPlacefieldGlobalComputationFunctions import decoding_continuous_cache_key
    
    time_bins_size: float = 0.025

    win = pg.GraphicsLayoutWidget(title="Decoded PBE Occupancy (Roam vs Sprinkle)")
    

    across_all_time_bin_p_x_given_n_dict = {}

    cmap = pg.colormap.get('viridis')  # added

    # Plot decoded PBES __________________________________________________________________________________________________ #
    curr_row: int = 0
    for a_decoder_name in ['roam', 'sprinkle']:
        pbes_df: pd.DataFrame = ensure_dataframe(curr_active_pipeline.filtered_sessions[a_decoder_name].pbe)
        a_decoder = masked_container.pf1D_Decoder_dict[a_decoder_name]
        decoded_PBEs_result: DecodedFilterEpochsResult = masked_container.epochs_decoded_result_cache_dict[time_bins_size][a_decoder_name]
        n_timebins, flat_time_bin_containers, timebins_p_x_given_n = decoded_PBEs_result.flatten()
        cumm_flattened_p_x_given_n = np.nansum(timebins_p_x_given_n, axis=-1)
        cumm_flattened_p_x_given_n = cumm_flattened_p_x_given_n / float(n_timebins)
        across_all_time_bin_p_x_given_n_dict[a_decoder_name] = cumm_flattened_p_x_given_n
    column_data = [(across_all_time_bin_p_x_given_n_dict['roam'], "decoded PBE occupancy roam"), (across_all_time_bin_p_x_given_n_dict['sprinkle'], "decoded PBE occupancy sprinkle")]
    curr_row = _subfn_add_single_row(win, curr_row, cmap, column_data)

    ## OUTPUTS: across_all_time_bin_p_x_given_n_dict


    # Plot measured occupancy as a separate row: _________________________________________________________________________ #
    occupancy_roam = masked_container.pf1D_Decoder_dict['roam'].pf.occupancy
    occupancy_sprinkle = masked_container.pf1D_Decoder_dict['sprinkle'].pf.occupancy
    column_data = [(occupancy_roam, "measured occupancy roam"), (occupancy_sprinkle, "measured occupancy sprinkle")]
    curr_row = _subfn_add_single_row(win, curr_row, cmap, column_data)


    # Plot decoded LAPs __________________________________________________________________________________________________ #

    decoding_time_bin_size: float = 0.100

    _laps_dec_cache_key = decoding_continuous_cache_key(decoding_time_bin_size, decoding_slideby)
    if _laps_dec_cache_key not in decoder_cache['laps']:
        decoder_cache['laps'][_laps_dec_cache_key] = {} ## INIT

    for a_decoder_name in ['roam', 'sprinkle']:
        laps_df: pd.DataFrame = ensure_dataframe(curr_active_pipeline.filtered_sessions[a_decoder_name].laps.to_dataframe())
        a_decoder = masked_container.pf1D_Decoder_dict[a_decoder_name]

        if a_decoder_name not in decoder_cache['laps'][_laps_dec_cache_key]:
            decoder_cache['laps'][_laps_dec_cache_key][a_decoder_name] = a_decoder.decode_specific_epochs(spikes_df=a_decoder.spikes_df, filter_epochs=laps_df, decoding_time_bin_size=decoding_time_bin_size, slideby=decoding_slideby)
            
        a_decoded_laps_epochs_result: DecodedFilterEpochsResult = decoder_cache['laps'][_laps_dec_cache_key][a_decoder_name]
        a_decoded_laps_epochs_result.filter_epochs._df['lap_id'] = a_decoded_laps_epochs_result.filter_epochs._df['label'].astype(int) ## it already is an inbetween_lap_id because it's built from the laps
        a_decoded_laps_epochs_result.filter_epochs._df['original_epoch_idx'] = a_decoded_laps_epochs_result.filter_epochs._df['lap_id'].astype(int) ## it already is an inbetween_lap_id because it's built from the laps
        n_timebins, flat_time_bin_containers, timebins_p_x_given_n = a_decoded_laps_epochs_result.flatten()
        cumm_flattened_p_x_given_n = np.nansum(timebins_p_x_given_n, axis=-1)
        cumm_flattened_p_x_given_n = cumm_flattened_p_x_given_n / float(n_timebins)
        across_all_time_bin_p_x_given_n_dict[a_decoder_name] = cumm_flattened_p_x_given_n


    column_data = [(across_all_time_bin_p_x_given_n_dict['roam'], "decoded Runs occupancy roam"), (across_all_time_bin_p_x_given_n_dict['sprinkle'], "decoded Runs occupancy sprinkle")]
    curr_row = _subfn_add_single_row(win, curr_row, cmap, column_data)


    # Plot measured lap-only occupancy as a separate row: _________________________________________________________________________ #
    occupancy_roam = masked_container.pf1D_Decoder_dict['roam'].pf.probability_normalized_occupancy
    occupancy_sprinkle = masked_container.pf1D_Decoder_dict['sprinkle'].pf.probability_normalized_occupancy
    column_data = [(occupancy_roam, "measured occupancy roam"), (occupancy_sprinkle, "measured occupancy sprinkle")]
    curr_row = _subfn_add_single_row(win, curr_row, cmap, column_data)


    # a_decoder.pf.occupancy
    # column_data = [(across_all_time_bin_p_x_given_n_dict['roam'], "decoded Runs occupancy roam"), (across_all_time_bin_p_x_given_n_dict['sprinkle'], "decoded Runs occupancy sprinkle")]
    # _subfn_add_single_row(win, curr_row, cmap, column_data)
    
    win.show()
    return across_all_time_bin_p_x_given_n_dict, (_subfn_add_single_row, win, cmap, curr_row)


# Call the function to execute code in notebook context:
across_all_time_bin_p_x_given_n_dict, (_subfn_add_single_row, win, cmap, curr_row) = plot_decoded_and_measured_occupancies(curr_active_pipeline=curr_active_pipeline, masked_container=masked_container, decoder_cache=decoder_cache)

# 2026-03-03 - General Occupanies during Laps

## ❌ 2026-03-03 - find the "surprising" or less-likely visited positions/occupanies (to eventually look at the decoding just for those)

In [ ]:
## Resolve laps into pos bins, count number of passes through each pos-bin



In [ ]:

## inversely normalize by occupancy (rarity)

# inverse_occupancy = np.zeros_like(a_decoder.pf.probability_normalized_occupancy)
inverse_occupancy = 1 + (-1.0 * a_decoder.pf.probability_normalized_occupancy) # varies between 0.0 for high occupancy bins and 1.0 for never visited bins
inverse_occupancy



## bin_visit_rarity: how rarely-visited each position bin is, where each bin is normalized to a range of 0.0 for the most common bin and 1.0 for the least common (but still visited) bin
bin_visit_rarity = 1/a_decoder.pf.occupancy ## compute from `a_decoder.pf.occupancy` which is the duration each bin is visited in seconds. If needed, we also have `a_decoder.pf.visited_occupancy_mask` which is a boolean NDArray where bins never visted contain False values.

# imv = pg.ImageView()
# pg.show(inverse_occupancy)
pg.show(bin_visit_rarity)

# title='inverse occupancy'

# a_decoded_laps_epochs_result


In [ ]:
occupancy = a_decoder.pf.occupancy
visited_mask = a_decoder.pf.visited_occupancy_mask.astype(bool)

visited_vals = occupancy[visited_mask]

min_occ = visited_vals.min()
max_occ = visited_vals.max()

bin_visit_rarity = np.where(
    visited_mask,
    (max_occ - occupancy) / (max_occ - min_occ + 1e-12),
    1.0, # np.nan
)

## OUTPUTS: bin_visit_rarity

pg.show(bin_visit_rarity)


## 📌✅ 2026-03-03 - find the "surprising" or less-likely routes and look at the decoding just for those (to eventually look at the decoding just for those)

In [ ]:
from neuropy.core.position import PositionAccessor, Position
from pyphoplacecellanalysis.SpecificResults.PendingNotebookCode import PositionNovelty

## Single filtered_sess an_epoch_name:
# an_epoch_name: str = 'sprinkle'

for an_epoch_name in ['maze']:
    a_sess = curr_active_pipeline.filtered_sessions[an_epoch_name]
    a_sess.position = a_sess.position.fixup_legacy()
    pos_df = a_sess.position.to_dataframe()
    laps_df: pd.DataFrame = ensure_dataframe(a_sess.laps)

    pos_df['speed_xy'] = np.sqrt(np.power(pos_df['velocity_x_smooth'], 2) +  np.power(pos_df['velocity_y_smooth'], 2))
    
    if ('novelty_knn_visited' not in pos_df) or ('novelty_lehman_p90' not in laps_df):
        print(f'needs recompute for {an_epoch_name}')
        active_pos_df, laps_df = PositionNovelty.compute_position_novelty(active_pos_df=pos_df, laps_df=laps_df) ## adds columns: ['novelty_lehman_max', 'novelty_lehman_p90', 'novelty_knn_max', 'novelty_knn_p90'] to laps_df
        ## update the internal dataframes of the session to keep the changes:
        a_sess.laps._df = laps_df.copy()
        a_sess.position._df = active_pos_df.copy()

# 1m 46s

## OUTPUTS: active_pos_df
# active_pos_df

### 2026-03-04 - Resolve laps into pos bins, count number of passes through each pos-bin



In [ ]:

from pyphoplacecellanalysis.SpecificResults.PendingNotebookCode import compute_lap_binned_occupancies

should_plot: bool = True
# should_plot: bool = False
## INPUTS: curr_active_pipeline, should_plot: bool = True

# def compute_lap_binned_matricies(should_plot: bool = True):

lap_occupancies_dict: Dict[types.epoch_index, Dict[str, Any]] = {'roam': {}, 'sprinkle': {}}

win: pg.GraphicsLayoutWidget = None
plotted_keys_list = ['total_laps_occupancy_sec'] # , 'first_visited_lap'

if should_plot:
    if win is None:
        win = pg.GraphicsLayoutWidget(show=True)
        win.setWindowTitle(f'Lap Positions to Occupancy Bins')
        win.resize(1000, 800)
        win.ci.setBorder((50, 50, 100))
        win_layout = win.addLayout()


for an_epoch_name in ['roam', 'sprinkle']:

    if an_epoch_name not in lap_occupancies_dict:
        lap_occupancies_dict[an_epoch_name] = {}

    a_sess = curr_active_pipeline.filtered_sessions[an_epoch_name]
    a_laps_obj: Laps = a_sess.laps
    laps_df: pd.DataFrame = ensure_dataframe(a_laps_obj)
    lap_occupancies_dict[an_epoch_name]['laps_df'] = laps_df

    a_decoder = _container_container.masked_container.pf1D_Decoder_dict[an_epoch_name]
    # a_decoder: DecodedFilterEpochsResult = decoded_local_epochs_result.frame_divided_epochs_results[an_epoch_name]
    ## INPUTS: directional_laps_results, decoder_ripple_filter_epochs_decoder_result_dict
    xbin = deepcopy(a_decoder.xbin)
    xbin_centers = deepcopy(a_decoder.xbin_centers)
    ybin_centers = deepcopy(a_decoder.ybin_centers)
    ybin = deepcopy(a_decoder.ybin)

    ## compute the occupancies per lap:
    occupancy_counts_df_dict, lap_occupancy_n_samples_dict, lap_occupancy_seconds_dict, a_lap_occupancy_matricies_dict = compute_lap_binned_occupancies(a_sess=a_sess, a_decoder=a_decoder)
    ## OUTPUTS: lap_occupancy_seconds_dict
    lap_occupancies_dict[an_epoch_name].update(a_lap_occupancy_matricies_dict)


    if should_plot:
        total_columns = len(plotted_keys_list) * 2 # 1. Double the colspan because each plotted key now takes up TWO columns (Image + Histogram)

        # 1. Use colspan to center the epoch label cleanly across all images below it
        win_layout.addLabel(f"<span style='font-size: 11pt'><b>{an_epoch_name}</b><br> Occupancies</span>", colspan=total_columns)
        win_layout.nextRow()

        # for a_var_key_name, a_var in lap_occupancies_dict[an_epoch_name].items():
            # if a_var_key_name in plotted_keys_list:
        # 2. Iterate strictly through plotted_keys_list for consistent column mapping
        for a_var_key_name in plotted_keys_list:
            if a_var_key_name in lap_occupancies_dict[an_epoch_name]:
                a_var = lap_occupancies_dict[an_epoch_name][a_var_key_name].copy()
                a_var[a_var == 0.0] = np.nan
                # 3. addPlot generates a layout cell with a title. Hiding the axes makes it look like a ViewBox.
                p1 = win_layout.addPlot(title=a_var_key_name)
                p1.hideAxis('left')
                p1.hideAxis('bottom')

                img_v = pg.ImageItem(a_var)
                # img_v.setColorMap(pg.colormap.get('viridis'))
                p1.addItem(img_v)

                # 2. Create the Histogram/Colorbar item
                hist = pg.HistogramLUTItem()
                # 3. Link the histogram to the image item
                hist.setImageItem(img_v)
                # 4. Set the colormap on the histogram (this applies it to the image automatically)
                hist.gradient.loadPreset('viridis')
                # 5. Add the histogram to the layout. It will automatically populate the column to the right of the plot.
                win_layout.addItem(hist)



        ## END for a_var_key_name, a_var in lap_occupanci...
        # 4. Advance to the next row at the very end of the epoch loop
        win_layout.nextRow()



    # (np.nanmin(total_laps_occupancy_sec), np.nanmax(total_laps_occupancy_sec))
    # img_window = pg.ImageWindow(total_laps_occupancy_sec)
    # img_window.setColorMap(pg.colormap.get('viridis'))

    # first_visit_occupancy
    # pg.




## OUTPUTS: lap_occupancies_dict






In [ ]:
# ensure_dataframe(a_laps_obj)

laps_df: pd.DataFrame = ensure_dataframe(a_laps_obj)
laps_df
# laps_df.lap_id

# laps_df.laps
# laps_df

In [ ]:
from pyphoplacecellanalysis.GUI.Napari.napari_helpers import napari_from_layers_dict

occupancy_layers_dict = {'is_new_visit_masks': dict(blending='translucent', colormap='viridis', name=f'{an_epoch_name}_is_new_visit_masks', img_data=is_new_visit_masks), # .transpose(1, 0, 2) reshape to be compatibile with C_i's dimensions 
    'is_new_visit_cumsum_intermediate': dict(blending='translucent', colormap='viridis', name=f'{an_epoch_name}_is_new_visit_cumsum_intermediate', img_data=is_new_visit_cumsum_intermediate),
}


viewer, image_layer_dict = napari_from_layers_dict(occupancy_layers_dict, title='Laps Occupancy Metrics', axis_labels=('xbin', 'ybin', 'lap'))


# for lap in np.arange(n_laps):

#     is_visited_mask[




In [ ]:
a_var.shape # a_var.shape

In [ ]:
from neuropy.utils.misc import compute_paginated_grid_config, RowColTuple, PaginatedGridIndexSpecifierTuple, RequiredSubplotsTuple
from pyphoplacecellanalysis.GUI.PyQtPlot.BinnedImageRenderingWindow import BasicBinnedImageRenderingWindow, LayoutScrollability

_out_lap_occupancy_plotter: Optional[BasicBinnedImageRenderingWindow] = None

plot_row_offset: int = 1

## INPUTS: lap_occupancy_seconds_dict
debug_print: bool = True
n_laps: int = len(lap_occupancy_seconds_dict)
included_lap_ids = np.array(list(lap_occupancy_seconds_dict.keys())) # np.arange(n_laps)

## INPUTS: n_laps, included_lap_idxs
subplot_no_pagination_configuration, included_combined_indicies_pages, page_grid_sizes = compute_paginated_grid_config(n_laps, max_num_columns=6, max_subplots_per_page=100, data_indicies=included_lap_ids, last_figure_subplots_same_layout=True)
num_pages: int = len(included_combined_indicies_pages)
page_idx: int = 0 # page_idx is zero here because we only have one page:

img_item_array = []
other_components_array = []
plot_array = []

for (a_linear_index, curr_row, curr_col, curr_included_lap_id) in included_combined_indicies_pages[page_idx]:
    # Need to convert to page specific:
    curr_page_relative_linear_index: int = np.mod(a_linear_index, int(page_grid_sizes[page_idx].num_rows * page_grid_sizes[page_idx].num_columns))
    curr_page_relative_row: int = np.mod(curr_row, page_grid_sizes[page_idx].num_rows)
    curr_page_relative_col: int = np.mod(curr_col, page_grid_sizes[page_idx].num_columns)
    is_first_column: bool = (curr_page_relative_col == 0)
    is_first_row: bool = (curr_page_relative_row == 0)
    is_last_column: bool = (curr_page_relative_col == (page_grid_sizes[page_idx].num_columns-1))
    is_last_row: bool = (curr_page_relative_row == (page_grid_sizes[page_idx].num_rows-1))
    if debug_print:
        print(f'a_linear_index: {a_linear_index}, curr_page_relative_linear_index: {curr_page_relative_linear_index}, curr_row: {curr_row}, curr_col: {curr_col}, curr_page_relative_row: {curr_page_relative_row}, curr_page_relative_col: {curr_page_relative_col}, curr_included_lap_id: {curr_included_lap_id}')
    a_lap_occupancy_seconds = lap_occupancy_seconds_dict[curr_included_lap_id]
    
    a_plotter_title: str = f'lap[{curr_included_lap_id}] Occupancy'
    if _out_lap_occupancy_plotter is None:
        _out_lap_occupancy_plotter = BasicBinnedImageRenderingWindow(a_lap_occupancy_seconds, xbins=a_decoder.xbin_labels, ybins=a_decoder.ybin_labels, name=f'Lap Occupancy', title=f"Lap Occupancy (sec)", variable_label='Lap Occupancy', scrollability_mode=LayoutScrollability.SCROLLABLE, drop_below_threshold=None, defer_show=False) # , colormap = 'viridis'
        print(f'created new plotter!')        
    else:
        _out_lap_occupancy_plotter.add_data(row=(plot_row_offset + curr_page_relative_row), col=curr_page_relative_col, matrix=a_lap_occupancy_seconds, xbins=a_decoder.xbin_labels, ybins=a_decoder.ybin_labels, name=a_plotter_title, title=a_plotter_title, variable_label=a_plotter_title)
        print(f'added data row: {(plot_row_offset + curr_page_relative_row)}, col: {curr_page_relative_col}')



_out_lap_occupancy_plotter.show()

## Plot by Epoch x Novelty

In [ ]:
plt.close('all')

In [ ]:
from pyphoplacecellanalysis.PhoPositionalData.plotting.mixins.decoder_plotting_mixins import DecodedTrajectoryMatplotlibPlotter
from pyphoplacecellanalysis.GUI.PyQtPlot.Widgets.ContainerBased.PhoContainerTool import GenericMatplotlibContainer
from neuropy.utils.matplotlib_helpers import perform_update_title_subtitle
from neuropy.utils.mixins.indexing_helpers import get_dict_subset
from pyphoplacecellanalysis.SpecificResults.PendingNotebookCode import compute_lap_binned_occupancies


# ==================================================================================================================================================================================================================================================================================== #
# BEGIN FUNCTION BODY                                                                                                                                                                                                                                                                  #
# ==================================================================================================================================================================================================================================================================================== #

# novelty_metric_col: str = 'novelty_lehman_p90'
novelty_metric_col: str = 'novelty_knn_p90'

_fig_out_dict = {}
widget_out_dict = {}
lap_occupancy_matricies_dict_dict: Dict = {}

for an_epoch_name in ['maze']:

    a_sess = curr_active_pipeline.filtered_sessions[an_epoch_name]
    a_sess.position.fixup_legacy()
    active_pos_df = a_sess.position.to_dataframe()
    laps_df: pd.DataFrame = ensure_dataframe(a_sess.laps)

    novel_only_laps_df: pd.DataFrame = laps_df[laps_df[novelty_metric_col] > laps_df[novelty_metric_col].quantile(0.6)]
    # novel_only_laps_df ## 11/107 rows

    ## OUTPUTS: novel_only_laps_df
    a_lap_only_pos_df: pd.DataFrame = active_pos_df.dropna(subset=['lap'], inplace=False)
    a_lap_only_pos_df['lap'] = a_lap_only_pos_df['lap'].astype(int)

    ## add 'is_novel' column to `a_lap_only_pos_df`
    a_lap_only_pos_df['is_novel'] = a_lap_only_pos_df['lap'].isin(novel_only_laps_df['lap_id'])


    curr_position_df_dict = a_lap_only_pos_df.pho.partition_df_dict('is_novel')
    # novel_only_lap_only_pos_df = a_lap_only_pos_df[a_lap_only_pos_df['lap'].isin(novel_only_laps_df['lap_id'])]
    
    # OUTPUTS: novel_only_lap_only_pos_df,  a_lap_only_pos_df


    ## #TODO 2026-03-03 17:54: - [ ] Got `novel_only_lap_only_pos_df`, but forgot the point :[
    a_decoder = _container_container.masked_container.pf1D_Decoder_dict[an_epoch_name]
    # a_result2D: DecodedFilterEpochsResult = decoded_local_epochs_result.frame_divided_epochs_results[an_epoch_name]
    ## INPUTS: directional_laps_results, decoder_ripple_filter_epochs_decoder_result_dict
    xbin = deepcopy(a_decoder.xbin)
    xbin_centers = deepcopy(a_decoder.xbin_centers)
    ybin_centers = deepcopy(a_decoder.ybin_centers)
    ybin = deepcopy(a_decoder.ybin)


    ## compute the occupancies per lap:
    occupancy_counts_df_dict, lap_occupancy_n_samples_dict, lap_occupancy_seconds_dict, a_lap_occupancy_matricies_dict = compute_lap_binned_occupancies(a_sess=a_sess, a_decoder=a_decoder)
    ## OUTPUTS: lap_occupancy_seconds_dict

    plotter_kwargs = dict(xbin=xbin, xbin_centers=xbin_centers, ybin=ybin, ybin_centers=ybin_centers)
    plot_lap_trajectories_2d_kwargs = dict(
        # curr_num_subplots=(6*5), 
        should_include_trajectory_arrows=True,
        active_page_index=0, fixed_columns = 6,
        plot_mode='time_gradient', #'line', 
    )

    for is_novel_key, curr_position_df in curr_position_df_dict.items():

        a_novelty_key: str = {True: 'novel', False: 'non-novel'}[is_novel_key]
        
        ## 2D:
        # Choose the ripple epochs to plot:\
        a_result: DecodedFilterEpochsResult = None # a_decoded_filter_epochs_decoder_result_dict['long'] # 2D
        # curr_position_df_split = curr_position_df.pho.partition_df_dict(partitionColumn='lap')

        # curr_position_df_dict = {'novel': novel_only_lap_only_pos_df, 'non-novel': }
        # curr_position_df
        
        identifier_key: str = f'{an_epoch_name} - {a_novelty_key}'

        lap_ids, partitioned_dfs_list = curr_position_df.pho.partition(partitionColumn='lap')
        
        active_lap_occupancy_seconds_dict = get_dict_subset(lap_occupancy_seconds_dict, subset_includelist=lap_ids, subset_excludelist=None)
        
        assert len(active_lap_occupancy_seconds_dict) == len(lap_ids)
        
        num_filter_epochs: int = len(lap_ids) # a_result.num_filter_epochs
        print(f'k: {identifier_key} has num_filter_epochs: {num_filter_epochs}, len(lap_ids): {len(lap_ids)}')
        a_decoded_traj_plotter = DecodedTrajectoryMatplotlibPlotter(a_result=a_result, **plotter_kwargs)
        fig, axs, laps_pages = a_decoded_traj_plotter.plot_decoded_trajectories_2d(curr_position_df=curr_position_df, epoch_specific_position_dfs=partitioned_dfs_list, epoch_ids=lap_ids,
                                                                                curr_num_subplots=num_filter_epochs,
                                                                                **plot_lap_trajectories_2d_kwargs, # active_page_index=0, fixed_columns=10,
                                                                                posteriors=active_lap_occupancy_seconds_dict,
                                                                                plot_actual_lap_lines=True, use_theoretical_tracks_instead=False)
        


        # lap_occupancy_seconds_dict

        _fig_out_dict[an_epoch_name] = GenericMatplotlibContainer.init_from_matplotlib_objects(name=f'splitTrajectories[{identifier_key}]', figures=[fig], axes=axs, plots_data={'laps_pages': laps_pages})

        p3, axs, laps_pages3 = _fig_out_dict[an_epoch_name].fig, _fig_out_dict[an_epoch_name].axes, _fig_out_dict[an_epoch_name].plots_data.laps_pages
        perform_update_title_subtitle(fig=_fig_out_dict[an_epoch_name].fig, ax=None, title_string=f"{identifier_key} - 2d runs", subtitle_string=f"{identifier_key}")
        # widget_out_dict[an_epoch_name] = p3.canvas.parent()
        
    ## END for is_novel_key, curr_position_df in curr_position_df_dict.items()...
    
    lap_occupancy_matricies_dict_dict[an_epoch_name] = a_lap_occupancy_matricies_dict
## END for an_epoch_name in ['roam', 'sprinkle']

# return lap_occupancy_matricies_dict_dict, _fig_out_dict



## Plot Just by Epoch (no novelty)

In [ ]:
from pyphoplacecellanalysis.PhoPositionalData.plotting.mixins.decoder_plotting_mixins import DecodedTrajectoryMatplotlibPlotter
from pyphoplacecellanalysis.GUI.PyQtPlot.Widgets.ContainerBased.PhoContainerTool import GenericMatplotlibContainer
from neuropy.utils.matplotlib_helpers import perform_update_title_subtitle
from neuropy.utils.mixins.indexing_helpers import get_dict_subset
from pyphoplacecellanalysis.SpecificResults.PendingNotebookCode import compute_lap_binned_occupancies
from pyphoplacecellanalysis.PhoPositionalData.plotting.mixins.decoder_plotting_mixins import RenderColoringMode
from scipy import ndimage

# ==================================================================================================================================================================================================================================================================================== #
# BEGIN FUNCTION BODY                                                                                                                                                                                                                                                                  #
# ==================================================================================================================================================================================================================================================================================== #

# novelty_metric_col: str = 'novelty_lehman_p90'
# novelty_metric_col: str = 'novelty_knn_p90'

_fig_out_dict = {}
widget_out_dict = {}

for an_epoch_name in ['maze']:

    a_sess = curr_active_pipeline.filtered_sessions[an_epoch_name]
    _ = a_sess.position.fixup_legacy()
    active_pos_df = a_sess.position.to_dataframe()
    laps_df: pd.DataFrame = ensure_dataframe(a_sess.laps)

    lap_novelty_score: NDArray = laps_df['lap_novelty_score'].to_numpy()
    
    def override_title_formatter_fn(curr_epoch_id) -> str:
        """ captures `laps_df` to get the data needed for the title """
        included_lap_info_columns = ['lap_novelty_score']
        curr_lap_info = laps_df[laps_df['lap_id'] == curr_epoch_id]
        assert len(curr_lap_info) == 1
        curr_lap_info: Dict = curr_lap_info.iloc[0].to_dict() # {'lap_dir': 0, 'start': 7449.502388840895, 'stop': 7467.4107279169175, 'duration': 17.908339076022457, 'label': '1', 'lap_id': 1, 'novelty_lehman_max': 0.4827321516328119, 'novelty_lehman_p90': 0.2796114892604357, 'novelty_knn_max': 1.7460081712930897, 'novelty_knn_p90': 0.9879667712313558, 'lap_novelty_score': 125}
        curr_lap_label_text: str = 'Epoch[{}]: t({:.2f}, {:.2f})'.format(curr_epoch_id, curr_lap_info['start'], curr_lap_info['stop'])
        for a_col_name in included_lap_info_columns:
            curr_lap_label_text = curr_lap_label_text + f' {curr_lap_info[a_col_name]}'
        return curr_lap_label_text
        
    
    ## OUTPUTS: novel_only_laps_df
    a_lap_only_pos_df: pd.DataFrame = active_pos_df.dropna(subset=['lap'], inplace=False)
    a_lap_only_pos_df['lap'] = a_lap_only_pos_df['lap'].astype(int)
    
    a_decoder = _container_container.masked_container.pf1D_Decoder_dict[an_epoch_name]
    # a_result2D: DecodedFilterEpochsResult = decoded_local_epochs_result.frame_divided_epochs_results[an_epoch_name]
    ## INPUTS: directional_laps_results, decoder_ripple_filter_epochs_decoder_result_dict
    xbin = deepcopy(a_decoder.xbin)
    xbin_centers = deepcopy(a_decoder.xbin_centers)
    ybin_centers = deepcopy(a_decoder.ybin_centers)
    ybin = deepcopy(a_decoder.ybin)

    ## compute the occupancies per lap:
    occupancy_counts_df_dict, lap_occupancy_n_samples_dict, lap_occupancy_seconds_dict, a_lap_occupancy_matricies_dict = compute_lap_binned_occupancies(a_sess=a_sess, a_decoder=a_decoder)
    ## OUTPUTS: lap_occupancy_seconds_dict

    arrow_concentration_kwargs = dict(
            # time_cmap='viridis', arrow_color_scheme=RenderColoringMode.ANGLE,
            time_cmap='viridis', arrow_color_scheme=RenderColoringMode.TIME,
            # arrow_skip=30, mutation_scale_multiplier=20, mutation_scale_constant=1, arrow_length_multiplier=0.2, arrow_length_constant=0.05, arrow_lw=0.5, arrow_opacity=0.8,
            arrow_skip=30, mutation_scale_multiplier=10, mutation_scale_constant=1.0, arrow_length_multiplier=0.2, arrow_length_constant=0.05, arrow_lw=0.5, arrow_opacity=0.8,
    )

    plotter_kwargs = dict(xbin=xbin, xbin_centers=xbin_centers, ybin=ybin, ybin_centers=ybin_centers)
    plot_lap_trajectories_2d_kwargs = dict(
        # curr_num_subplots=(6*5), 
        should_include_trajectory_arrows=True,
        active_page_index=0, fixed_columns = 8,
        plot_mode='time_gradient', #'line', 
        override_title_formatter_fn=override_title_formatter_fn,
        arrow_concentration_kwargs=arrow_concentration_kwargs,
    )

    curr_position_df = deepcopy(a_lap_only_pos_df)
    
    ## 2D:
    # Choose the ripple epochs to plot:\
    a_result: DecodedFilterEpochsResult = None # a_decoded_filter_epochs_decoder_result_dict['long'] # 2D
    
    identifier_key: str = f'{an_epoch_name}'
    # identifier_key: str = f'{an_epoch_name} - {a_novelty_key}'
    
    lap_ids, partitioned_dfs_list = curr_position_df.pho.partition(partitionColumn='lap')
    
    # dict(map(np.arange(len(lap_ids),

    is_new_visit_masks = a_lap_occupancy_matricies_dict['is_new_visit_masks']
    # n_laps = np.shape(is_new_visit_masks)[-1]
    # is_new_visit_masks = np.

    # If mask is your binary NDArray (0s and 1s or Booleans)
    # iterations=3 will expand it by roughly 3 pixels on each side
    # expanded_mask = ndimage.binary_dilation(mask, iterations=3)

    # active_is_new_visit_masks_dict = {lap_id:np.squeeze(is_new_visit_masks[:, :, i]) for i, lap_id in enumerate(lap_ids)}
    active_is_new_visit_masks_dict = {lap_id:ndimage.binary_dilation(np.squeeze(is_new_visit_masks[:, :, i]), iterations=3) for i, lap_id in enumerate(lap_ids)}

    
    ## need to invert the mask:
    # active_is_new_visit_masks_dict = {lap_id:(-1.0 * np.squeeze(is_new_visit_masks[:, :, i])) + 1.0 for i, lap_id in enumerate(lap_ids)} ## invert the mask
    # active_is_new_visit_masks_dict = {lap_id:((-1.0 * a_mask) + 1.0) for lap_id, a_mask in active_is_new_visit_masks_dict.items()} ## invert the mask

    # active_lap_occupancy_seconds_dict = get_dict_subset(lap_occupancy_seconds_dict, subset_includelist=lap_ids, subset_excludelist=None)
    # assert len(active_lap_occupancy_seconds_dict) == len(lap_ids)

    # active_is_new_visit_masks_dict = get_dict_subset(is_new_visit_masks, subset_includelist=lap_ids, subset_excludelist=None)
    # assert len(active_is_new_visit_masks_dict) == len(lap_ids)

    posterior_cmap = 'gray_r' ## reversed so the 1.0 values are black (and therefore visible)
    num_filter_epochs: int = len(lap_ids) # a_result.num_filter_epochs
    print(f'k: {identifier_key} has num_filter_epochs: {num_filter_epochs}, len(lap_ids): {len(lap_ids)}')
    a_decoded_traj_plotter = DecodedTrajectoryMatplotlibPlotter(a_result=a_result, **plotter_kwargs)
    fig, axs, laps_pages = a_decoded_traj_plotter.plot_decoded_trajectories_2d(curr_position_df=curr_position_df, epoch_specific_position_dfs=partitioned_dfs_list, epoch_ids=lap_ids,
                                                                            curr_num_subplots=num_filter_epochs,
                                                                            **plot_lap_trajectories_2d_kwargs, # active_page_index=0, fixed_columns=10,
                                                                            # posteriors=active_lap_occupancy_seconds_dict,
                                                                            # posteriors=active_is_new_visit_masks_dict, posterior_cmap=posterior_cmap, 
                                                                            plot_actual_lap_lines=True, use_theoretical_tracks_instead=False)
    
    # lap_occupancy_seconds_dict

    _fig_out_dict[an_epoch_name] = GenericMatplotlibContainer.init_from_matplotlib_objects(name=f'splitTrajectories[{identifier_key}]', figures=[fig], axes=axs, plots_data={'laps_pages': laps_pages})

    p3, axs, laps_pages3 = _fig_out_dict[an_epoch_name].fig, _fig_out_dict[an_epoch_name].axes, _fig_out_dict[an_epoch_name].plots_data.laps_pages
    perform_update_title_subtitle(fig=_fig_out_dict[an_epoch_name].fig, ax=None, title_string=f"{identifier_key} - 2d runs", subtitle_string=f"{identifier_key}")

    # widget_out_dict[an_epoch_name] = p3.canvas.parent()



In [ ]:
from pyphoplacecellanalysis.Analysis.Decoder.computer_vision import ComputerVisionComputations

# Visualization ______________________________________________________________________________________________________ #
from pyphoplacecellanalysis.GUI.PyQtPlot.BinnedImageRenderingWindow import BasicBinnedImageRenderingWindow, LayoutScrollability

out = ComputerVisionComputations.interactive_image_preview(input_img=is_new_visit_masks[:, :, 0], blur_h_sigma=2, blur_v_sigma=2, hessian_sigma=1.0) # (decoders_dict=decoders_dict, binned_x_transition_matrix_higher_order_list_dict=binned_x_transition_matrix_higher_order_list_dict)
out

In [ ]:
import numpy as np
from scipy import ndimage

def dilate_mask(mask, radius=2):
    """
    Dilates a binary mask using a circular/spherical structuring element.
    """
    # 1. Create a coordinate grid centered at zero
    # The size of the kernel needs to be odd to have a clear center
    size = 2 * radius + 1
    coords = np.arange(size) - radius
    
    # 2. Generate a grid for N-dimensions
    # This works for 2D, 3D, or higher
    grids = np.meshgrid(*[coords for _ in range(mask.ndim)], indexing='ij')
    
    # 3. Calculate Euclidean distance from center and threshold
    dist_sq = sum(g**2 for g in grids)
    structuring_element = dist_sq <= radius**2
    
    # 4. Perform dilation
    dilated_mask = ndimage.binary_dilation(mask, structure=structuring_element)
    
    return dilated_mask

# Example usage:
# mask = np.zeros((100, 100), dtype=bool)
# mask[50, 50] = True
dilated = dilate_mask(mask=is_new_visit_masks[:, :, 0], radius=3)
dilated

In [ ]:
'arrow_color_scheme': RenderColoringMode.ANGLE, 'mut'

In [ ]:
np.shape(is_new_visit_masks)


In [ ]:
from pyphoplacecellanalysis.GUI.Napari.napari_helpers import napari_from_layers_dict

occupancy_layers_dict = {'is_new_visit_masks': dict(blending='translucent', colormap='viridis', name=f'{an_epoch_name}_is_new_visit_masks', img_data=a_lap_occupancy_matricies_dict['roam']['is_new_visit_masks']), # .transpose(1, 0, 2) reshape to be compatibile with C_i's dimensions 
    'is_new_visit_cumsum_intermediate': dict(blending='translucent', colormap='viridis', name=f'{an_epoch_name}_is_new_visit_cumsum_intermediate', img_data=is_new_visit_cumsum_intermediate),
}


viewer, image_layer_dict = napari_from_layers_dict(occupancy_layers_dict, title='Laps Occupancy Metrics', axis_labels=('xbin', 'ybin', 'lap'))


# for lap in np.arange(n_laps):

#     is_visited_mask[




In [ ]:
laps_df

In [ ]:
curr_epoch_id = 1

included_lap_info_columns = ['lap_novelty_score']
curr_lap_info = laps_df[laps_df['lap_id'] == curr_epoch_id]
assert len(curr_lap_info) == 1
curr_lap_info: Dict = curr_lap_info.iloc[0].to_dict() # {'lap_dir': 0, 'start': 7449.502388840895, 'stop': 7467.4107279169175, 'duration': 17.908339076022457, 'label': '1', 'lap_id': 1, 'novelty_lehman_max': 0.4827321516328119, 'novelty_lehman_p90': 0.2796114892604357, 'novelty_knn_max': 1.7460081712930897, 'novelty_knn_p90': 0.9879667712313558, 'lap_novelty_score': 125}
curr_lap_label_text = 'Epoch[{}]: t({:.2f}, {:.2f})'.format(curr_epoch_id, curr_lap_info['start'], curr_lap_info['stop'])
for a_col_name in included_lap_info_columns:
	curr_lap_label_text = curr_lap_label_text + f' {lap_novelty_score}'
return curr_lap_label_text

In [ ]:
## Resolve laps into pos bins, count number of passes through each pos-bin


dict(roam = [],
	 sprinkle = [],
)



lap_occupancy_seconds_dict

In [ ]:
## compute the occupany of only and plot that:
df: pd.DataFrame = a_lap_only_pos_df.position.adding_binned_position_columns(xbin_edges=a_decoder.xbin, ybin_edges=a_decoder.ybin)

# Performed 1 aggregation grouped on columns: 'binned_x', 'binned_y'
occupancy_counts_df: pd.DataFrame = df.groupby(['binned_x', 'binned_y']).agg(t_count=('t', 'count')).reset_index()
occupancy_counts_df


# 2026-04-28 - Momentum Analysis of Behavior + PBEs

In [ ]:
from pyphoplacecellanalysis.SpecificResults.PendingNotebookCode import MomentumHelpers
from neuropy.core.epoch import ensure_dataframe
from pyphoplacecellanalysis.Analysis.Decoder.reconstruction import BasePositionDecoder, DecodedFilterEpochsResult
from pyphoplacecellanalysis.General.Pipeline.Stages.ComputationFunctions.MultiContextComputationFunctions.DirectionalPlacefieldGlobalComputationFunctions import DirectionalDecodersContinuouslyDecodedResult
# from pyphoplacecellanalysis.General.Pipeline.Stages.ComputationFunctions.MultiContextComputationFunctions.PredictiveDecodingComputations import PredictiveDecoding, DecodingLocalityMeasures, PredictiveDecodingComputationsContainer, PredictiveDecodingComputationsContainerContainer
from pyphoplacecellanalysis.Analysis.Decoder.reconstruction import BayesianPlacemapPositionDecoder


maze_names = ['roam', 'sprinkle']

## INPUTS: maze_names = ['roam', 'sprinkle']
MomentumHelpers.main_behavior_momentum_analysis(curr_active_pipeline=curr_active_pipeline, maze_names=maze_names)



## do the same for decoded LAPs __________________________________________________________________________________________________ #
def _safe_epoch_ids_from_label_or_order(filter_epochs_df: pd.DataFrame, label_column_name: str = 'label') -> pd.Series:
    """Build int epoch IDs from numeric labels, falling back to row order for blank/non-numeric labels."""
    if label_column_name in filter_epochs_df.columns:
        label_ids = pd.to_numeric(filter_epochs_df[label_column_name], errors='coerce')
    else:
        label_ids = pd.Series(np.nan, index=filter_epochs_df.index)
    fallback_ids = pd.Series(np.arange(len(filter_epochs_df), dtype=int), index=filter_epochs_df.index)
    return label_ids.fillna(fallback_ids).astype(int)


directional_decoders_decode_result: DirectionalDecodersContinuouslyDecodedResult = curr_active_pipeline.global_computation_results.computed_data['DirectionalDecodersDecoded']
all_directional_pf1D_Decoder_dict: Dict[str, BasePositionDecoder] = directional_decoders_decode_result.pf1D_Decoder_dict

## INPUTS: all_directional_pf1D_Decoder_dict
decoding_time_bin_size: float = 0.100

decoder_cache = {'laps': {}, }

if decoding_time_bin_size not in decoder_cache['laps']:
    decoder_cache['laps'][decoding_time_bin_size] = {} ## INIT

decoder_laps_result_out_dicts = {}

for a_decoder_name in maze_names:
    laps_df: pd.DataFrame = ensure_dataframe(curr_active_pipeline.filtered_sessions[a_decoder_name].laps.to_dataframe())
    a_decoder: BasePositionDecoder = all_directional_pf1D_Decoder_dict[a_decoder_name]
    
    if a_decoder_name not in decoder_cache['laps'][decoding_time_bin_size]:
        decoder_cache['laps'][decoding_time_bin_size][a_decoder_name] = a_decoder.decode_specific_epochs(spikes_df=a_decoder.spikes_df, filter_epochs=laps_df, decoding_time_bin_size=decoding_time_bin_size)
        
    a_decoded_laps_epochs_result: DecodedFilterEpochsResult = decoder_cache['laps'][decoding_time_bin_size][a_decoder_name]
    a_decoded_laps_epochs_result.filter_epochs._df['lap_id'] = _safe_epoch_ids_from_label_or_order(a_decoded_laps_epochs_result.filter_epochs._df) ## Prefer labels when numeric; fall back to row order for blank/non-numeric labels.
    a_decoded_laps_epochs_result.filter_epochs._df['original_epoch_idx'] = a_decoded_laps_epochs_result.filter_epochs._df['lap_id'].astype(int) ## it already is an inbetween_lap_id because it's built from the laps

    decoder_laps_result_out_dicts[a_decoder_name] = MomentumHelpers.perform_decoded_epochs_momentum_analysis(a_decoded_PBEs_result=a_decoded_laps_epochs_result, figure_name=f'Laps[{a_decoder_name}]')
    
    # n_timebins, flat_time_bin_containers, timebins_p_x_given_n = a_decoded_laps_epochs_result.flatten()
    # cumm_flattened_p_x_given_n = np.nansum(timebins_p_x_given_n, axis=-1)
    # cumm_flattened_p_x_given_n = cumm_flattened_p_x_given_n / float(n_timebins)
    # across_all_time_bin_p_x_given_n_dict[a_decoder_name] = cumm_flattened_p_x_given_n


In [ ]:
# session_epochs: Epoch = BapunDataSessionFormatRegisteredClass.session_fixup_epochs(sess=curr_active_pipeline.sess)
session_epochs: Epoch = BapunDataSessionFormatRegisteredClass.session_fixup_epochs(sess=curr_active_pipeline.sess, override_extant=True)
session_epochs

In [ ]:
for k in epochs:
    cr = curr_active_pipeline.computation_results.get(k, None)
    print(f"\n{k}: exists={cr is not None}")
    if cr is not None:
        keys = list(cr.computed_data.keys())
        print("has pf2D:", 'pf2D' in keys, "has pf2D_Decoder:", 'pf2D_Decoder' in keys)
        if hasattr(cr, 'accumulated_errors'):
            print("errors:", cr.accumulated_errors)

In [ ]:
epochs = hardcoded_params.non_global_activity_session_names  # e.g. ['roam', 'sprinkle']

# 0) sanity: do these epoch result containers exist?
print("missing epochs:", [k for k in epochs if k not in curr_active_pipeline.computation_results])

In [ ]:

# 1) ensure pf1D/pf2D exist
curr_active_pipeline.reload_default_computation_functions()

curr_active_pipeline.perform_specific_computation(
    computation_functions_name_includelist=['pf_computation'],
    enabled_filter_names=epochs, fail_on_exception=True, debug_print=False
)


In [ ]:

start     stop               label  duration
0      0.0   7124.0                 pre    7124.0
1   7125.0  11745.0         maze_GLOBAL    4620.0
2   8031.0  10421.0                roam    2390.0
7   8031.0  11745.0  maze_GLOBAL_GLOBAL    3714.0
3  10477.0  11745.0            sprinkle    1268.0
4  11746.0  18957.0                  sd    7211.0
5  11746.0  37933.0                post   26187.0
6  18958.0  37933.0                  rs   18975.0

[8 rows x 4 columns]

In [ ]:

# 2) build pf1D_Decoder/pf2D_Decoder
curr_active_pipeline.reload_default_computation_functions()

curr_active_pipeline.perform_specific_computation(
    computation_functions_name_includelist=['position_decoding'],
    enabled_filter_names=epochs, fail_on_exception=True, debug_print=False
)

# 3) verify
for k in epochs:
    print(k, sorted(curr_active_pipeline.computation_results[k].computed_data.keys()))

In [ ]:
from copy import deepcopy
from neuropy.core.session.Formats.Specific.BapunDataSessionFormat import BapunDataSessionFormatRegisteredClass
from pyphoplacecellanalysis.SpecificResults.PendingNotebookCode import build_contextual_pf2D_decoder, decode_using_contextual_pf2D_decoder
from pyphoplacecellanalysis.General.Pipeline.Stages.ComputationFunctions.MultiContextComputationFunctions.DirectionalPlacefieldGlobalComputationFunctions import DirectionalDecodersContinuouslyDecodedResult, decoding_continuous_cache_key

# 1) Pick Bapun activity epochs
hardcoded_params = BapunDataSessionFormatRegisteredClass._get_session_specific_parameters(session_context=curr_active_pipeline.get_session_context())
epochs_to_create_global_from_names = hardcoded_params.non_global_activity_session_names  # usually ['roam', 'sprinkle'] or ['maze1','maze2']

# 2) Build merged contextual pseudo2D decoder
pf2D_Decoder_dict, contextual_pf2D, contextual_pf2D_Decoder = build_contextual_pf2D_decoder(
    curr_active_pipeline,
    epochs_to_create_global_from_names=epochs_to_create_global_from_names
)

# 3) Decode globally at desired bin size
desired_t_bin = 0.060
decoded_result, global_only_epoch = decode_using_contextual_pf2D_decoder(
    curr_active_pipeline,
    contextual_pf2D_Decoder=contextual_pf2D_Decoder,
    desired_global_created_epoch_name=hardcoded_params.global_session_name,
    active_laps_decoding_time_bin_size=desired_t_bin,
    slideby=None
)

# 4) Store canonical global key
global_spikes_df = deepcopy(curr_active_pipeline.sess.spikes_df)
curr_active_pipeline.global_computation_results.computed_data['DirectionalDecodersDecoded'] = DirectionalDecodersContinuouslyDecodedResult(
    pf1D_Decoder_dict=pf2D_Decoder_dict,
    pseudo2D_decoder=contextual_pf2D_Decoder,
    spikes_df=global_spikes_df,
    continuously_decoded_result_cache_dict={decoding_continuous_cache_key(desired_t_bin, None): {'pseudo2D': decoded_result}}
)

## 11m 38.0s

### Overflow

In [ ]:

from pyphoplacecellanalysis.General.Pipeline.Stages.ComputationFunctions.MultiContextComputationFunctions.PredictiveDecodingComputations import PredictiveDecoding, DecodingLocalityMeasures, PredictiveDecodingComputationsContainer, PredictiveDecodingComputationsContainerContainer
from pyphoplacecellanalysis.Analysis.Decoder.reconstruction import BayesianPlacemapPositionDecoder


## do the same for decoded LAPs __________________________________________________________________________________________________ #
def _safe_epoch_ids_from_label_or_order(filter_epochs_df: pd.DataFrame, label_column_name: str = 'label') -> pd.Series:
    """Build int epoch IDs from numeric labels, falling back to row order for blank/non-numeric labels."""
    if label_column_name in filter_epochs_df.columns:
        label_ids = pd.to_numeric(filter_epochs_df[label_column_name], errors='coerce')
    else:
        label_ids = pd.Series(np.nan, index=filter_epochs_df.index)
    fallback_ids = pd.Series(np.arange(len(filter_epochs_df), dtype=int), index=filter_epochs_df.index)
    return label_ids.fillna(fallback_ids).astype(int)


directional_decoders_decode_result: DirectionalDecodersContinuouslyDecodedResult = curr_active_pipeline.global_computation_results.computed_data['DirectionalDecodersDecoded']
all_directional_pf1D_Decoder_dict: Dict[str, BasePositionDecoder] = directional_decoders_decode_result.pf1D_Decoder_dict

## INPUTS: all_directional_pf1D_Decoder_dict
decoding_time_bin_size: float = 0.100

# decoder_cache = {'laps': {}, 

if decoding_time_bin_size not in decoder_cache['laps']:
    decoder_cache['laps'][decoding_time_bin_size] = {} ## INIT

decoder_laps_result_out_dicts = {}

for a_decoder_name in decoder_names:
    laps_df: pd.DataFrame = ensure_dataframe(curr_active_pipeline.filtered_sessions[a_decoder_name].laps.to_dataframe())
    a_decoder: BasePositionDecoder = all_directional_pf1D_Decoder_dict[a_decoder_name]
    
    if a_decoder_name not in decoder_cache['laps'][decoding_time_bin_size]:
        decoder_cache['laps'][decoding_time_bin_size][a_decoder_name] = a_decoder.decode_specific_epochs(spikes_df=a_decoder.spikes_df, filter_epochs=laps_df, decoding_time_bin_size=decoding_time_bin_size)
        
    a_decoded_laps_epochs_result: DecodedFilterEpochsResult = decoder_cache['laps'][decoding_time_bin_size][a_decoder_name]
    a_decoded_laps_epochs_result.filter_epochs._df['lap_id'] = _safe_epoch_ids_from_label_or_order(a_decoded_laps_epochs_result.filter_epochs._df) ## Prefer labels when numeric; fall back to row order for blank/non-numeric labels.
    a_decoded_laps_epochs_result.filter_epochs._df['original_epoch_idx'] = a_decoded_laps_epochs_result.filter_epochs._df['lap_id'].astype(int) ## it already is an inbetween_lap_id because it's built from the laps

    decoder_laps_result_out_dicts[a_decoder_name] = MomentumHelpers.perform_decoded_epochs_momentum_analysis(a_decoded_PBEs_result=a_decoded_laps_epochs_result, figure_name=f'Laps[{a_decoder_name}]')
    
    # n_timebins, flat_time_bin_containers, timebins_p_x_given_n = a_decoded_laps_epochs_result.flatten()
    # cumm_flattened_p_x_given_n = np.nansum(timebins_p_x_given_n, axis=-1)
    # cumm_flattened_p_x_given_n = cumm_flattened_p_x_given_n / float(n_timebins)
    # across_all_time_bin_p_x_given_n_dict[a_decoder_name] = cumm_flattened_p_x_given_n



In [ ]:



a_decoded_PBEs_result = masked_container.epochs_decoded_result_cache_dict[0.025]['roam']
MomentumHelpers.perform_decoded_epochs_momentum_analysis(a_decoded_PBEs_result=a_decoded_PBEs_result)

In [ ]:
# import sys
# import numpy as np
# import pyqtgraph as pg
from qtpy import QtWidgets, QtCore, QtGui

class ColoredLineItem(pg.GraphicsObject):
    """
    A highly optimized custom pyqtgraph item to draw line segments
    with varying colors (to represent acceleration stress).
    """
    def __init__(self):
        super().__init__()
        self.picture = QtGui.QPicture()
        
    def update_data(self, x, y, colors):
        self.picture = QtGui.QPicture()
        p = QtGui.QPainter(self.picture)
        p.setRenderHint(QtGui.QPainter.Antialiasing)
        
        # Draw all segments into a QPicture to be cached for ultra-fast rendering
        for i in range(len(x)-1):
            p.setPen(pg.mkPen(colors[i], width=4, cap=QtCore.Qt.RoundCap))
            p.drawLine(QtCore.QPointF(x[i], y[i]), QtCore.QPointF(x[i+1], y[i+1]))
        p.end()
        
        # Force a redraw
        self.prepareGeometryChange()
        self.update()
        
    def paint(self, p, *args):
        p.drawPicture(0, 0, self.picture)
        
    def boundingRect(self):
        return QtCore.QRectF(self.picture.boundingRect())


class KinematicSimulator(QtWidgets.QMainWindow):
    def __init__(self):
        super().__init__()
        self.setWindowTitle("Kinematic Envelope Simulator (PyQtGraph)")
        self.resize(1100, 700)
        
        # Simulation State
        self.dt = 0.02
        self.kp = 25.0
        self.kd = 5.0
        self.playing = True
        self.frame = 0
        self.waypoints = np.array([
            [100, 150], [700, 150], [700, 450],
            [250, 450], [100, 300], [250, 150]
        ], dtype=float)
        
        self.init_ui()
        self.run_simulation()
        
        # Animation Timer (50 FPS)
        self.timer = QtCore.QTimer()
        self.timer.timeout.connect(self.tick)
        self.timer.start(int(self.dt * 1000))


    def init_ui(self):
        # Main Widget & Layout
        central_widget = QtWidgets.QWidget()
        self.setCentralWidget(central_widget)
        main_layout = QtWidgets.QHBoxLayout(central_widget)
        main_layout.setContentsMargins(0, 0, 0, 0)
        
        # --- Left Panel (Controls) ---
        left_panel = QtWidgets.QWidget()
        left_panel.setFixedWidth(320)
        left_layout = QtWidgets.QVBoxLayout(left_panel)
        left_layout.setSpacing(20)
        
        # Title
        title = QtWidgets.QLabel("Kinematic Bounds")
        title.setFont(QtGui.QFont("Arial", 16, QtGui.QFont.Bold))
        left_layout.addWidget(title)
        
        # Play/Pause & Restart Buttons
        btn_layout = QtWidgets.QHBoxLayout()
        self.btn_play = QtWidgets.QPushButton("Pause")
        self.btn_play.clicked.connect(self.toggle_play)
        self.btn_restart = QtWidgets.QPushButton("Restart")
        self.btn_restart.clicked.connect(self.restart)
        btn_layout.addWidget(self.btn_play)
        btn_layout.addWidget(self.btn_restart)
        left_layout.addLayout(btn_layout)
        
        # Max Force Slider
        self.lbl_amax = QtWidgets.QLabel("Max Force Limit (a_max): 1200")
        self.slider_amax = QtWidgets.QSlider(QtCore.Qt.Horizontal)
        self.slider_amax.setRange(200, 5000)
        self.slider_amax.setValue(1200)
        self.slider_amax.valueChanged.connect(self.on_params_changed)
        
        amax_desc = QtWidgets.QLabel("Simulates maximum muscle force. Lower limits force wider, smoother turns.")
        amax_desc.setWordWrap(True)
        amax_desc.setStyleSheet("color: gray; font-size: 11px;")
        
        left_layout.addWidget(self.lbl_amax)
        left_layout.addWidget(amax_desc)
        left_layout.addWidget(self.slider_amax)
        
        # Speed Slider
        self.lbl_speed = QtWidgets.QLabel("Animal Speed (v): 250 px/s")
        self.slider_speed = QtWidgets.QSlider(QtCore.Qt.Horizontal)
        self.slider_speed.setRange(100, 600)
        self.slider_speed.setValue(250)
        self.slider_speed.valueChanged.connect(self.on_params_changed)
        
        speed_desc = QtWidgets.QLabel("Higher momentum requires vastly more force to turn.")
        speed_desc.setWordWrap(True)
        speed_desc.setStyleSheet("color: gray; font-size: 11px;")
        
        left_layout.addWidget(self.lbl_speed)
        left_layout.addWidget(speed_desc)
        left_layout.addWidget(self.slider_speed)
        
        # Live Telemetry Panel
        telemetry_group = QtWidgets.QGroupBox("Live Telemetry")
        telemetry_layout = QtWidgets.QFormLayout(telemetry_group)
        
        self.lbl_val_speed = QtWidgets.QLabel("0")
        self.lbl_val_speed.setFont(QtGui.QFont("Consolas", 12))
        self.lbl_val_accel = QtWidgets.QLabel("0")
        self.lbl_val_accel.setFont(QtGui.QFont("Consolas", 12))
        
        telemetry_layout.addRow("Current Speed:", self.lbl_val_speed)
        telemetry_layout.addRow("Acceleration:", self.lbl_val_accel)
        
        self.force_gauge = QtWidgets.QProgressBar()
        self.force_gauge.setTextVisible(False)
        self.force_gauge.setFixedHeight(10)
        telemetry_layout.addRow(self.force_gauge)
        left_layout.addWidget(telemetry_group)
        
        left_layout.addStretch()
        main_layout.addWidget(left_panel)
        
        # --- Right Panel (PyQtGraph Plot) ---
        pg.setConfigOption('background', 'w')
        pg.setConfigOption('foreground', 'k')
        
        self.plot_widget = pg.PlotWidget()
        self.plot_widget.setAspectLocked(True)
        self.plot_widget.getViewBox().invertY(True) # Invert Y to match SVG standard
        self.plot_widget.setXRange(0, 800)
        self.plot_widget.setYRange(50, 550)
        self.plot_widget.showGrid(x=True, y=True, alpha=0.3)
        main_layout.addWidget(self.plot_widget)
        
        # Add Plot Items
        # 1. Theoretical dashed path
        self.target_line = pg.PlotCurveItem(pen=pg.mkPen(color=(203, 213, 225), width=2, style=QtCore.Qt.DashLine))
        self.plot_widget.addItem(self.target_line)
        
        # 2. Actual physical path (Colored)
        self.actual_line = ColoredLineItem()
        self.plot_widget.addItem(self.actual_line)
        
        # 3. Target Ghost Dot
        self.dot_target = pg.ScatterPlotItem(size=8, brush=pg.mkBrush(148, 163, 184), pen=None)
        self.plot_widget.addItem(self.dot_target)
        
        # 4. Animal Dot
        self.dot_animal = pg.ScatterPlotItem(size=14, brush=pg.mkBrush(59, 130, 246), pen=None)
        self.plot_widget.addItem(self.dot_animal)


    def toggle_play(self):
        self.playing = not self.playing
        self.btn_play.setText("Pause" if self.playing else "Play")


    def restart(self):
        self.frame = 0


    def on_params_changed(self):
        self.lbl_amax.setText(f"Max Force Limit (a_max): {self.slider_amax.value()}")
        self.lbl_speed.setText(f"Animal Speed (v): {self.slider_speed.value()} px/s")
        self.run_simulation()


    def run_simulation(self):
        """Re-calculates the entire trajectory arrays when sliders change."""
        aMax = self.slider_amax.value()
        speed = self.slider_speed.value()
        
        # 1. Generate theoretical target path
        t_path = []
        for i in range(len(self.waypoints)):
            wp = self.waypoints[i]
            next_wp = self.waypoints[(i + 1) % len(self.waypoints)]
            dist = np.hypot(next_wp[0] - wp[0], next_wp[1] - wp[1])
            steps = int(dist / (speed * self.dt))
            
            if steps == 0: continue
            for step in range(steps):
                frac = step / steps
                t_path.append([
                    wp[0] + (next_wp[0] - wp[0]) * frac,
                    wp[1] + (next_wp[1] - wp[1]) * frac
                ])
                
        self.target_path = np.array(t_path)
        
        # 2. Simulate animal following the target
        a_path, v_mags, a_mags, is_maxed = [], [], [], []
        
        pos = np.array(self.target_path[0], dtype=float)
        vel = np.array([0.0, 0.0], dtype=float)
        
        for target in self.target_path:
            # PD Controller attempting to reach the target dot
            ax = self.kp * (target[0] - pos[0]) - self.kd * vel[0]
            ay = self.kp * (target[1] - pos[1]) - self.kd * vel[1]
            
            a_mag = np.hypot(ax, ay)
            actual_a_mag = a_mag
            
            # The Core Physics Constraint
            if a_mag > aMax:
                ax = ax * (aMax / a_mag)
                ay = ay * (aMax / a_mag)
                actual_a_mag = aMax
                
            vel[0] += ax * self.dt
            vel[1] += ay * self.dt
            pos[0] += vel[0] * self.dt
            pos[1] += vel[1] * self.dt
            
            a_path.append(pos.copy())
            v_mags.append(np.hypot(vel[0], vel[1]))
            a_mags.append(actual_a_mag)
            is_maxed.append(actual_a_mag >= aMax - 0.1)
            
        self.actual_path = np.array(a_path)
        self.v_mags = np.array(v_mags)
        self.a_mags = np.array(a_mags)
        self.is_maxed = np.array(is_maxed)
        
        # Update static plot graphics
        self.target_line.setData(self.target_path[:, 0], self.target_path[:, 1])
        
        # Generate colors for the trajectory line (Green -> Red based on stress)
        stress = np.clip(self.a_mags / aMax, 0, 1)
        r = (stress * 255).astype(int)
        g = ((1 - stress) * 200 + 50).astype(int)
        
        colors = [QtGui.QColor(r[i], g[i], 50) for i in range(len(r))]
        self.actual_line.update_data(self.actual_path[:, 0], self.actual_path[:, 1], colors)
        
        # Reset animation safety bound
        if self.frame >= len(self.actual_path):
            self.frame = 0


    def tick(self):
        """Called every frame to update the animation positions and UI"""
        if not self.playing or len(self.actual_path) == 0:
            return
            
        self.frame = (self.frame + 1) % len(self.actual_path)
        
        # Update target dot
        t_pos = self.target_path[self.frame]
        self.dot_target.setData([t_pos[0]], [t_pos[1]])
        
        # Update animal dot & color
        a_pos = self.actual_path[self.frame]
        maxed = self.is_maxed[self.frame]
        brush_color = (239, 68, 68) if maxed else (59, 130, 246) # Red if maxed, Blue otherwise
        self.dot_animal.setData([a_pos[0]], [a_pos[1]], brush=pg.mkBrush(*brush_color))
        
        # Update Telemetry Text
        speed_val = self.v_mags[self.frame]
        accel_val = self.a_mags[self.frame]
        
        self.lbl_val_speed.setText(f"{int(speed_val)}")
        self.lbl_val_accel.setText(f"{int(accel_val)}")
        
        # Color the acceleration text
        if maxed:
            self.lbl_val_accel.setStyleSheet("color: #ef4444; font-weight: bold;")
        else:
            self.lbl_val_accel.setStyleSheet("color: #16a34a;")
            
        # Update Force Gauge
        pct = int(min(100, (accel_val / self.slider_amax.value()) * 100))
        self.force_gauge.setValue(pct)
        if maxed:
            self.force_gauge.setStyleSheet("QProgressBar::chunk { background-color: #ef4444; }")
        else:
            self.force_gauge.setStyleSheet("QProgressBar::chunk { background-color: #22c55e; }")



app = pg.mkQApp('momentum_viz_sim')
window = KinematicSimulator()
window.show()

# 2026-05-07 - Modern Bapun Fixup for bad `Pf2D`, bad `Decoder_2d`, etc

In [ ]:
for name in curr_active_pipeline.active_configs.keys():
    in_results = name in curr_active_pipeline.computation_results
    has_pf2d = in_results and ('pf2D' in curr_active_pipeline.computation_results[name].computed_data)
    pf_colors = None
    try:
        pf_colors = curr_active_pipeline.active_configs[name].plotting_config.pf_colors
    except Exception:
        pass
    print(name, "in_results:", in_results, "has_pf2D:", has_pf2d, "pf_colors_is_none:", (pf_colors is None))


# maze_GLOBAL in_results: True has_pf2D: True pf_colors_is_none: False
# roam in_results: True has_pf2D: True pf_colors_is_none: False
# sprinkle in_results: True has_pf2D: True pf_colors_is_none: False
# maze_GLOBAL_GLOBAL in_results: True has_pf2D: False pf_colors_is_none: True


### Most Important: drop bad epoch key `"maze_GLOBAL_GLOBAL"` which exists due to a bug and has invalid Pf2Ds and such

In [ ]:
bad_key = "maze_GLOBAL_GLOBAL"
p = curr_active_pipeline

def _safe_pop(container, key):
    try:
        if container is not None and key in container:
            container.pop(key, None)
            return True
    except Exception:
        pass
    return False

removed = {}

# Primary synchronized maps (most important)
for attr in ["filtered_sessions", "filtered_epochs", "filtered_contexts", "active_configs", "computation_results"]:
    container = getattr(p, attr, None)
    removed[attr] = _safe_pop(container, bad_key)

# If currently in Display stage, also clean stage-owned maps directly (usually same objects, but safe)
stage = getattr(p, "stage", None)
if stage is not None:
    for attr in ["filtered_sessions", "filtered_epochs", "filtered_contexts", "active_configs", "computation_results"]:
        container = getattr(stage, attr, None)
        removed[f"stage.{attr}"] = _safe_pop(container, bad_key)

# Remove display outputs whose context/filter points to the bad key
display_removed = 0
disp = getattr(p, "display_output", None)
if disp is not None:
    keys_to_remove = []
    for k in list(disp.keys()):
        # Case A: key is the raw string
        if k == bad_key:
            keys_to_remove.append(k)
            continue
        # Case B: key is an IdentifyingContext-like object
        try:
            if getattr(k, "filter_name", None) == bad_key:
                keys_to_remove.append(k)
        except Exception:
            pass
    for k in keys_to_remove:
        disp.pop(k, None)
        display_removed += 1

print("Removed bad_key from:", {k: v for k, v in removed.items() if v})
print(f"Removed display_output contexts: {display_removed}")

# Sanity check current top-level key alignment
def _keys(x):
    try:
        return set(x.keys())
    except Exception:
        return set()

key_sets = {
    "filtered_sessions": _keys(getattr(p, "filtered_sessions", None)),
    "filtered_contexts": _keys(getattr(p, "filtered_contexts", None)),
    "active_configs": _keys(getattr(p, "active_configs", None)),
    "computation_results": _keys(getattr(p, "computation_results", None)),
}
print({k: len(v) for k, v in key_sets.items()})
print("bad_key still present anywhere:", {k: (bad_key in v) for k, v in key_sets.items()})

In [ ]:
## stwuggling a bit, which I'm strangely surprised by?

## Afraid of limits, afraid of tonight, etc


In [ ]:
# curr_active_pipeline.filtered_contexts
list(curr_active_pipeline.filtered_sessions.keys()) # ['maze_GLOBAL', 'roam', 'sprinkle']


# ⚓🟢 2026-05-13 - Renewed Attempts at Sprinkle Segmentation by Burst Changepoint Detection of Behaviors

In [ ]:
from pyphoplacecellanalysis.SpecificResults.MovementBurstDetection import compute_movement_trajectories_from_bursts

hardcoded_params: HardcodedProcessingParameters = BapunDataSessionFormatRegisteredClass._get_session_specific_parameters(session_context=curr_active_pipeline.get_session_context())
burst_detector_kwargs = dict(
            min_burst_duration=1.5,      # Minimum burst duration in seconds
            min_rest_duration=0.1,       # Minimum rest period between bursts
            velocity_smoothing=0.15,     # Smoothing for velocity calculation
            bocd_hazard=120,             # Sensitivity of changepoint detection
            # clustering_method='hdbscan',  # Clustering algorithm
            # clustering_method='dbscan',  # Clustering algorithm
            clustering_method='dip',  # Use DipExt from clustpy
            use_gpu=False,             # Set to True if you have CUDA
)
_lap_burst_detection_results, _out_laps = compute_movement_trajectories_from_bursts(curr_active_pipeline, epoch_names=hardcoded_params.decoder_building_session_names, **burst_detector_kwargs)
_out_laps
## OUTPUTS: _lap_burst_detection_results, _out_laps

In [ ]:
from pyphoplacecellanalysis.SpecificResults.MovementBurstDetection import OptimizedMovementBurstDetector, BurstAnalyzer

# Initialize optimized detector
detector = OptimizedMovementBurstDetector(
    # min_burst_duration=0.8,
    # min_rest_duration=1.2,
    min_burst_duration=0.8,
    min_rest_duration=0.2,
    velocity_smoothing=0.15,
    # bocd_hazard=100,
    bocd_hazard=1000,
    clustering_method='dip',  # Use DipExt from clustpy
    use_gpu=False  # Set to True if you have CUDA
)

# Detect bursts
print("\nDetecting bursts with optimized pipeline...")
results = detector.detect_bursts(pos_df)

# Analyze results
analyzer = BurstAnalyzer()
summary = analyzer.summarize_bursts(results)

print(f"\nDetection Summary:")
print(f"  Total bursts: {summary['total_bursts']}")
if summary['total_bursts'] > 0:
    print(f"  Total burst duration: {summary['total_burst_duration']:.1f}s")
    print(f"  Mean burst duration: {summary['mean_burst_duration']:.2f} ± {summary['std_burst_duration']:.2f}s")
    print(f"  Mean burst speed: {summary['mean_burst_speed']:.3f}")
    print(f"  Total distance during bursts: {summary['total_distance']:.2f}")
    if 'burst_frequency' in summary:
        print(f"  Burst frequency: {summary['burst_frequency']:.3f} Hz")

# Display individual bursts
print(f"\nDetected Bursts:")
for i, burst in enumerate(results['bursts']):
    print(f"  Burst {i+1}: {burst['start']:.1f}-{burst['end']:.1f}s "
            f"(dur: {burst['duration']:.1f}s, speed: {burst['mean_speed']:.3f}, "
            f"dist: {burst.get('total_distance', 0):.2f})")

# Visualize
print("\nGenerating visualization...")
analyzer.visualize_results(results, save_path='optimized_burst_detection.png')

## OUTPUTS: results

In [ ]:
from pyphoplacecellanalysis.SpecificResults.MovementBurstDetection import EmpericalSprintDetection

out = EmpericalSprintDetection(threshold_method="gmm", rolling_window_s=60.0).detect(pos_df)
# out["sprints"], out["stop_intervals"], out["is_running"], out["threshold_speed"], ...
out

# 2026-05-12 - Bapun Batch Processing

In [ ]:
from pyphoplacecellanalysis.SpecificResults.PendingNotebookCode import BapunBatchHelpers

_out_dict = BapunBatchHelpers.run_all(curr_active_pipeline=curr_active_pipeline)
_out_dict